# AI Hub 외부 데이터 EDA

## EDA 목적

- 대회 train(232장, 56개 클래스)을 보충할 AI Hub 576 "경구약제 이미지 데이터"의 조합 데이터를 확인한다.
- 대회 데이터와 같은 기준으로 파일 구성, 라벨 형식, 라벨 품질, bbox와 클래스 분포를 본다.
- AI Hub 라벨을 어떻게 바꾸고 고칠지, 어떤 클래스와 이미지를 학습에 넣을지 정한다.
- test에만 나오는 알약을 AI Hub로 채울 수 있는지 확인한다.

## 데이터

| 데이터 | 위치 | 이 노트북에서 쓰는 방법 |
|---|---|---|
| 대회 데이터(Kaggle) | `data/raw/`의 `train_images`, `train_annotations`, `test_images` | 비교 기준 |
| AI Hub 576 조합 데이터 | `data/external/aihub/raw/`의 `train_annotations`, `train_images`와 `data/external/aihub/sources.csv` | 분석 대상. 허용 zip 8쌍만 쓴다 |
| 정제 목록 | `data/label_fix/label_fixes.csv` | kaggle 줄 21개로 대회 train 라벨을 고치고, aihub 줄 1,093개로 AI Hub 라벨을 고친다 (읽기만 함). aihub 줄을 만든 과정은 [`label_fix/01_aihub_label_fix.ipynb`](label_fix/01_aihub_label_fix.ipynb)에 있다 |
| test 기록표 | `notebooks/eda/00_test_classes_review.csv` | test 사진을 보고 train에 없는 알약을 적은 표. 5절에서 쓴다 |

- 대회 규칙에 따라 `TL_2_조합.zip`, `TS_2_조합.zip`은 받지도 쓰지도 않는다. (`RULE.md`)

## 목차

| 섹션 | 내용 |
|---|---|
| 0. 사전 설정 | 라이브러리, 경로, 한글 글꼴, seed, 도우미 함수, 데이터 읽기, 이미지 값 미리 계산 |
| 1. 데이터 구성 | 받은 파일과 금지 범위, 라벨 구조, Kaggle 형식으로 바꾸기, 촬영 조건과 밝기, 샘플 이미지, 대회 데이터와 겹치는 사진 |
| 2. 라벨 품질과 정제 | 라벨 누락, 좌표 오류, 정제 목록 적용과 정제 뒤 수치, 뺀 이미지, 원본 → 1차 → 2차 정제 총정리 표 |
| 3. bbox 분포 | 객체 수와 크기, 라벨 여백, 위치와 겹침 |
| 4. 클래스 분포 | 클래스별 객체 수, 조합당 이미지 수, 같은 조합 사진, 대회 train과 합친 분포, 균일하게 고르기 |
| 5. test 전용 클래스 채우기 | test 각인 대조, test와 AI Hub 알약 비교, test 사진 커버리지, 이미지 고르는 방법 세 가지, 고르는 방법 총정리 표 |
| 6. 평가 기준선 | 합친 라벨로 mAP 계산 검증 |
| 7. 정리 | 이 EDA로 정한 것 |

## 실행 방법

- 데이터는 레포 `SETUP.md`의 "학습 데이터 준비"대로 받는다.
    - 대회 데이터는 `data/raw/`에 푼다.
    - AI Hub는 `uv run scripts/prepare_train_data.py`가 `data/external/aihub/`에 받는다.
- 처음부터 순서대로 실행한다. 뒤 섹션은 앞 섹션에서 만든 변수를 쓴다.
- 원본 데이터와 레포의 파일은 읽기만 한다. 이 노트북이 쓰는 파일은 `runs/eda_raewoo/` 아래(그림, 갤러리, 계산 캐시)뿐이다.
- 처음 실행은 이미지 약 17,000장을 읽어 값을 계산하느라 몇 분 걸린다. 계산 결과를 `runs/eda_raewoo/cache/`에 저장해서, 다시 실행하면 빨라진다.
- 학습 데이터는 이 노트북이 아니라 `scripts/prepare_train_data.py`가 만든다. 이 노트북은 AI Hub를 어떻게 쓸지 정한 근거이고, 정제 목록을 만든 과정은 `label_fix/01_aihub_label_fix.ipynb`에 있다.

## 용어

| 용어 | 뜻 |
|---|---|
| 조합 | 함께 놓고 찍은 알약 묶음. 파일명의 첫 부분(`K-001900-016548-019607-029451`)으로, 알약의 K-코드를 이은 것이다. 조합마다 70°, 75°, 90° 사진이 있다 |
| K-코드 | 알약 코드(`K-001900`). 숫자 부분이 클래스 번호(category_id)다 |
| 촬영 PNG | 이미지 zip에 든 실제 사진 (`<조합>_0_2_0_2_<각도>_000_200.png`) |
| `_index.png` | 이미지 zip에 조합마다 하나씩 든 참고 사진. 학습에 쓰지 않는다 (1-1-3) |
| copy 파일 | 이름에 ` copy`, ` copy 2`가 붙은 AI Hub JSON (1-2, 2-1) |
| `-오류` 폴더 | 이름이 `K-<코드>-오류`인 AI Hub 알약 폴더 (1-2, 2-1) |
| IoU | 두 box가 겹친 넓이 ÷ 합친 넓이. 1이면 같은 box, 0이면 겹치지 않는다. 대회 지표는 IoU 0.75~0.95에서 계산한다 |
| 정제 목록 | 고칠 annotation과 고칠 값을 한 줄씩 적은 표. `data/label_fix/label_fixes.csv` (2-3) |
| 정제 라벨 | 정제 목록을 적용한 AI Hub 라벨 (2-3). 3절부터 이 라벨로 본다 |

## 0. 사전 설정

뒤 섹션에서 공통으로 쓰는 것을 준비하고, 데이터를 한 번에 읽어 둔다. 0-1~0-5는 설정과 도우미 함수, 0-6은 데이터 읽기, 0-7은 이미지 값 계산(캐시)이다.

### 0-1. 라이브러리 불러오기

- 버전을 출력해 둔다. 실행 결과가 다를 때 버전 차이부터 확인하기 위해서다.

In [ ]:
import collections
import concurrent.futures
import contextlib
import csv
import hashlib
import io
import json
import os
import sys
import unicodedata
from pathlib import Path

import cv2
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import scipy.optimize
from IPython.display import Markdown, display
from matplotlib import font_manager
from PIL import Image, ImageDraw, ImageFont
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# 금지 파일 검사, 파일명 해석, 정제 목록 읽기는 공용 모듈의 함수를 쓴다. 모두 파일을 읽기만 한다
from pill_detection.data import check_aihub_file, combo_name, pill_codes, read_label_fixes

# 실행 환경의 버전 확인 (팀원 간 환경 차이로 오류가 생겼을 때 원인 파악용)
print(f"Python       : {sys.version.split()[0]}")
print(f"numpy        : {np.__version__}")
print(f"matplotlib   : {matplotlib.__version__}")
print(f"OpenCV       : {cv2.__version__}")

### 0-2. 경로 설정

- `pyproject.toml`이 있는 폴더를 레포 최상위로 잡고, 데이터 경로를 그 아래로 정한다.
- 그림, 갤러리, 계산 캐시는 `runs/eda_raewoo/`에만 저장한다. `runs/`는 git에서 제외된다.
- 데이터 폴더가 없으면 받는 방법을 알려 주고 멈춘다.

In [ ]:
# 노트북을 어디서 실행해도 레포 최상위(pyproject.toml이 있는 폴더)를 기준으로 경로를 잡는다
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
TRAIN_IMG_DIR = RAW_DIR / 'train_images'
TRAIN_ANN_DIR = RAW_DIR / 'train_annotations'
TEST_IMG_DIR = RAW_DIR / 'test_images'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'
AIHUB_DIR = EXTERNAL_DIR / 'aihub'  # scripts/prepare_train_data.py가 AI Hub를 받아 푸는 폴더
AIHUB_RAW_ANN_DIR = AIHUB_DIR / 'raw' / 'train_annotations'  # 라벨 zip을 그대로 푼 원본 JSON
AIHUB_IMG_DIR = AIHUB_DIR / 'raw' / 'train_images'  # 이미지 zip의 PNG
AIHUB_SOURCES_CSV = AIHUB_DIR / 'sources.csv'  # 파일마다 나온 zip (zip, path)
LABEL_FIXES_CSV = PROJECT_ROOT / 'data' / 'label_fix' / 'label_fixes.csv'  # 학습 데이터의 정제 목록. 대회 train 줄만 읽는다
EDA_DIR = PROJECT_ROOT / 'notebooks' / 'eda'
# 그림, 갤러리, 계산 결과(캐시)를 저장하는 곳. 이 노트북은 runs/ 밖에는 아무것도 쓰지 않는다 (runs/는 git에서 제외됨)
FIG_DIR = PROJECT_ROOT / 'runs' / 'eda_raewoo' / 'figures_01'
CACHE_DIR = PROJECT_ROOT / 'runs' / 'eda_raewoo' / 'cache'
FIG_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 대회와 AI Hub 이미지 크기 (너비, 높이)
IMG_W, IMG_H = 976, 1280

for folder in [TRAIN_IMG_DIR, TRAIN_ANN_DIR, TEST_IMG_DIR, AIHUB_RAW_ANN_DIR, AIHUB_IMG_DIR]:
    assert folder.is_dir(), (f'폴더가 없다: {folder.relative_to(PROJECT_ROOT)}\n'
                             '대회 데이터는 data/raw에 풀고, AI Hub는 uv run scripts/prepare_train_data.py로 받는다 (SETUP.md의 학습 데이터 준비)')
print(f"레포 최상위  : {PROJECT_ROOT.name}")
print(f"그림 저장    : {FIG_DIR.relative_to(PROJECT_ROOT)}")
print(f"캐시 저장    : {CACHE_DIR.relative_to(PROJECT_ROOT)}")

### 0-3. 한글 글꼴 설정

- macOS는 AppleGothic, Windows는 맑은 고딕, 둘 다 없으면 나눔고딕을 쓴다.

In [ ]:
# 그래프의 한글이 깨지지 않도록 설치된 한글 폰트 사용 (macOS, Windows, Linux 순)
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font_name in ['AppleGothic', 'Malgun Gothic', 'NanumGothic']:
    if font_name in installed_fonts:
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False  # 한글 폰트에서 마이너스 기호가 깨지는 문제 방지
print(f"그래프 폰트  : {plt.rcParams['font.family'][0]}")

### 0-4. seed 고정

- 무작위로 고르는 셀은 이 `SEED`로 고른다.

In [ ]:
# 랜덤 샘플 시각화 등의 재현성을 위한 seed 고정
SEED = 42
np.random.seed(SEED)
print(f'SEED : {SEED}')

### 0-5. 도우미 함수

- 표 출력(`show_table`), 그림 저장(`save_fig`), IoU 계산, box 중심 crop, box 위 글자, 알약별 JSON을 이미지 단위로 묶어 읽기. 하는 일은 각 함수의 docstring에 있다.

In [ ]:
def save_fig(fig, name):
    """그림을 FIG_DIR에 png 파일로 저장한다."""
    fig.savefig(FIG_DIR / f'{name}.png', dpi=120, bbox_inches='tight')


def show_table(header, rows):
    """열 이름 리스트와 행 리스트를 마크다운 표로 출력한다."""
    lines = ['| ' + ' | '.join(header) + ' |', '|' + ' --- |' * len(header)]
    # 값 속 |(예: 분할선 각인 K|50)가 칸 구분자로 읽히지 않게 \|로 바꾼다
    lines += ['| ' + ' | '.join(str(value).replace('|', r'\|') for value in row) + ' |' for row in rows]
    display(Markdown('\n'.join(lines)))


def box_iou(box_a, box_b):
    """두 xywh box의 IoU(겹친 넓이 ÷ 합친 넓이, 1이면 완전히 같음)를 계산한다."""
    inter_w = max(0, min(box_a[0] + box_a[2], box_b[0] + box_b[2]) - max(box_a[0], box_b[0]))
    inter_h = max(0, min(box_a[1] + box_a[3], box_b[1] + box_b[3]) - max(box_a[1], box_b[1]))
    inter = inter_w * inter_h
    return inter / (box_a[2] * box_a[3] + box_b[2] * box_b[3] - inter)


def crop_square(image, box, pad=0.1, size=224):
    """box 중심에서 긴 변보다 pad만큼 큰 정사각형을 잘라 size×size로 줄인다. 알약 모양이 늘어나지 않는다."""
    x, y, w, h = box
    side = max(w, h) * (1 + pad)
    cx, cy = x + w / 2, y + h / 2
    return image.crop((int(cx - side / 2), int(cy - side / 2), int(cx + side / 2), int(cy + side / 2))).resize((size, size))


def pill_label(ann):
    """box 위에 적을 K-코드와 각인(메타데이터 print_front/print_back)을 만든다."""
    marks = [ann['image'][key] for key in ('print_front', 'print_back') if ann['image'].get(key)]
    return f"K-{ann['category_id']:06d} ({'/'.join(marks) or '각인 없음'})"

def load_image_records(ann_dir):
    """annotation 폴더의 알약별 JSON을 이미지 단위로 묶는다.

    Args:
        ann_dir (Path): annotation 폴더. 하위 폴더의 JSON을 경로 순서로 모두 읽는다.

    Returns:
        dict: 이미지 파일명 → {'image': 그 이미지의 첫 JSON의 images[0], 'annotations': 알약마다 dict 하나}.
            annotation dict는 annotations[0]에 그 JSON의 images[0]('image')와 JSON 경로('json_path')를 더한 것이다.
    """
    records = {}
    for path in sorted(ann_dir.rglob('*.json')):
        data = json.loads(path.read_text(encoding='utf-8'))
        image = data['images'][0]
        record = records.setdefault(image['file_name'], {'image': image, 'annotations': []})
        for ann in data['annotations']:
            record['annotations'].append({**ann, 'image': image, 'json_path': str(path)})
    return records

### 0-6. 데이터 읽기

| 변수 | 내용 |
|---|---|
| `train_records` | 대회 train 원본 라벨. 이미지 하나에 annotation 여러 개 |
| `clean_records` | 대회 train 정제 라벨. `data/label_fix/label_fixes.csv`의 kaggle 줄 21개를 적용했다 (이미지 232장, 객체 771개) |
| `review_rows` | test 기록표. test 사진에서 train에 없는 알약 하나가 한 줄이다 |
| `aihub_jsons` | AI Hub 원본 JSON 전체. JSON 파일 하나가 한 줄이고, 읽지 못한 JSON도 오류와 함께 남긴다 |

- AI Hub 원본은 아무것도 거르지 않고 읽는다. 무엇을 뺄지는 1절부터 정한다.
- zip은 풀고 지웠으므로, JSON과 PNG가 어느 zip에서 나왔는지는 `sources.csv`(zip, 경로)로 본다.
- 대회 train 정제에서 빠진 알약의 box를 넣을 때는, 알약 정보를 같은 클래스의 다른 annotation에서 가져온다.

In [ ]:
train_records = load_image_records(TRAIN_ANN_DIR)
train_files = sorted(os.listdir(TRAIN_IMG_DIR))
test_files = sorted(os.listdir(TEST_IMG_DIR), key=lambda name: int(name.split('.')[0]))
train_classes = sorted({a['category_id'] for r in train_records.values() for a in r['annotations']})
train_combos = sorted({combo_name(fn) for fn in train_files})
assert set(train_records) == set(train_files), '라벨이 있는 이미지와 이미지 파일 목록이 다르다'

# 대회 train 정제: 학습 데이터의 정제 목록에서 대회 train 줄(source=kaggle)만 읽어 원본 라벨에 적용한다
IMAGE_FIELDS = ['back_color', 'camera_la', 'camera_lo', 'drug_S', 'drug_dir', 'file_name', 'height', 'id', 'imgfile',
                'light_color', 'size', 'width']  # 메타데이터 중 같은 이미지의 알약끼리 값이 같은 필드 (나머지는 알약 필드)
train_fixes = read_label_fixes(LABEL_FIXES_CSV, source='kaggle')  # add: 빠진 알약의 box를 넣음, modify: box를 고침
class_example = {}  # K-코드 → 대회 train에서 그 클래스의 annotation 하나 (그려 넣은 box의 메타데이터용)
for fn in train_files:
    for ann in train_records[fn]['annotations']:
        class_example.setdefault(ann['category_id'], ann)
clean_records = {}
for fn, record in sorted(train_records.items()):
    by_code = {a['category_id']: a for a in record['annotations']}
    anns = [dict(a) for a in record['annotations']]
    for fix in (row for row in train_fixes if row['file_name'] == fn):
        code, box = fix['category_id'], fix['bbox_after']
        if fix['action'] == 'add':  # 그려 넣은 box
            example = class_example[code]
            anns.append({**example, 'id': None, 'image_id': record['image']['id'], 'json_path': None, 'bbox': box,
                         'image': {**example['image'], **{key: record['image'][key] for key in IMAGE_FIELDS}}})
            continue
        # 원본 annotation 순서로 찾는다 (라벨을 서로 바꾸면 category_id가 바뀌므로 바뀐 값으로 찾지 않는다)
        ann = anns[next(i for i, a in enumerate(record['annotations']) if a['category_id'] == code)]
        ann['bbox'] = box
        if fix['category_id_after']:  # 라벨이 서로 바뀐 경우: 클래스와 알약 메타데이터를 상대 라벨의 것으로 바꾼다
            other = by_code[fix['category_id_after']]
            ann['category_id'], ann['image'] = other['category_id'], other['image']
    for ann in anns:
        ann['area'] = ann['bbox'][2] * ann['bbox'][3]
    clean_records[fn] = {**record, 'annotations': anns}
clean_object_count = collections.Counter(a['category_id'] for r in clean_records.values() for a in r['annotations'])
assert (len(clean_records), sum(clean_object_count.values()), len(clean_object_count)) == (232, 771, 56), \
    (f'대회 train 정제 결과가 이미지 232장, 객체 771개, 클래스 56개가 아니다.\n'
     f'{LABEL_FIXES_CSV.relative_to(PROJECT_ROOT)}의 kaggle 줄이 바뀌었는지 확인한다: git log -p -- data/label_fix/label_fixes.csv')

with open(EDA_DIR / '00_test_classes_review.csv', encoding='utf-8') as f:
    review_rows = list(csv.DictReader(f))

# AI Hub: zip은 풀고 지웠으므로, 파일마다 나온 zip은 sources.csv(zip, path)로 본다. path는 raw/ 기준이다
PREPARE_HINT = 'AI Hub 받는 명령: uv run scripts/prepare_train_data.py (SETUP.md의 학습 데이터 준비)'
assert AIHUB_SOURCES_CSV.is_file(), f'{AIHUB_SOURCES_CSV.relative_to(PROJECT_ROOT)}이 없다.\n{PREPARE_HINT}'
with open(AIHUB_SOURCES_CSV, encoding='utf-8') as f:
    sources = [(unicodedata.normalize('NFC', row['zip']), unicodedata.normalize('NFC', row['path'])) for row in csv.DictReader(f)]
for zip_name in sorted({z for z, _ in sources}):
    check_aihub_file(zip_name)  # 금지 파일이면 여기서 오류가 난다
label_zips = sorted({z for z, path in sources if path.startswith('train_annotations/')})
assert len(label_zips) == 8, f'허용 조합 라벨 zip 8개를 모두 받아야 한다. 지금 {len(label_zips)}개\n{PREPARE_HINT}'
label_zip_of_combo = {}  # 조합 폴더 이름 → 라벨 zip
zip_json_paths = set()  # sources.csv에 적힌 JSON 경로 (train_annotations/ 기준)
for zip_name, path in sources:
    if path.startswith('train_annotations/') and path.endswith('.json'):
        json_rel = path.removeprefix('train_annotations/')
        combo_folder = json_rel.split('/')[0]
        assert label_zip_of_combo.setdefault(combo_folder, zip_name) == zip_name, f'{combo_folder} 폴더가 두 zip에 있다'
        zip_json_paths.add(json_rel)
raw_paths = sorted(AIHUB_RAW_ANN_DIR.rglob('*.json'))
assert {unicodedata.normalize('NFC', p.relative_to(AIHUB_RAW_ANN_DIR).as_posix()) for p in raw_paths} == zip_json_paths, \
    f'{AIHUB_RAW_ANN_DIR.relative_to(PROJECT_ROOT)}의 JSON이 sources.csv와 다르다. 받다가 멈췄으면 다시 실행한다.\n{PREPARE_HINT}'
aihub_jsons = []  # 원본 JSON 파일 하나가 한 줄. 아무것도 거르지 않고 읽은 그대로 둔다
for path in raw_paths:
    combo_folder, pill_folder = (unicodedata.normalize('NFC', p.name) for p in (path.parent.parent, path.parent))
    try:
        data, error = json.loads(path.read_text(encoding='utf-8')), None
    except json.JSONDecodeError as e:
        data, error = None, str(e)
    aihub_jsons.append({
        'path': path, 'zip': label_zip_of_combo[combo_folder], 'combo_folder': combo_folder, 'pill_folder': pill_folder,
        # JSON 쪽 이미지 파일명: JSON 안의 file_name이 아니라 JSON 파일 이름에서 ' copy'를 떼고 만든다
        'file_name': unicodedata.normalize('NFC', path.name)[:-len('.json')].split(' copy')[0] + '.png',
        'data': data, 'error': error,
    })
aihub_png = sorted(p.name for p in AIHUB_IMG_DIR.glob('*.png'))
aihub_index_files = [fn for fn in aihub_png if fn.endswith('_index.png')]  # 90° 사진에 K-코드와 제품명을 적은 참고용 사진 (1-1)
aihub_files = [fn for fn in aihub_png if not fn.endswith('_index.png')]  # 촬영 사진

print(f'대회 train {len(train_files)}장 (정제 라벨 객체 {sum(clean_object_count.values())}개), test {len(test_files)}장, '
      f'test 기록표 {len(review_rows)}줄')
print(f"AI Hub 라벨 zip {len(label_zips)}개: 원본 JSON {len(aihub_jsons):,}개 (읽지 못한 JSON {sum(j['error'] is not None for j in aihub_jsons)}개), "
      f'받은 PNG {len(aihub_png):,}장 (그중 _index.png {len(aihub_index_files)}장)')

### 0-7. 이미지 값 미리 계산

대회 train 232장, test 842장, AI Hub 이미지를 한 번씩 읽어 아래 값을 계산해 둔다.

| 값 | 쓰는 곳 |
|---|---|
| 파일 내용 해시(MD5) | 1-6. 내용이 똑같은 파일 찾기 |
| 크기(너비, 높이)와 읽기 성공 여부 | 1-1-2. 해상도와 깨진 파일 확인 |
| 채널별 평균 밝기(R, G, B) | 1-4-2. 밝기 비교 |
| 32×40 회색 축소 그림 | 1-6. 다시 저장되었거나 조금 달라진 같은 사진 찾기 |

- 처음 실행은 몇 분 걸린다. 결과는 `runs/eda_raewoo/cache/image_stats.npz`에 저장하고, 파일 목록이 같으면 다시 계산하지 않는다.

In [ ]:
THUMB_W, THUMB_H = 32, 40  # 축소 그림 크기 (원본 976×1280과 같은 비율)


def image_stats(path):
    """이미지 하나의 MD5, 크기, 채널 평균 밝기, 회색 축소 그림을 계산한다. 읽지 못하면 크기를 (0, 0)으로 둔다."""
    data = path.read_bytes()
    bgr = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if bgr is None:
        return hashlib.md5(data).hexdigest(), (0, 0), np.zeros(3), np.zeros((THUMB_H, THUMB_W), np.uint8)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    thumb = cv2.resize(gray, (THUMB_W, THUMB_H), interpolation=cv2.INTER_AREA)
    return hashlib.md5(data).hexdigest(), (bgr.shape[1], bgr.shape[0]), bgr.reshape(-1, 3).mean(axis=0)[::-1], thumb


stat_paths = ([('train', TRAIN_IMG_DIR / fn) for fn in train_files] + [('test', TEST_IMG_DIR / fn) for fn in test_files]
              + [('aihub', AIHUB_IMG_DIR / fn) for fn in aihub_png])
stat_keys = np.array([f'{group}/{path.name}' for group, path in stat_paths])
cache_path = CACHE_DIR / 'image_stats.npz'
cache = np.load(cache_path) if cache_path.exists() else None
if cache is not None and np.array_equal(cache['keys'], stat_keys):
    stats = {name: cache[name] for name in cache.files}
else:
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:  # OpenCV는 이미지를 읽는 동안 다른 스레드를 막지 않는다
        results = list(pool.map(image_stats, [path for _, path in stat_paths]))
    stats = {'keys': stat_keys, 'md5': np.array([r[0] for r in results]), 'size': np.array([r[1] for r in results]),
             'mean_rgb': np.array([r[2] for r in results]), 'thumb': np.array([r[3] for r in results])}
    np.savez(cache_path, **stats)
stat_index = {key: i for i, key in enumerate(stats['keys'])}  # 'aihub/<파일명>' → 행 번호
print(f"이미지 {len(stat_keys):,}장의 값을 계산했다. (대회 train {len(train_files)}, test {len(test_files)}, AI Hub {len(aihub_png):,})")

## 1. 데이터 구성

AI Hub 576에서 무엇을 받을 수 있고, 받은 라벨과 사진이 대회 데이터와 같은 형식인지 확인한다. 대회 데이터의 원본도 이 데이터셋이다. (`DATA.md`)

| 단계 | 하는 일 | 이유 |
|---|---|---|
| 1-1 | 받은 파일과 금지 범위 | 쓸 수 있는 파일을 정하고, 라벨과 사진의 짝이 맞는지 본다 |
| 1-2 | 라벨 구조 비교 | AI Hub JSON이 대회 JSON과 어디가 다른지 찾는다 |
| 1-3 | Kaggle 형식으로 바꾸기 | 대회 train과 같은 코드로 읽도록 바꾸는 규칙을 정한다 |
| 1-4 | 촬영 조건과 밝기 | AI Hub 사진을 대회 사진과 그대로 섞어도 되는지 본다 |
| 1-5 | 샘플 이미지 | 라벨 오류가 없는 사진에서 box가 알약을 감싸는지 눈으로 본다 |
| 1-6 | 대회 데이터와 겹치는 사진 | 대회 train·test와 같은 사진(누수)이나 AI Hub 안의 같은 사진이 있는지 본다 |
| 1-7 | 정리 | 확인한 것과 정한 것 |

라벨 값이 맞는지(누락, 좌표 오류)는 2절에서 본다. 1절은 파일과 형식만 본다.

### 1-1. 받은 파일과 금지 범위

- AI Hub 576의 파일 목록에서 쓸 수 있는 zip을 정하고(1-1-1), 라벨 zip과 이미지 zip의 짝이 맞는지 보고(1-1-2), `_index.png`가 무엇인지 확인한 뒤(1-1-3), 짝이 없어 쓸 수 없는 파일을 뺀다(1-1-4).

#### 1-1-1. AI Hub 576 파일 목록

- AI Hub 576의 파일 목록(2026-10-24 조회)을 정리한다.
- TL·VL은 라벨(JSON) zip, TS·VS는 이미지 zip이다. T는 Training, V는 Validation이다.
- 용량은 목록에 적힌 반올림 값이다.

In [ ]:
AIHUB_FILES = [
    # (구분, 파일, 개수, 파일 하나 용량, filekey, 사용)
    ('Training 조합 라벨', 'TL_1, TL_3~8_조합.zip', 7, '7~8MB', '66065, 66067~66072', '사용 가능'),
    ('Training 조합 라벨', 'TL_2_조합.zip', 1, '7MB', '66066', '**금지**'),
    ('Training 조합 이미지', 'TS_1, TS_3~8_조합.zip', 7, '3GB', '66154, 66156~66161', '사용 가능'),
    ('Training 조합 이미지', 'TS_2_조합.zip', 1, '3GB', '66155', '**금지**'),
    ('Validation 조합 라벨', 'VL_1_조합.zip', 1, '7MB', '66243', '사용 가능'),
    ('Validation 조합 이미지', 'VS_1_조합.zip', 1, '3GB', '66254', '사용 가능'),
    ('Training 단일 라벨', 'TL_1~81_단일.zip', 81, '11~73MB (합계 2.6GB)', '66073~66153', '사용 가능'),
    ('Training 단일 이미지', 'TS_1~81_단일.zip', 81, '13~95GB (합계 3,473GB)', '66162~66242', '사용 가능 (로컬 디스크 부족)'),
    ('Validation 단일 라벨', 'VL_1~10_단일.zip', 10, '15~24MB', '66244~66253', '사용 가능'),
    ('Validation 단일 이미지', 'VS_1~10_단일.zip', 10, '19~31GB (합계 228GB)', '66255~66264', '사용 가능 (로컬 디스크 부족)'),
]
show_table(['구분', '파일', '개수', '용량', 'filekey', '사용'], [list(row) for row in AIHUB_FILES])

#### 1-1-1 결과

| 구분 | 내용 |
|---|---|
| 조합 데이터 | 사진 한 장에 알약 3~4개. 대회 사진과 같은 형태다 |
| 단일 데이터 | 이름으로 보아 알약을 하나씩 찍은 사진이다. 이미지 합계 약 3,701GB(Training 3,473GB, Validation 228GB)다 |

- 조합 데이터에서 쓸 수 있는 것은 라벨 zip 8개(TL_1, TL_3~8, VL_1)와 이미지 zip 8개(TS_1, TS_3~8, VS_1)다.
- 금지 파일은 TL_2, TS_2 둘뿐이다.
- 이미지 zip은 하나에 약 3GB라 8개를 모두 받아도 약 24GB다.

조합 데이터의 허용 zip 8쌍만 쓴다. 단일 데이터는 로컬 디스크에 받을 수 없는 크기라 쓰지 않는다.

#### 1-1-2. 라벨 zip과 이미지 zip의 짝

같은 번호의 라벨 zip(TL_n, VL_1)과 이미지 zip(TS_n, VS_1)이 서로 맞는지 zip 쌍마다 센다.

| 세는 대상 | 뜻 |
|---|---|
| JSON 파일 | 라벨 zip의 파일. 알약 하나의 라벨이다. 경로는 `<조합>_json/K-<코드>/<이미지 파일명>.json` |
| JSON 쪽 이미지 파일명 | JSON 파일 이름에서 ` copy`를 떼고 `.png`로 바꾼 이름. 알약이 3~4개라 JSON 3~4개가 같은 이름을 가리킨다 |
| 촬영 PNG | 이미지 zip의 사진 |
| `_index.png` | 이미지 zip에 조합마다 하나씩 있는 참고 사진. 가리키는 JSON이 없다 |

- 두 쪽을 이미지 파일명으로 맞대어 "둘 다 있음", "JSON만 있음", "촬영 PNG만 있음"으로 나눈다.
- PNG마다 실제 크기가 976×1280인지, 읽지 못하는 파일이 있는지도 센다. (0-7에서 계산한 값)
- JSON과 촬영 PNG가 번호가 다른 zip 쌍에 든 경우("다른 번호의 zip 쌍")는 표에서 두 쌍에 한 번씩 세인다. 결과에서 종류별로 다시 정리한다.

In [ ]:
image_zip_of = {path.removeprefix('train_images/'): z for z, path in sources if path.startswith('train_images/')}  # PNG 파일명 → 나온 이미지 zip
label_zip_of = {j['file_name']: j['zip'] for j in aihub_jsons}  # JSON 쪽 이미지 파일명 → 라벨 zip
PAIR = {f'{kind}L_{n}_조합.zip': f'{kind}S_{n}_조합.zip' for kind, n in [('T', 1), *[('T', n) for n in range(3, 9)], ('V', 1)]}
aihub_sizes = {fn: tuple(stats['size'][stat_index[f'aihub/{fn}']]) for fn in aihub_png}

rows = []
for label_zip, image_zip in PAIR.items():
    labeled = {fn for fn, z in label_zip_of.items() if z == label_zip}
    images = {fn for fn, z in image_zip_of.items() if z == image_zip and not fn.endswith('_index.png')}
    index = {fn for fn, z in image_zip_of.items() if z == image_zip and fn.endswith('_index.png')}
    on_disk = (images | index) & set(aihub_sizes)
    rows.append([f'{label_zip} / {image_zip}', f'{len(labeled):,}', f'{len(images):,}', len(index), len(labeled - images), len(images - labeled),
                 sum(aihub_sizes[fn] == (0, 0) for fn in on_disk),
                 sum(aihub_sizes[fn] not in ((0, 0), (IMG_W, IMG_H)) for fn in on_disk)])
rows.append(['**합계**', *[f'**{sum(int(str(r[i]).replace(",", "")) for r in rows):,}**' for i in range(1, 8)]])
show_table(['라벨 zip / 이미지 zip', 'JSON 쪽 이미지 파일명', '촬영 PNG', '_index.png', 'JSON만 있음', '촬영 PNG만 있음', '읽지 못한 PNG',
            f'{IMG_W}×{IMG_H}이 아닌 PNG'], rows)

#### 1-1-2 결과

- 8쌍 합계는 JSON 쪽 이미지 파일명 12,003개, 촬영 PNG 12,009장, `_index.png` 4,003장(조합마다 하나)이다.
- 읽지 못하는 PNG와 976×1280이 아닌 PNG는 없다.
- 짝이 맞지 않는 이미지 파일명은 세 가지다.

| 구분 | 이미지 파일명 | 상태 | 처리 |
|---|---|---|---|
| 다른 번호의 zip 쌍 | 24개 (TS_3~TS_6의 마지막 2조합씩) | 촬영 PNG는 TS_n에, JSON은 TL_(n+1)에 있다. 한 번호씩 밀려 들어갔다 | 둘 다 있으므로 그대로 쓴다 |
| JSON만 있음 | 6개 (TL_3, 조합 2개) | 촬영 PNG가 없다. 밀린 규칙대로면 금지된 TS_2에 있을 것으로 보인다 | 쓸 수 없다 (1-1-4) |
| 촬영 PNG만 있음 | 12개 (TS_1 6개, TS_7 6개, 조합 4개) | JSON이 없다. TS_1의 JSON은 밀린 규칙대로면 금지된 TL_2에 있을 것으로 보인다. TS_7의 JSON은 찾을 수 없다 | 2절에서 사진의 알약을 찾아 box를 그려 넣는다 |

- 표의 JSON만 있음 30개와 촬영 PNG만 있음 36개에는 다른 번호의 zip 쌍 24개가 양쪽에 한 번씩 들어 있다. 그래서 정말 짝이 없는 것은 6개와 12개다.

#### 1-1-3. `_index.png`

- 같은 조합의 90° 촬영 사진과 `_index.png`를 나란히 놓는다.
    - 왼쪽은 90° 촬영 사진, 가운데는 `_index.png`, 오른쪽은 `_index.png`에서 알약 하나 옆을 확대한 것이다.
- 제목에 두 사진의 픽셀 차이(R, G, B 차이의 평균)를 적는다.

In [ ]:
def example_box(photo):
    """그 사진의 첫 JSON(알약 폴더 순서, copy 파일은 뺌)에서 (bbox, K-코드)를 꺼낸다. 읽을 수 없거나 bbox가 숫자 4개가 아니면 None이다."""
    j = next((j for j in aihub_jsons if j['file_name'] == photo and ' copy' not in j['path'].name), None)
    if j is None or j['data'] is None or not j['data']['annotations']:
        return None
    bbox = j['data']['annotations'][0].get('bbox')
    if not (isinstance(bbox, list) and len(bbox) == 4 and all(isinstance(v, (int, float)) for v in bbox)):
        return None
    return bbox, j['data']['images'][0]['drug_N']


photo_set = set(aihub_files)
candidates = ((fn, fn.replace('_index.png', '_0_2_0_2_90_000_200.png')) for fn in sorted(aihub_index_files))
index_example, photo_example, (bbox, drug_code) = next((index_fn, photo, found) for index_fn, photo in candidates
                                                       if photo in photo_set and (found := example_box(photo)))
index_image, photo_image = Image.open(AIHUB_IMG_DIR / index_example).convert('RGB'), Image.open(AIHUB_IMG_DIR / photo_example).convert('RGB')
pixel_diff = np.abs(np.asarray(index_image, float) - np.asarray(photo_image, float)).mean()
x, y, w, h = bbox
zoom = (max(0, x - 40), max(0, y - 40), min(IMG_W, x + w + 260), min(IMG_H, y + h + 60))
fig, axes = plt.subplots(1, 3, figsize=(20, 8), layout='constrained', gridspec_kw={'width_ratios': [1, 1, 1.3]})
axes[0].imshow(photo_image)
axes[0].set_title(f'90° 촬영 사진\n{photo_example}', fontsize=10)
axes[1].imshow(index_image)
axes[1].set_title(f'_index.png (90° 사진과 픽셀 차이 평균 {pixel_diff:.2f})\n{index_example}', fontsize=10)
axes[2].imshow(index_image.crop(zoom))
axes[2].set_title(f'_index.png 확대 ({drug_code} 주변)', fontsize=10)
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('이미지 zip의 _index.png와 같은 조합의 90° 촬영 사진', fontsize=14)
save_fig(fig, '1-1b_index_png')
plt.show()

#### 1-1-3 결과

그림에서 보이는 것

- `_index.png`는 같은 조합의 90° 촬영 사진 위에 알약마다 K-코드와 제품명을 적은 사진이다.
- 두 사진의 픽셀 차이는 평균 1.03(0~255)으로, 글자 말고는 같은 사진이다.

정답 글자가 그려진 참고 사진이라 학습에 쓰지 않는다. 1-1-4에서 뺀다.

#### 1-1-4. 1차 정제

어떤 전처리로도 살릴 수 없는 파일을 먼저 뺀다. 사진은 보지 않고 파일의 짝만으로 정한다.

| 대상 | 처리 | 이유 |
|---|---|---|
| JSON만 있는 이미지 파일명의 JSON | 뺀다 | 그 이름의 촬영 PNG가 없어서 학습에 쓸 수 없다 |
| `_index.png` | 뺀다 | 정답이 그려진 참고 사진이다 (1-1-3) |
| 촬영 PNG만 있는 이미지 | 남긴다 | 2절에서 사진의 알약을 찾아 box를 그려 넣을 수 있다 |
| 다른 번호의 zip 쌍 | 남긴다 | JSON과 촬영 PNG가 둘 다 있다 |
| 읽지 못하는 JSON, `-오류` 폴더, copy 파일, bbox 오류 | 남긴다 | 촬영 PNG가 있어서, 사진으로 고칠 수 있는지 2절에서 본다 |

- 아래 셀은 1차 정제 후의 JSON과 촬영 PNG를 변수에 담고 수를 출력한다.

In [ ]:
aihub_jsons_1st = [j for j in aihub_jsons if j['file_name'] in set(aihub_files)]  # 1차 정제 후 JSON: 촬영 PNG가 있는 JSON만 남긴다
aihub_photos_1st = aihub_files  # 1차 정제 후 PNG: 촬영 PNG만 남긴다 (_index.png를 뺀다)
print(f'1차 정제: JSON {len(aihub_jsons):,}개 → {len(aihub_jsons_1st):,}개, PNG {len(aihub_png):,}장 → 촬영 PNG {len(aihub_photos_1st):,}장 '
      f'(조합 {len({combo_name(fn) for fn in aihub_photos_1st}):,}개)')


#### 1-1-4 결과

- JSON 45,934개 중 24개를 빼서 45,910개가 남는다. 뺀 24개는 조합 2개(`K-003544-012247-016548-029345`, `K-003544-012247-016548-029451`)의 사진 6장 몫이다.
- PNG 16,012장 중 `_index.png` 4,003장을 빼서 촬영 PNG 12,009장이 남는다. 조합 4,003개와 클래스 116개는 그대로다.

1-2부터는 1차 정제한 JSON 45,910개와 촬영 PNG 12,009장으로 본다.

### 1-2. 라벨 구조 비교

- AI Hub JSON(1차 정제 후, 바꾸기 전)과 대회 train JSON을 폴더 구조, JSON 키 구성, id 값, 클래스 표기로 비교한다.
    - JSON 키 구성은 최상위, images, annotations, categories의 키 이름을 순서대로 모아 비교한다.
    - `dl_idx`는 알약 식별자 필드이고, `drug_N`은 K-코드(`K-016548`)이다.
- `비고` 열에 대회 train과 다른 예외와 그 이유를 적는다.

In [ ]:
def key_structure(data):
    """JSON의 최상위, images, annotations, categories에 있는 키 이름을 순서대로 모은다."""
    return tuple(tuple(part) for part in (data, data['images'][0], data['annotations'][0], data['categories'][0]))


def json_name(j):
    """표에 적을 JSON 경로(`<조합 폴더>/<K-코드 폴더>/<파일 이름>`)를 만든다."""
    return f"`{j['combo_folder']}/{j['pill_folder']}/{unicodedata.normalize('NFC', j['path'].name)}`"


def broken_line(path):
    """읽지 못한 JSON에서 오류가 난 줄 번호와 그 줄의 내용을 찾는다."""
    text = path.read_text(encoding='utf-8')
    try:
        json.loads(text)
    except json.JSONDecodeError as e:
        return e.lineno, text.splitlines()[e.lineno - 1].strip()


def key_difference(data):
    """JSON 키 구성에서 Kaggle과 다른 곳을 문장으로 적는다."""
    notes = []
    for part, keys, kaggle_keys in zip(['최상위', 'images', 'annotations', 'categories'], key_structure(data), kaggle_structure):
        missing, extra = [k for k in kaggle_keys if k not in keys], [k for k in keys if k not in kaggle_keys]
        if missing:
            notes.append(f"{part}에 {', '.join(missing)} 키가 없다")
        if extra:
            notes.append(f"{part}에 {', '.join(extra)} 키가 더 있다")
        if not missing and not extra and keys != kaggle_keys:
            notes.append(f'{part}의 키 순서가 다르다')
    return ', '.join(notes)


def top_values(counter):
    """가장 많은 값 두 개를 '값 (개수)' 문자열로 만든다."""
    return ', '.join(f'{v} ({n})' for v, n in counter.most_common(2))


kaggle_json_paths = sorted(TRAIN_ANN_DIR.rglob('*.json'))
kaggle_structures = collections.Counter(key_structure(json.loads(p.read_text(encoding='utf-8'))) for p in kaggle_json_paths)
kaggle_structure = kaggle_structures.most_common(1)[0][0]
kaggle_anns = [ann for r in train_records.values() for ann in r['annotations']]

# 1차 정제 후 JSON(1-1)을 센다. JSON 내용으로 세는 항목은 읽을 수 있는 JSON만 센다
broken = [j for j in aihub_jsons_1st if j['error']]
valid = [j for j in aihub_jsons_1st if not j['error']]
images = [j['data']['images'][0] for j in valid]
outside_error = [j for j in valid if not j['pill_folder'].endswith('-오류')]
key_mismatch = [j for j in valid if key_structure(j['data']) != kaggle_structure]
aihub_ids = {
    'image': collections.Counter(image['id'] for image in images),
    'category': collections.Counter(a.get('category_id') for j in valid for a in j['data']['annotations']),
    'annotation': collections.Counter(a.get('id') for j in valid for a in j['data']['annotations']),
    'category_name': collections.Counter(j['data']['categories'][0]['name'] for j in valid),
}
dl_idx_other = collections.defaultdict(list)  # drug_N → dl_idx가 K-코드 - 1이 아닌 JSON의 images
for image in images:
    if int(image['dl_idx']) != int(image['drug_N'][2:]) - 1:
        dl_idx_other[image['drug_N']].append(image)
error_folder_jsons = [j for j in aihub_jsons_1st if j['pill_folder'].endswith('-오류')]
error_folders = sorted({f"{j['combo_folder']}/{j['pill_folder']}" for j in error_folder_jsons})
copy_jsons = [j for j in aihub_jsons_1st if ' copy' in j['path'].name]
copy_kinds = collections.defaultdict(list)  # 파일 이름에 붙은 ' copy', ' copy 2' → 그 JSON
for j in copy_jsons:
    name = unicodedata.normalize('NFC', j['path'].name)
    copy_kinds[name[name.index(' copy'):-len('.json')]].append(j)

broken_note = '; '.join(f'{json_name(j)}의 {line_no}번째 줄 `{line}`에 값이 없어 JSON으로 읽을 수 없다'
                        for j in broken for line_no, line in [broken_line(j['path'])])
dl_idx_note = f'나머지 {sum(len(v) for v in dl_idx_other.values())}개는 ' + '; '.join(
    f"{code}({imgs[0]['dl_name']})의 JSON {len(imgs)}개이다. dl_idx가 모두 {'/'.join(sorted({str(i['dl_idx']) for i in imgs}))}이고, "
    f"dl_mapping_code는 {'/'.join(sorted({i['dl_mapping_code'] for i in imgs}))}이다" for code, imgs in sorted(dl_idx_other.items()))
rows = [
    ['읽을 수 없는 JSON', '0', len(broken),
     f'{broken_note}. JSON 내용으로 세는 아래 행(폴더 이름 = drug_N, 키 구성, id, name, dl_idx)은 이 {len(broken)}개를 뺀 {len(valid)}개로 센다'],
    ['폴더 구조', '`<조합>_json/K-<코드>/<이미지명>.json`', '같음',
     f'예외 2가지: K-코드 뒤에 `-오류`가 붙은 폴더 {len(error_folders)}개 (`-오류` 폴더 JSON 행), '
     f'파일 이름에 ` copy`가 붙은 JSON {len(copy_jsons)}개 (copy 중복 파일 행). `-오류` 폴더 외의 JSON은 K-코드 폴더 이름 = drug_N이다 '
     f"({sum(j['pill_folder'] == j['data']['images'][0]['drug_N'] for j in outside_error)}/{len(outside_error)})"],
    ['JSON 키 구성 (최상위, images, annotations, categories)',
     f"한 가지 {kaggle_structures[kaggle_structure]}/{len(kaggle_json_paths)} (키 {', '.join(str(len(p)) for p in kaggle_structure)}개)",
     f'Kaggle과 같음 {len(valid) - len(key_mismatch)}/{len(valid)}',
     '; '.join(f"다른 {len(key_mismatch)}개: {json_name(j)}의 {key_difference(j['data'])}" for j in key_mismatch)],
    ['JSON 파일 수', len(kaggle_json_paths), len(aihub_jsons_1st), f'1-1의 1차 정제 후 수이다. 읽을 수 없는 JSON {len(broken)}개도 센다'],
    ['image id', f'이미지마다 다름 ({len({r["image"]["id"] for r in train_records.values()})}종)', top_values(aihub_ids['image']), ''],
    ['category_id', f'K-코드 숫자 ({len(train_classes)}종)', top_values(aihub_ids['category']), ''],
    ['annotation id', f'annotation마다 다름 ({len({a["id"] for a in kaggle_anns})}종)', top_values(aihub_ids['annotation']), ''],
    ['categories의 name', '제품명 (dl_name)', top_values(aihub_ids['category_name']), ''],
    ['dl_idx = K-코드', f"{sum(int(a['image']['dl_idx']) == a['category_id'] for a in kaggle_anns)}/{len(kaggle_anns)}",
     f"{sum(int(i['dl_idx']) == int(i['drug_N'][2:]) for i in images)}/{len(valid)}", ''],
    ['dl_idx = K-코드 - 1', '0', f"{len(valid) - sum(len(v) for v in dl_idx_other.values())}/{len(valid)}", dl_idx_note],
    ['-오류 폴더 JSON', '0', len(error_folder_jsons),
     f"폴더 {len(error_folders)}개: {', '.join(f'`{folder}`' for folder in error_folders)}. "
     f"읽을 수 없는 JSON {sum(j['error'] is not None for j in error_folder_jsons)}개도 이 폴더에 있다"],
    ['copy.json, copy 2.json파일', '0', len(copy_jsons),
     ', '.join(f'`<이미지명>{kind}.json` {len(js)}개' for kind, js in sorted(copy_kinds.items()))
     + '. 예: ' + ', '.join(json_name(js[0]) for kind, js in sorted(copy_kinds.items()))],
]
show_table(['항목', 'Kaggle train', 'AI Hub 1차 정제 후', '비고'], rows)

#### 1-2 결과

| 항목 | 대회 train | AI Hub (1차 정제 후) |
|---|---|---|
| 폴더 구조 | `<조합>_json/K-<코드>/<이미지명>.json` | 같음. 예외는 `-오류` 폴더 3개(JSON 9개)와 copy 파일 24개 |
| JSON 키 구성 | 한 가지 (763/763) | 45,909개 중 45,908개가 같음. 1개는 iscrowd 키가 없다 |
| 읽을 수 없는 JSON | 0개 | 1개 (`"area": ,`로 값이 빠짐, `-오류` 폴더 안) |
| image id, annotation id | 이미지·annotation마다 다름 | 모두 1 |
| category_id, categories.name | K-코드 숫자, 제품명 | 모두 1, `Drug` |
| dl_idx | = K-코드 (763/763) | = K-코드 − 1 (45,459개). 나머지 450개는 K-053384의 JSON으로, 모두 130376이다 |

- 대회 측이 id와 클래스 번호를 다시 매겼다는 대회 설명(`DATA.md`)과 맞는다. AI Hub 원본은 이 값이 채워져 있지 않다.
- 클래스를 dl_idx로 정하면 하나씩 어긋난다. drug_N(K-코드)으로 정해야 한다.
- iscrowd가 없는 JSON(`K-006835-020004-038972-053384` 90°의 K-020004)은 bbox가 정상이다. 같은 알약의 다른 각도 JSON에는 iscrowd가 있어서, 만들 때 생긴 누락으로 보인다.
- `-오류` 폴더 JSON 9개 중 8개는 bbox가 비어 있고, 1개는 읽을 수 없는 JSON이다. 사진을 보면 파일명의 알약 대신 다른 알약이 놓여 있다. (2-1)
- copy 파일은 `<이미지명> copy.json` 21개, `<이미지명> copy 2.json` 3개다. 한 사진에 같은 알약의 box가 2~3개 붙는 원인이다. (2-1)

### 1-3. Kaggle 형식으로 바꾸기

1차 정제 후 JSON에서 아래 키 8개만 바꾼다. 값이 맞는지 따로 분석하지 않아도 규칙으로 정해지는 키다.

| 키 | 바꾸는 값 |
|---|---|
| `images.id` | 이미지 파일명 순서로 1501부터. 같은 이미지의 JSON끼리 같다 |
| `annotations.id` | 같은 이미지 순서로, 이미지 안에서는 JSON 경로 순서로 5692부터 |
| `annotations.image_id` | 그 이미지의 `images.id` |
| `annotations.category_id`, `categories.id`, `images.dl_idx` | drug_N의 숫자 (`dl_idx`는 대회처럼 문자열) |
| `categories.name` | 제품명(`images.dl_name`) |
| `annotations.iscrowd` | 0. 키가 없던 JSON에는 대회 키 순서에 맞춰 넣는다 |

- 번호가 겹치지 않게 시작 값을 정했다. 대회 train과 test가 image id 1~1500을 쓰고, 대회 train의 annotation id 최댓값이 5691이다.
- bbox, area 등 나머지 키는 원본 값 그대로 둔다. bbox와 area 오류는 2절에서 고친다.
- 읽을 수 없는 JSON은 바꿀 수 없어서 내용 없이 남긴다. 그 알약은 정제 목록이 box를 새로 넣는다. (2-3)
- 바꾼 결과는 메모리에만 둔다.
- 같은 검사를 대회 train, AI Hub 바꾸기 전, AI Hub 바꾼 뒤에 똑같이 해서 비교한다. `그 밖의 키` 행은 키 8개 말고 나머지 값이 바꾸기 전과 같은지 센다.

In [ ]:
KAGGLE_ANN_KEYS = kaggle_structure[2]  # Kaggle annotations의 키 순서 (1-2)
# images.id는 이미지 파일명 순서로 1501부터, annotations.id는 같은 순서(이미지 안에서는 JSON 경로 순서)로 5692부터 준다
image_ids = {fn: image_id for image_id, fn in enumerate(sorted({j['file_name'] for j in valid}), start=1501)}
annotation_ids = {j['path']: ann_id for ann_id, j in enumerate(sorted(valid, key=lambda j: (j['file_name'], j['path'])), start=5692)}


def to_kaggle_keys(j):
    """JSON 하나에서 Kaggle과 형식이 다른 키 8개만 바꾼 새 내용을 만든다. 나머지 키는 원본 값 그대로 둔다."""
    image, ann, category = j['data']['images'][0], j['data']['annotations'][0], j['data']['categories'][0]
    code = int(image['drug_N'][2:])  # K-코드 숫자 (K-001900 → 1900)
    new_image = {**image, 'dl_idx': str(code), 'id': image_ids[j['file_name']]}
    new_ann = {**ann, 'iscrowd': 0, 'category_id': code, 'id': annotation_ids[j['path']], 'image_id': new_image['id']}
    new_ann = {**{key: new_ann[key] for key in KAGGLE_ANN_KEYS if key in new_ann}, **new_ann}  # iscrowd가 없던 JSON도 Kaggle 키 순서로 맞춘다
    new_category = {**category, 'id': code, 'name': image['dl_name']}
    return {**j['data'], 'images': [new_image], 'annotations': [new_ann], 'categories': [new_category]}


# 1차 정제 후 JSON 하나가 한 줄 (aihub_jsons_1st와 같은 구조). 읽을 수 없는 JSON은 바꿀 수 없어 data가 None인 채로 둔다
aihub_kaggle_jsons = [{**j, 'data': None if j['data'] is None else to_kaggle_keys(j)} for j in aihub_jsons_1st]
aihub_kaggle = {}  # 이미지 파일명 → 대회 train_records(load_image_records)와 같은 구조. 1-4부터 이미지 단위로 볼 때 쓴다
for j in aihub_kaggle_jsons:
    if j['data'] is not None:
        image = j['data']['images'][0]
        record = aihub_kaggle.setdefault(j['file_name'], {'image': image, 'annotations': []})
        record['annotations'].append({**j['data']['annotations'][0], 'image': image, 'json_path': str(j['path'])})

TARGET_KEYS = [('images', 'id'), ('images', 'dl_idx'), ('annotations', 'id'), ('annotations', 'image_id'),
               ('annotations', 'category_id'), ('annotations', 'iscrowd'), ('categories', 'id'), ('categories', 'name')]


def doc_field(d, part, key):
    """JSON 내용 d의 part(images, annotations, categories) 첫 항목에서 key 값을 꺼낸다. 키가 없으면 '(키 없음)'이다."""
    return d[part][0].get(key, '(키 없음)')


def other_values(d):
    """키 8개를 뺀 나머지 값을 모은다. 바꾸기 전과 바꾼 뒤가 같은지 볼 때 쓴다."""
    values = {('최상위', 'type'): d['type']}
    for part in ['images', 'annotations', 'categories']:
        values.update({(part, key): v for key, v in d[part][0].items() if (part, key) not in TARGET_KEYS})
    return values


def format_checks(docs):
    """Kaggle, 바꾸기 전, 바꾼 뒤에 똑같이 적용하는 검사. 항목 → 표에 적을 값."""
    n = len(docs)
    files = {doc_field(d, 'images', 'file_name') for d in docs}
    ids, ann_ids = [doc_field(d, 'images', 'id') for d in docs], [doc_field(d, 'annotations', 'id') for d in docs]
    codes = [int(doc_field(d, 'images', 'drug_N')[2:]) for d in docs]
    category_ids = [doc_field(d, 'annotations', 'category_id') for d in docs]
    iscrowd = collections.Counter(doc_field(d, 'annotations', 'iscrowd') for d in docs)
    return {
        'JSON 파일 (이미지)': f'{n}개 ({len(files)}장)',
        'images.id': f'{min(ids)}~{max(ids)} ({len(set(ids))}종 / 이미지 {len(files)}장)',
        'images.dl_idx': f"= K-코드 {sum(int(doc_field(d, 'images', 'dl_idx')) == c for d, c in zip(docs, codes))}/{n}",
        'annotations.id': f'{min(ann_ids)}~{max(ann_ids)} ({len(set(ann_ids))}종 / annotation {n}개)',
        'annotations.image_id': f"= images.id {sum(doc_field(d, 'annotations', 'image_id') == doc_field(d, 'images', 'id') for d in docs)}/{n}",
        'annotations.category_id': f'= drug_N 숫자 {sum(a == c for a, c in zip(category_ids, codes))}/{n} ({len(set(category_ids))}종)',
        'annotations.iscrowd': ', '.join(f'{v} ({count})' for v, count in iscrowd.most_common()),
        'categories.id': f"= category_id {sum(doc_field(d, 'categories', 'id') == a for d, a in zip(docs, category_ids))}/{n}",
        'categories.name': f"= dl_name {sum(doc_field(d, 'categories', 'name') == doc_field(d, 'images', 'dl_name') for d in docs)}/{n}",
        'JSON 키 구성': f'Kaggle과 같음 {sum(key_structure(d) == kaggle_structure for d in docs)}/{n}',
    }


kaggle_docs = [json.loads(p.read_text(encoding='utf-8')) for p in kaggle_json_paths]
before_docs = [j['data'] for j in valid]  # 1-2의 읽을 수 있는 JSON
after_docs = [j['data'] for j in aihub_kaggle_jsons if j['data'] is not None]  # before_docs와 같은 순서
checks = [format_checks(docs) for docs in (kaggle_docs, before_docs, after_docs)]
rows = [[item] + [c[item] for c in checks] for item in checks[0]]
rows[0][3] += f'. 읽을 수 없는 JSON {len(broken)}개는 바꾸지 못했다'
rows.append(['그 밖의 키 (bbox, area 등)', '-', '-',
             f'원본과 같음 {sum(other_values(b) == other_values(a) for b, a in zip(before_docs, after_docs))}/{len(after_docs)}'])
show_table(['항목', 'Kaggle train', 'AI Hub 바꾸기 전', 'AI Hub 바꾼 뒤'], rows)

#### 1-3 결과

- 읽을 수 있는 JSON 45,909개를 모두 바꿨다. 검사 8항목이 모두 대회 train과 같아졌고(45,909/45,909), 키 8개 밖의 값은 원본과 같다.
- 새 번호(images.id 1501~13497, annotations.id 5692~51600)는 대회 train(image id 14~1499, annotation id 50~5691)과 test(image id 1~1500)의 번호와 겹치지 않는다.
- 읽을 수 없는 1개는 그대로 남겼다. bbox와 area는 원본 그대로라 copy 파일, `-오류` 폴더, bbox 오류도 그대로 남아 있다.

이후 분석은 바꾼 라벨로 한다.

#### 1-3 예시: 같은 알약의 JSON 두 개

- 같은 알약 K-001900(보령부스파정 5mg)을 같은 각도(70°)에서 찍은 대회 JSON 하나와 AI Hub JSON 하나를 키마다 나란히 놓는다. AI Hub 쪽은 바꾸기 전과 바꾼 뒤를 함께 보인다.
    - 대회 쪽은 `K-001900-016548-019607-029451` 70° 사진의 K-001900 JSON이다. AI Hub 쪽은 같은 K-코드 폴더의 70° JSON 중 경로 순서로 첫 번째다. copy 파일과 `-오류` 폴더는 뺀다.
    - `바꿈` 열은 바꾸기 전과 바꾼 뒤의 값이 다른 키, `Kaggle과 비교` 열은 바꾼 뒤의 값이 대회 JSON과 같은지다. 긴 값은 앞 30자만 보인다.
- 같은 알약이므로 알약 정보는 같아야 한다. 형식을 다 맞췄다면, 대회 JSON과 다른 키는 사진마다 다른 값(파일명, bbox, area)과 새로 매긴 id만 남아야 한다.

In [ ]:
EXAMPLE_CODE = 'K-001900'
kaggle_json_path = (TRAIN_ANN_DIR / 'K-001900-016548-019607-029451_json' / EXAMPLE_CODE
                    / 'K-001900-016548-019607-029451_0_2_0_2_70_000_200.json')
kaggle_json = json.loads(kaggle_json_path.read_text(encoding='utf-8'))
aihub_example = next(j for j in aihub_jsons_1st if j['pill_folder'] == EXAMPLE_CODE
                     and '_70_' in j['path'].name and ' copy' not in j['path'].name)
aihub_json_path, aihub_json = aihub_example['path'], aihub_example['data']


def short(value, n=30):
    """표에 넣을 값을 앞 n자만 남긴다. 빈 문자열은 (빈 값)으로 쓴다."""
    text = str(value).replace('|', '/')
    if text == '':
        return '(빈 값)'
    return text if len(text) <= n else text[:n] + '…'

converted_example = next(j['data'] for j in aihub_kaggle_jsons if j['path'] == aihub_json_path)  # 1-2 예시 JSON을 바꾼 뒤
rows = [['파일', '-', short(kaggle_json_path.name, 40), short(aihub_json_path.name, 40), short(aihub_json_path.name, 40), '-', '-'],
        ['최상위', '(키 목록)', ', '.join(kaggle_json), ', '.join(aihub_json), ', '.join(converted_example), '',
         '같음' if list(kaggle_json) == list(converted_example) else '**다름**']]
for part in ['images', 'annotations', 'categories']:
    for key, kaggle_value in kaggle_json[part][0].items():
        before, after = aihub_json[part][0].get(key, '(키 없음)'), converted_example[part][0].get(key, '(키 없음)')
        rows.append([part, key, short(kaggle_value), short(before), short(after), '**바꿈**' if before != after else '',
                     '같음' if after == kaggle_value else '**다름**'])
n_keys = len(rows) - 2
n_changed, n_diff = sum(row[5] == '**바꿈**' for row in rows[2:]), sum(row[6] == '**다름**' for row in rows[2:])
rows.append([f'**바꾼 키 {n_changed}/{n_keys}개, 바꾼 뒤에도 Kaggle과 다른 키 {n_diff}/{n_keys}개**', '', '', '', '', '', ''])
show_table(['위치', '키', 'Kaggle train', 'AI Hub 바꾸기 전', 'AI Hub 바꾼 뒤', '바꿈', 'Kaggle과 비교'], rows)

#### 1-3 예시 결과

- 제품명, 각인, 모양, 색, 크기 같은 알약 정보는 두 JSON이 모두 같다. 61개 키 중 바꾼 키는 7개다. (이 JSON은 iscrowd가 원래 0이었다)
- 바꾼 뒤 대회 JSON과 다른 키는 7개만 남는다.

| 남은 차이 | 키 |
|---|---|
| 사진이 다른 값 | file_name, imgfile, bbox, area |
| 새로 매긴 번호 | images.id(2848), annotations.id(11080), annotations.image_id(2848) |

- 클래스 표기 4개(category_id, categories.id, categories.name, dl_idx)는 대회와 같아졌다. category_id가 1900이 되어, 같은 알약이 대회 train과 같은 번호로 묶인다. (6-1에서 mAP로 다시 확인)

### 1-4. 촬영 조건과 밝기

메타데이터의 촬영 조건 필드(1-4-1)와 이미지 채널 밝기(1-4-2)를 대회 train, test와 비교한다.

#### 1-4-1. 촬영 조건 필드

- 촬영 조건 필드를 이미지 단위로 세어 대회 train과 비교한다.
    - back_color(배경), light_color(조명), camera_la(카메라 위도, 70°/75°/90°), camera_lo(카메라 경도), drug_dir(알약 앞면/뒷면), drug_S(알약 상태), size, width×height
- 알약마다 JSON이 따로 있으므로, 이미지 하나의 값은 그 이미지 JSON들의 값을 모은 것이다. 알약마다 값이 다르면 `섞임`으로 센다.

In [ ]:
CONDITION_FIELDS = ['back_color', 'light_color', 'camera_la', 'camera_lo', 'drug_dir', 'drug_S', 'size']


def condition_counts(records):
    """촬영 조건 필드마다 이미지 단위 값의 개수를 센다. 같은 이미지의 알약끼리 값이 다르면 '섞임'으로 센다."""
    counts = {field: collections.Counter() for field in CONDITION_FIELDS + ['width×height']}
    for record in records.values():
        for field in CONDITION_FIELDS:
            values = {a['image'][field] for a in record['annotations']} or {record['image'][field]}
            counts[field][values.pop() if len(values) == 1 else '섞임'] += 1
        counts['width×height'][f"{record['image']['width']}×{record['image']['height']}"] += 1
    return counts


def count_text(counter, total):
    """값별 이미지 수를 많은 순서로 '값 n장 (비율)'로 적는다. 셋이 넘으면 나머지는 합친다."""
    items = counter.most_common()
    text = [f'{v} {n:,}장 ({n / total:.1%})' for v, n in items[:3]]
    if len(items) > 3:
        text.append(f'그 외 {sum(n for _, n in items[3:]):,}장')
    return ', '.join(text)


train_conditions = condition_counts(train_records)
aihub_conditions = condition_counts(aihub_kaggle)
show_table(['필드', f'대회 train ({len(train_records)}장)', f'AI Hub ({len(aihub_kaggle):,}장)'],
           [[field, count_text(train_conditions[field], len(train_records)), count_text(aihub_conditions[field], len(aihub_kaggle))]
            for field in CONDITION_FIELDS + ['width×height']])

#### 1-4-1 결과

| 필드 | 대회 train (232장) | AI Hub (11,997장) |
|---|---|---|
| back_color | 연회색 배경 100% | 연회색 배경 11,996장, 검은색 배경 1장 |
| light_color, camera_lo, drug_S, size, 해상도 | 주백색, 0, 정상알약, 200, 976×1280 (모두 100%) | 같음 (모두 100%) |
| camera_la | 90° 36.6%, 75° 32.3%, 70° 31.0% | 세 각도 3,999장씩 (33.3%) |
| drug_dir 뒷면 | 4장 (1.7%) | 6장 (0.1%) |

- 촬영 조건은 두 데이터가 같다. AI Hub는 조합마다 세 각도를 모두 찍었다. (4-2)
- "검은색 배경" 1장(`K-012247-016551-024850-029451_0_0_0_2_90_000_200.png`)은 메타데이터만 틀렸다.
    - 사진은 같은 조합의 75° 사진과 같은 연회색 배경이다.
    - 파일명의 배경 자리도 다른 사진(`_0_2_`)과 달리 `_0_0_`이다.

이 1장도 그대로 쓰고, 정제 목록(2-3)에서 back_color를 연회색 배경으로 고친다.

#### 1-4-2. 채널 밝기

- 이미지마다 R, G, B 채널의 평균 밝기(0~255)를 구해 대회 train, test, AI Hub로 나눠 비교한다. (0-7에서 계산한 값)
    - 세 묶음의 이미지 수가 크게 달라서(232장, 842장, 약 12,000장) 넓이가 1인 밀도로 그린다.
- 아래 표는 묶음(대회 train, test, AI Hub 이미지 zip 8개)별 중앙값이다.
    - 밝기는 R, G, B 평균의 평균(0~255)이다. `test 대비`는 test 밝기 중앙값과의 차이를 비율로 적은 것이다.

In [ ]:
groups = {'대회 train': 'train', 'test': 'test', 'AI Hub': 'aihub'}
group_means = {name: stats['mean_rgb'][[k.startswith(prefix + '/') and not k.endswith('_index.png') for k in stats['keys']]]
               for name, prefix in groups.items()}
fig, axes = plt.subplots(1, 3, figsize=(20, 5), layout='constrained', sharey=True)
bins = np.linspace(min(m.min() for m in group_means.values()) - 2, max(m.max() for m in group_means.values()) + 2, 60)
for c, (ax, channel) in enumerate(zip(axes, ['R', 'G', 'B'])):
    for (name, means), color in zip(group_means.items(), ['tab:blue', 'tab:orange', 'tab:green']):
        values = means[:, c]
        ax.hist(values, bins=bins, density=True, histtype='step', linewidth=2, color=color,
                label=f'{name} {len(values):,}장 (중앙값 {np.median(values):.1f})')
        ax.axvline(np.median(values), color=color, linestyle='--', linewidth=1)
    ax.set_xlabel(f'{channel} 채널 평균 밝기 (0~255)')
    ax.set_title(f'{channel} 채널')
    ax.legend(fontsize=9)
axes[0].set_ylabel('밀도 (넓이 합 = 1)')
fig.suptitle('이미지별 채널 평균 밝기 분포 (점선 = 중앙값, 분포가 겹칠수록 촬영 조건이 비슷함)', fontsize=14)
save_fig(fig, '1-4_channel_means')
plt.show()

- 묶음별 밝기 중앙값을 표로 본다.

In [ ]:
def brightness_row(name, means):
    med = np.median(means, axis=0)
    gray = means.mean(axis=1)
    return [name, f'{len(means):,}장', *[f'{v:.1f}' for v in med], f'{np.median(gray):.1f}',
            f'{(np.median(gray) / np.median(test_gray) - 1):+.1%}', f'{np.percentile(gray, 10):.1f} ~ {np.percentile(gray, 90):.1f}']


test_gray = group_means['test'].mean(axis=1)
rows = [brightness_row('대회 train', group_means['대회 train']), brightness_row('test', group_means['test'])]
for image_zip in PAIR.values():
    idx = [stat_index[f'aihub/{fn}'] for fn, z in sorted(image_zip_of.items()) if z == image_zip and not fn.endswith('_index.png')]
    if idx:
        rows.append(brightness_row(f'AI Hub {image_zip}', stats['mean_rgb'][idx]))
show_table(['묶음', '이미지', 'R 중앙값', 'G 중앙값', 'B 중앙값', '밝기 중앙값', 'test 대비', '밝기 10%~90% 범위'], rows)

#### 1-4-2 결과

| 묶음 | 밝기 중앙값 | test 대비 |
|---|---|---|
| 대회 train | 129.2 | +6.0% |
| test | 122.0 | 0% |
| AI Hub zip 8개 | 116.3 (VS_1) ~ 154.9 (TS_1) | −4.6% ~ +27.0% |

그림에서 보이는 것

- AI Hub의 채널 중앙값(R 111.2, G 126.8, B 146.5)은 대회 train과 test 사이에 있다.
- test가 가장 어둡다. R 106.4, G 115.4로 대회 train(114.4, 129.4)보다 낮다. B는 세 묶음이 144.5~147.4로 비슷하다.
- AI Hub 분포(초록)는 대회 train(파랑)과 test(주황)의 범위를 모두 덮는다. test에만 있는 밝기 구간이 없다.
    - G 채널은 세 묶음 모두 봉우리가 둘이다. zip마다 밝기가 달라서(표), 촬영 회차마다 조명이 달랐던 것으로 보인다. 어느 묶음에만 있는 특징이 아니다.

AI Hub를 섞으면 학습 데이터의 밝기 범위가 test를 덮는다. 밝기를 맞추는 전처리는 하지 않는다.

### 1-5. 샘플 이미지

- AI Hub 이미지 zip 8개에서 1장씩, 라벨 오류가 없는 이미지를 무작위로 골라 원본 box를 겹쳐 그린다. (seed 고정)
    - 라벨 오류가 없는 이미지: 2절에서 다룰 문제(읽을 수 없는 JSON, `-오류` 폴더, copy 파일, bbox 오류, 라벨 누락)가 하나도 없는 이미지다.
    - box 위 글자는 K-코드와 메타데이터의 각인이다. 사진 속 알약의 글자와 맞대어 보면 box가 맞는 알약인지 알 수 있다.
- box가 알약을 빈틈없이 감싸는지, 장면이 대회 사진과 같은지 본다.

In [ ]:
jsons_of = collections.defaultdict(list)  # 이미지 파일명 → 그 이미지의 JSON (1-3에서 키를 바꾼 것)
for j in aihub_kaggle_jsons:
    jsons_of[j['file_name']].append(j)


def has_label_problem(fn):
    """2절에서 다룰 라벨 문제(읽을 수 없는 JSON, -오류 폴더, copy 파일, bbox 오류, 라벨 누락)가 있는 이미지인지 본다."""
    js = jsons_of[fn]
    if any(j['data'] is None or j['pill_folder'].endswith('-오류') or ' copy' in j['path'].name for j in js):
        return True
    boxes = [j['data']['annotations'][0]['bbox'] for j in js]
    if any(not isinstance(b, list) or len(b) != 4 or b[2] <= 0 or b[3] <= 0 for b in boxes):
        return True
    return len({j['data']['annotations'][0]['category_id'] for j in js}) < len(pill_codes(fn))


rng = np.random.default_rng(SEED)
look_files = []
for image_zip in PAIR.values():
    pool = sorted(fn for fn, z in image_zip_of.items() if z == image_zip and fn in aihub_kaggle and not has_label_problem(fn))
    if pool:  # 아직 받지 않은 zip은 건너뛴다
        look_files.append(pool[rng.integers(len(pool))])

fig, axes = plt.subplots(2, 4, figsize=(24, 18), layout='constrained')
for ax, fn in zip(axes.flat, look_files):
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
    for ann in aihub_kaggle[fn]['annotations']:
        x, y, w, h = ann['bbox']
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2.5))
        ax.text(x, y - 8, pill_label(ann), fontsize=9, va='bottom', bbox=dict(facecolor='lime', alpha=0.85, pad=1.5, linewidth=0))
    ax.set_title(f'{image_zip_of[fn]}\n{fn}', fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('AI Hub 이미지 zip마다 무작위 1장 (연두 box = 1-3 라벨의 box, 글자 = K-코드(각인))', fontsize=14)
save_fig(fig, '1-5_aihub_samples')
plt.show()

#### 1-5 결과

그림에서 보이는 것

- 8장 모두 box가 알약 하나를 빈틈없이 감싼다.
- box 위 각인이 사진 속 알약의 각인과 맞는다. (예: TS_3의 K-023223 V100, TS_6의 K-011220 Bayer, TS_8의 K-015710 Entelon150)
- 장면은 대회 사진과 같다. 연회색 바닥에 알약 3~4개가 서로 떨어져 놓여 있다.
- TS_1 사진은 다른 zip보다 밝고, 조명 원 밖의 모서리가 하얗다. (1-4-2 표의 +27.0%)
- TS_6의 K-031885처럼 파란 알약은 바닥과 색이 비슷하다.

라벨 오류가 없는 AI Hub box는 대회 box처럼 알약을 감싼다. 여백은 3-1-2에서 수치로 비교한다.

### 1-6. 대회 데이터와 겹치는 사진

AI Hub 사진 중에 대회 train, test와 같은 사진이 있는지 세 가지로 찾는다. AI Hub끼리도 다른 조합에 같은 사진이 있는지 본다.

| 방법 | 보는 것 |
|---|---|
| 파일 이름 | 대회 train 이미지와 조합, 라벨이 누락된 대회 train 이미지의 이름이 AI Hub에 있는지 |
| 파일 내용(MD5) | 1바이트라도 다르면 값이 바뀌는 해시가 같은 파일 |
| 축소 그림 | 다시 저장해 파일 내용이 달라져도, 같은 사진이면 32×40 회색 축소 그림이 거의 같다 |

- 축소 그림은 RMS 차이(픽셀 밝기 차이를 제곱해 평균한 뒤 제곱근, 0~255)로 비교한다. 0이면 똑같고, 같은 사진을 다시 저장하면 1~2 정도다.
- AI Hub끼리는 같은 조합의 각도별 사진이 원래 비슷하므로, 다른 조합의 사진끼리만 비교한다.
- `_index.png`는 90° 사진에 글자를 적은 것이라(1-1-3) 모든 비교에서 뺀다.

In [ ]:
def thumbs(prefix):
    """0번에서 계산한 회색 축소 그림을 묶음별로 꺼낸다. _index.png는 뺀다. (파일명 목록, float32 행렬)"""
    idx = [i for i, k in enumerate(stats['keys']) if k.startswith(prefix + '/') and not k.endswith('_index.png')]
    return [stats['keys'][i].split('/', 1)[1] for i in idx], stats['thumb'][idx].reshape(len(idx), -1).astype(np.float32)


def nearest(a, b, groups=None, chunk=2000):
    """a의 축소 그림마다 b에서 RMS 차이가 가장 작은 그림의 번호와 차이를 찾는다. groups를 주면(a와 b가 같을 때) 같은 묶음은 뺀다."""
    b_sq = (b ** 2).sum(axis=1)
    best_idx, best_rms = np.zeros(len(a), int), np.zeros(len(a))
    for start in range(0, len(a), chunk):
        block = a[start:start + chunk]
        d2 = (block ** 2).sum(axis=1)[:, None] + b_sq[None, :] - 2 * block @ b.T  # 제곱 거리 = |a|² + |b|² - 2a·b
        if groups is not None:
            d2[groups[start:start + chunk, None] == groups[None, :]] = np.inf
        best_idx[start:start + chunk] = d2.argmin(axis=1)
        best_rms[start:start + chunk] = np.sqrt(np.maximum(d2.min(axis=1), 0) / a.shape[1])
    return best_idx, best_rms


train_names, train_thumbs = thumbs('train')
test_names, test_thumbs = thumbs('test')
aihub_names, aihub_thumbs = thumbs('aihub')
near = {
    '대회 train': nearest(train_thumbs, aihub_thumbs),
    'test': nearest(test_thumbs, aihub_thumbs),
    'AI Hub 다른 조합 사진': nearest(aihub_thumbs, aihub_thumbs, groups=np.array([combo_name(fn) for fn in aihub_names])),
}
md5_of = collections.defaultdict(list)
for key, digest in zip(stats['keys'], stats['md5']):
    md5_of[digest].append(key)
same_md5 = {group: sum(any(k.startswith('aihub/') for k in keys) and any(k.startswith(group + '/') for k in keys)
                       for keys in md5_of.values()) for group in ['train', 'test']}
aihub_dup_md5 = sum(sum(k.startswith('aihub/') and not k.endswith('_index.png') for k in keys) > 1 for keys in md5_of.values())
missing_train = [fn for fn in train_files if len({a['category_id'] for a in train_records[fn]['annotations']}) < len(pill_codes(fn))]
aihub_photo_names = set(aihub_photos_1st)  # 1차 정제 후 촬영 PNG (1-1)
aihub_combos = {combo_name(fn) for fn in aihub_photo_names}
NEAR_RMS = 2  # 이 값보다 작으면 같은 사진 후보로 센다 (같은 사진에 글자만 적은 _index.png와 90° 사진이 0.6 정도, 1-1)

rows = [
    ['파일 이름: 대회 train 이미지', f'{len(set(train_files) & aihub_photo_names)}/{len(train_files)}'],
    ['파일 이름: 대회 train 조합', f'{len(set(train_combos) & aihub_combos)}/{len(train_combos)}'],
    ['파일 이름: 라벨이 누락된 대회 train 이미지', f'{len(set(missing_train) & aihub_photo_names)}/{len(missing_train)}'],
    ['파일 내용(MD5)이 같은 사진: AI Hub와 대회 train', same_md5['train']],
    ['파일 내용(MD5)이 같은 사진: AI Hub와 test', same_md5['test']],
    ['파일 내용(MD5)이 같은 사진: AI Hub끼리', aihub_dup_md5],
]
for name, (_, rms) in near.items():
    rows.append([f'축소 그림: {name}에서 가장 가까운 AI Hub 사진의 RMS 차이',
                 f'최소 {rms.min():.1f}, 중앙값 {np.median(rms):.1f}, {NEAR_RMS} 미만 {(rms < NEAR_RMS).sum()}장'])
show_table(['확인 항목', '결과'], rows)

- 축소 그림이 가장 가까운 쌍을 묶음마다 4쌍씩, RMS 차이가 작은 순서로 나란히 그린다.
    - 윗줄은 대회 train 또는 test 사진(AI Hub끼리는 앞 사진), 아랫줄은 가장 가까운 AI Hub 사진이다.
- 같은 사진(알약 배치와 위치가 같음)인지, 비슷한 장면의 다른 사진인지 본다.

In [ ]:
pair_sources = [('대회 train', train_names, TRAIN_IMG_DIR), ('test', test_names, TEST_IMG_DIR), ('AI Hub 다른 조합 사진', aihub_names, AIHUB_IMG_DIR)]
fig, axes = plt.subplots(2, 12, figsize=(30, 7.5), layout='constrained')
col = 0
for name, names, folder in pair_sources:
    idx, rms = near[name]
    shown = []
    for i in np.argsort(rms):
        pair = frozenset((names[i], aihub_names[idx[i]]))
        if name == 'AI Hub 다른 조합 사진' and pair in shown:  # 같은 쌍이 순서만 바뀌어 두 번 나오는 것을 뺀다
            continue
        shown.append(pair)
        axes[0, col].imshow(Image.open(folder / names[i]))
        axes[0, col].set_title(f'{name}\n{names[i][:34]}', fontsize=8)
        axes[1, col].imshow(Image.open(AIHUB_IMG_DIR / aihub_names[idx[i]]))
        axes[1, col].set_title(f'AI Hub (RMS 차이 {rms[i]:.1f})\n{aihub_names[idx[i]][:34]}', fontsize=8)
        col += 1
        if len(shown) == 4:
            break
for ax in axes.flat:
    ax.axis('off')
fig.suptitle('축소 그림이 가장 가까운 사진 쌍 (윗줄 = 찾는 사진, 아랫줄 = 가장 가까운 AI Hub 사진, RMS 차이가 작은 순서)', fontsize=14)
save_fig(fig, '1-6_nearest_pairs')
plt.show()

#### 1-6 결과

| 확인 | 결과 |
|---|---|
| 파일 이름이 같은 사진 | 대회 train 이미지 0/232, 대회 train 조합 0/114, 라벨이 누락된 대회 train 이미지 0/8 |
| MD5가 같은 파일 | AI Hub와 대회 train 0, AI Hub와 test 0, AI Hub끼리 0 |
| 가장 가까운 AI Hub 사진의 RMS 최솟값 | 대회 train 2.0, test 1.9, AI Hub의 다른 조합 0.4 |

그림에서 보이는 것

- 대회 train, test와 가장 가까운 AI Hub 사진(RMS 1.9~2.0)은 알약 하나만 바꿔 같은 자리에서 찍은 다른 사진이다.
- RMS 0.4인 AI Hub 쌍은 같은 장면을 두 번 찍은 사진이 다른 조합 이름으로 들어간 것이다.
    - 조합 `K-012247-016548-033009-044199`와 `K-012247-016551-033009-044199`, `K-004543-016548-023203-024850`과 `K-016548-016551-023203-024850`의 각도별 사진 3쌍씩, 모두 6쌍이다.
    - 파일 내용이 달라 MD5로는 찾지 못한다. 원본 해상도로 비교하면 알약 배치와 위치가 같고, 픽셀 차이는 평균 2.3(0~255)의 노이즈뿐이다. (따로 확인)
    - 두 쌍 모두 사진에 K-016551이 없다. 이름에 K-016551이 든 쪽이 틀렸다.
- RMS 1.2~2.0인 나머지 AI Hub 쌍은 알약 하나만 바꿔 찍은 다른 사진이다.

누수로 빼는 AI Hub 사진은 없다. 같은 장면 6쌍은 이름이 틀린 쪽 6장만 정제 목록(2-3)에서 뺀다.

### 1-7. 정리

#### 확인한 것

- 파일: 허용 zip 8쌍에서 JSON 45,934개, 촬영 PNG 12,009장, `_index.png` 4,003장을 받았다. 깨진 PNG는 없고 해상도는 모두 976×1280이다.
- 짝: JSON만 있는 이미지 6장, 촬영 PNG만 있는 이미지 12장이 있다. 24장은 번호가 하나 밀린 zip 쌍에 들어 있다.
- 라벨 형식: 폴더 구조와 키 구성은 대회와 같다. 다른 것은 id(모두 1)와 클래스 표기(category_id 1, name `Drug`, dl_idx = K-코드 − 1)다.
- 예외 파일: 읽을 수 없는 JSON 1개, iscrowd가 없는 JSON 1개, `-오류` 폴더 3개(JSON 9개), copy 파일 24개가 있다.
- 촬영 조건: 배경, 조명, 해상도가 대회와 같다. 밝기 분포는 대회 train과 test 범위를 덮는다.
- 누수: 대회 train·test와 같은 사진은 없다.
    - 대회 train의 이미지와 조합이 허용 zip에 하나도 없다. 대회 데이터는 금지된 TL_2·TS_2에서 나온 것으로 보인다.
    - 그래서 대회 train의 라벨 누락 8장은 AI Hub로 복구할 수 없다. 이 8개는 대회 train 정제 목록이 box를 넣었다.
- AI Hub 안에 같은 장면이 다른 조합 이름으로 든 쌍이 6쌍 있다.

#### 정한 것

- 조합 데이터 허용 zip 8쌍만 쓴다. TL_2·TS_2와 단일 데이터는 쓰지 않는다. (1-1-1)
- 1차 정제로 JSON만 있는 이미지의 JSON 24개와 `_index.png` 4,003장을 뺀다. (1-1-2~1-1-4)
- 라벨은 키 8개만 대회 규칙으로 바꾸고, 클래스는 drug_N으로 정한다. (1-2, 1-3)
- 밝기 전처리 없이 대회 사진과 그대로 섞는다. (1-4, 1-5)
- 같은 장면 6쌍은 이름이 틀린 쪽 6장을 뺀다. (1-6, 정제 목록 2-3)

#### 뒤 섹션에서 볼 것

- 촬영 PNG만 있는 12장, 읽을 수 없는 JSON, `-오류` 폴더, copy 파일, bbox 오류는 2절에서 규모를 세고, 정제 목록으로 고친다.
- "검은색 배경" 1장의 메타데이터는 정제 목록(2-3)에서 고친다.

## 2. 라벨 품질과 정제

1-3에서 형식을 바꾼 AI Hub 라벨의 오류를 세고, 정제 목록을 적용해 정제 라벨을 만든다. 평가 지표 mAP@[0.75:0.95]는 IoU 0.75 미만인 box를 오답으로 보므로, 라벨 box가 조금만 틀려도 학습과 평가가 모두 흔들린다.

| 단계 | 하는 일 | 이유 |
|---|---|---|
| 2-1 | 라벨 누락과 클래스 불일치 | 파일명의 알약마다 쓸 수 있는 box가 하나씩 있는지 본다 |
| 2-2 | 좌표 오류 | 좌표가 틀린 box가 얼마나 있는지 본다 |
| 2-3 | 정제 목록 적용 | 레포의 정제 목록을 적용해 정제 라벨을 만든다. 3절부터 이 라벨로 본다 |
| 2-4 | 뺀 이미지 | 정제로도 살리지 못해 뺀 이미지 35장을 본다 |
| 2-5 | 정리 | 확인한 것과 정한 것 |

오류를 어떻게 고쳤는지(사진에서 알약 찾기, 오타 고치기, 갤러리로 눈 확인)는 이 노트북이 아니라 [`label_fix/01_aihub_label_fix.ipynb`](label_fix/01_aihub_label_fix.ipynb)(이하 정제 노트북)에 있다. 그 결과가 `data/label_fix/label_fixes.csv`의 aihub 줄이다.

### 2-1. 라벨 누락과 클래스 불일치

- 1차 정제 후 촬영 PNG마다 파일명의 K-코드와, 그 이미지 파일명을 가리키는 JSON의 category_id를 맞대어 본다.
    - 파일명의 K-코드 하나가 사진 속 알약 하나, 곧 객체(annotation) 하나다.
    - 쓸 수 있는 bbox는 값이 4개이고 너비와 높이가 0보다 큰 bbox다.
- 라벨 누락은 파일명에는 있는데 쓸 수 있는 bbox가 없는 알약이다. 원인을 아래처럼 나눈다.

| 원인 | 뜻 |
|---|---|
| JSON 없음: 촬영 PNG만 있는 이미지 | 조합의 JSON 폴더가 통째로 없다 (1-1-2) |
| JSON 없음: copy 파일이 있는 알약 폴더 | 폴더의 JSON 3개 중 둘 이상이 같은 각도를 가리켜, 이 각도의 JSON이 없다 |
| JSON 없음: 그 밖 | 위 두 경우가 아닌데 이 각도의 JSON이 없다 |
| bbox 빈 값 | bbox가 `[]`다 |
| 숫자가 붙은 bbox | bbox 값이 4개가 아니다 (예: `[121125, 333, 475]`) |
| `-오류` 폴더 | JSON이 `K-<코드>-오류` 폴더에 있다. bbox도 비었지만 이 원인으로 센다 |
| 읽을 수 없는 JSON | JSON 문법이 깨졌다 |

- 클래스 불일치는 이미지 한 장 기준으로 센다.
    - 파일명에 없는 K-코드: 파일명에 없는 K-코드의 JSON이 그 이미지를 가리킨다.
    - 같은 K-코드의 box가 파일명보다 많음: copy 파일 때문에 한 사진에 같은 알약의 box가 2~3개 붙는다.
        - 예: `K-001900-003544-004543-016548_json/K-016548/`의 `_70`, `_75`, `_75 copy` JSON 3개는 사진 3장에 1개, 2개, 0개(90°)로 나뉜다.
- 표 읽는 법
    - `객체 수`는 라벨 누락 행에서는 빠진 알약 수, 클래스 불일치 행에서는 남는 box 수다.
    - `확인할 파일 경로`는 행마다 처음, 가운데, 끝에서 고른 3개다. JSON 없음 행은 있어야 할 JSON의 경로다.

In [ ]:
def usable_bbox(bbox):
    """학습에 쓸 수 있는 bbox인지 본다. 값이 4개이고 너비와 높이가 0보다 커야 한다."""
    return isinstance(bbox, list) and len(bbox) == 4 and bbox[2] > 0 and bbox[3] > 0


def bbox_problem(j):
    """JSON 하나의 bbox를 쓸 수 없는 원인을 돌려준다. 쓸 수 있으면 None이다. 원인이 둘 이상이면 앞의 것으로 센다."""
    if j['data'] is None:
        return '읽을 수 없는 JSON'
    if j['pill_folder'].endswith('-오류'):
        return '-오류 폴더'
    bbox = j['data']['annotations'][0]['bbox']
    if usable_bbox(bbox):
        return None
    if bbox == []:
        return 'bbox 빈 값'
    if not isinstance(bbox, list) or len(bbox) != 4:
        return '숫자가 붙은 bbox'
    return 'bbox 너비·높이 0 이하'


def json_code(j):
    """JSON의 K-코드 숫자. 읽을 수 없는 JSON은 알약 폴더 이름(`K-016551-오류`)에서 읽는다."""
    return int(j['pill_folder'][2:8]) if j['data'] is None else j['data']['annotations'][0]['category_id']


missing_labels = []  # (이미지, 빠진 K-코드, 원인, JSON 경로, bbox). JSON 없음이면 경로와 bbox가 None이다
extra_labels = []  # (이미지, 파일명에 없는 K-코드, JSON 경로)
surplus_labels = []  # (이미지, K-코드, 남는 box 수, 그 K-코드의 쓸 수 있는 JSON 경로)
for fn in aihub_photos_1st:
    named = collections.Counter(pill_codes(fn))
    jsons_by_code = collections.defaultdict(list)
    for j in jsons_of.get(fn, []):  # jsons_of: 1-5에서 만든 이미지 파일명 → 1-3의 JSON
        jsons_by_code[json_code(j)].append(j)
    for code in sorted(set(named) | set(jsons_by_code)):
        js = jsons_by_code[code]
        if code not in named:
            extra_labels += [(fn, code, j['path']) for j in js]
            continue
        usable = [j for j in js if bbox_problem(j) is None]
        unusable = [j for j in js if bbox_problem(j) is not None]
        for k in range(named[code] - len(usable)):  # 쓸 수 없는 JSON의 원인을 먼저 붙이고, JSON이 모자라면 JSON 없음이다
            if k < len(unusable):
                j = unusable[k]
                bbox = None if j['data'] is None else j['data']['annotations'][0]['bbox']
                missing_labels.append((fn, code, bbox_problem(j), j['path'], bbox))
            else:
                missing_labels.append((fn, code, 'JSON 없음', None, None))
        if len(usable) > named[code]:
            surplus_labels.append((fn, code, len(usable) - named[code], [j['path'] for j in usable]))

copy_pill_folders = {(j['combo_folder'], j['pill_folder']) for j in aihub_kaggle_jsons if ' copy' in j['path'].name}
MISSING_GROUPS = ['JSON 없음: 촬영 PNG만 있는 이미지 (1-1)', 'JSON 없음: copy 파일이 있는 알약 폴더 (1-2)', 'JSON 없음: 그 밖',
                  'bbox 빈 값', '숫자가 붙은 bbox', '-오류 폴더', '읽을 수 없는 JSON', 'bbox 너비·높이 0 이하']


def missing_group(m):
    """라벨 누락 하나가 들어갈 표의 행. JSON 없음은 이유에 따라 셋으로 나눈다."""
    fn, code, cause = m[:3]
    if cause != 'JSON 없음':
        return cause
    if not jsons_of.get(fn):
        return MISSING_GROUPS[0]
    if (f'{combo_name(fn)}_json', f'K-{code:06d}') in copy_pill_folders:
        return MISSING_GROUPS[1]
    return MISSING_GROUPS[2]


def missing_path(m):
    """라벨 누락의 JSON 경로. JSON 없음이면 있어야 할 경로이다."""
    fn, code, _, path, _ = m
    return path or AIHUB_RAW_ANN_DIR / f'{combo_name(fn)}_json' / f'K-{code:06d}' / f"{fn[:-len('.png')]}.json"


def repo_path(path):
    """표에 적을 레포 최상위 기준 경로."""
    return f"`{unicodedata.normalize('NFC', path.relative_to(PROJECT_ROOT).as_posix())}`"


def spread(items, n=3):
    """목록의 처음, 가운데, 끝에서 고르게 n개를 고른다."""
    return [items[i] for i in sorted(set(np.linspace(0, len(items) - 1, min(n, len(items))).round().astype(int)))]


rows = []
for group in MISSING_GROUPS:
    items = [m for m in missing_labels if missing_group(m) == group]
    if items:
        rows.append([group, len(items), len({m[0] for m in items}), len({m[1] for m in items}),
                     '<br>'.join(repo_path(missing_path(m)) for m in spread(items))])
missing_files = sorted({m[0] for m in missing_labels})
per_image = collections.Counter(collections.Counter(m[0] for m in missing_labels).values())
rows.append(['**라벨 누락 합계**', f'**{len(missing_labels)}**', f'**{len(missing_files)}장**', f"**{len({m[1] for m in missing_labels})}**", ''])
rows.append(['이미지당 빠진 알약 수', '', ', '.join(f'{k}개 {v}장' for k, v in sorted(per_image.items())), '', ''])
rows.append(['클래스 불일치: 파일명에 없는 K-코드', len(extra_labels), len({e[0] for e in extra_labels}), len({e[1] for e in extra_labels}),
             '<br>'.join(repo_path(e[2]) for e in spread(extra_labels))])
rows.append(['클래스 불일치: 한 이미지에 같은 K-코드의 box가 파일명보다 많음 (copy 파일이 있는 알약 폴더)', sum(s[2] for s in surplus_labels), len({s[0] for s in surplus_labels}),
             len({s[1] for s in surplus_labels}), '<br>'.join(repo_path(p) for s in spread(surplus_labels, n=2) for p in s[3])])
show_table(['원인', '객체 수', '이미지 수', '클래스 수', '확인할 파일 경로'], rows)

- 위 표의 행마다 예시 이미지를 최대 2장씩 그린다.
    - 연두 box: 그 이미지를 가리키는 JSON 중 쓸 수 있는 bbox
    - 주황 box: 한 이미지에 파일명보다 많이 붙은 K-코드의 box. 위에 box를 가져온 JSON 이름(각도와 copy 부분)을 적는다.
    - 제목 셋째 줄부터: 라벨 누락 행은 빠진 알약의 K-코드와 각인, box가 많은 행은 그 K-코드의 box 수와 파일명에 적힌 수
- 빠진 알약이 사진에 실제로 있는지, 남는 box가 어디에 걸치는지 본다.

In [ ]:
aihub_example = {}  # K-코드 → 바꾼 AI Hub 라벨에서 그 클래스의 annotation 하나 (각인 표시용)
for fn in sorted(aihub_kaggle):
    for ann in aihub_kaggle[fn]['annotations']:
        aihub_example.setdefault(ann['category_id'], ann)


def missing_text(fn):
    """그 이미지에서 빠진 알약의 K-코드와 각인을 적는다. 그림 제목이 옆 그림과 겹치지 않게 두 개씩 줄을 나눈다."""
    labels = [pill_label(aihub_example[m[1]]) if m[1] in aihub_example else f'K-{m[1]:06d}' for m in missing_labels if m[0] == fn]
    return '\n'.join(', '.join(labels[i:i + 2]) for i in range(0, len(labels), 2))


surplus_of = collections.defaultdict(dict)  # 이미지 → {box가 파일명보다 많은 K-코드: (box 수, 파일명에 적힌 수)}
for fn, code, extra, paths in surplus_labels:
    surplus_of[fn][code] = (len(paths), len(paths) - extra)


def json_part(ann):
    """box를 가져온 JSON 이름에서 각도와 copy 부분('75 copy')만 꺼낸다."""
    return Path(ann['json_path']).stem.split('_0_2_0_2_')[1].replace('_000_200', '')


SURPLUS_GROUP = '클래스 불일치: 한 이미지에 같은 K-코드의 box가 파일명보다 많음'
examples = []
for group in MISSING_GROUPS:
    files = sorted({m[0] for m in missing_labels if missing_group(m) == group})
    examples += [(group, fn) for fn in files[:2]]
surplus_files = sorted(surplus_of)
if surplus_files:  # 파일명 순서로 첫 이미지와, 남는 box가 가장 많은 이미지 중 첫 이미지를 그린다
    most = max(surplus_files, key=lambda f: sum(n - named for n, named in surplus_of[f].values()))
    examples += [(SURPLUS_GROUP, fn) for fn in dict.fromkeys([surplus_files[0], most])]
n_cols = 5
fig, axes = plt.subplots(int(np.ceil(len(examples) / n_cols)), n_cols, figsize=(30, 9.5 * np.ceil(len(examples) / n_cols)),
                         layout='constrained', squeeze=False)
for ax, (group, fn) in zip(axes.flat, examples):
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
    for ann in aihub_kaggle.get(fn, {'annotations': []})['annotations']:
        if not usable_bbox(ann['bbox']):
            continue  # 쓸 수 없는 bbox는 그릴 수 없다 (위 표의 라벨 누락)
        x, y, w, h = ann['bbox']
        surplus = ann['category_id'] in surplus_of.get(fn, {})  # 남는 box가 있는 K-코드는 주황으로 그리고 JSON 이름을 붙인다
        color = 'orange' if surplus else 'lime'
        text = f'{pill_label(ann)} [{json_part(ann)}]' if surplus else pill_label(ann)
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2.5))
        ax.text(x, y - 8, text, fontsize=8, va='bottom', bbox=dict(facecolor=color, alpha=0.85, pad=1.5, linewidth=0))
    if group == SURPLUS_GROUP:
        detail = 'box가 남는 알약: ' + ', '.join(f'K-{code:06d} box {n}개 (파일명 {named}개)' for code, (n, named) in surplus_of[fn].items())
    else:
        detail = f'bbox 없음: {missing_text(fn)}'
    ax.set_title(f'[{group}]\n{fn}\n{detail}', fontsize=9)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
    if not ax.images:
        ax.axis('off')
fig.suptitle('2-1 표의 행별 예시 (연두 box = 쓸 수 있는 bbox, 주황 box = 한 이미지에 파일명보다 많이 붙은 K-코드의 box와 그 JSON 이름, '
             '제목 셋째 줄부터 = 빠진 알약 또는 box가 남는 알약)', fontsize=14)
save_fig(fig, '2-1_missing_examples')
plt.show()

#### 2-1 결과

| 원인 | 객체 | 이미지 | 클래스 |
|---|---|---|---|
| 라벨 누락: bbox 빈 값 | 110 | 83 | 53 |
| 라벨 누락: 숫자가 붙은 bbox | 45 | 45 | 37 |
| 라벨 누락: JSON 없음 (촬영 PNG만 있는 이미지) | 42 | 12 | 10 |
| 라벨 누락: JSON 없음 (copy 파일이 있는 알약 폴더) | 23 | 22 | 15 |
| 라벨 누락: `-오류` 폴더 | 8 | 8 | 2 |
| 라벨 누락: JSON 없음 (그 밖) | 5 | 5 | 4 |
| 라벨 누락: 읽을 수 없는 JSON | 1 | 1 | 1 |
| **라벨 누락 합계** | **234** | **175** | **85** |
| 클래스 불일치: 파일명에 없는 K-코드 | 0 | 0 | 0 |
| 클래스 불일치: 같은 K-코드의 box가 파일명보다 많음 | 23 | 18 | 15 |

- 라벨 누락 이미지 175장은 촬영 PNG 12,009장의 1.5%다. 이미지당 빠진 알약은 1개가 147장으로 대부분이다.
- 파일명에 없는 K-코드는 0개다. JSON의 drug_N과 파일명의 K-코드는 서로 맞는다.
- box가 남는 23개와 "JSON 없음 (copy 파일이 있는 알약 폴더)" 23개는 같은 copy 폴더에서 생긴다.

그림에서 보이는 것

- JSON 없음, bbox 빈 값, 숫자가 붙은 bbox 행은 빠진 알약이 사진에 그대로 있다. box만 없다.
    - 예: 촬영 PNG만 있는 `K-001900-010224-016551-029345` 사진에는 알약 4개가 모두 있다.
- `-오류` 폴더와 읽을 수 없는 JSON 행은 파일명의 알약이 사진에 없고, 다른 알약이 놓여 있다.
    - `-오류` 폴더의 K-023319(마크/P) 대신 흰 D\|CT 알약이 있다.
    - 읽을 수 없는 JSON의 K-016551(DGTH/800) 대신 노란 GB 알약이 있다.
- copy 폴더 행은 한 알약에 주황 box 2~3개가 조금씩 어긋나 겹친다.

빠진 알약은 사진에서 찾아 box를 그려 넣고, `-오류` 폴더 알약은 사진 속 실제 클래스로 붙이고, copy 파일은 맞는 각도에 다시 배정한다. 이 처리는 모두 정제 목록에 들어 있다. (2-3)

#### 2-1 보충: 같은 K-코드의 알약 정보

- 같은 K-코드의 JSON끼리 알약 정보가 같은지 본다. 다르면 다른 알약의 JSON이 섞였을 수 있다.
    - 알약 정보 필드: dl_name(제품명), print_front·print_back(각인), drug_shape(모양), color_class1(색), leng_long·leng_short·thick(크기)
    - 1-3에서 바꾼 읽을 수 있는 JSON 전체로 본다. copy 파일, `-오류` 폴더, bbox 오류가 있는 JSON도 넣는다.
- 대회 train과 같은 클래스는 대회 JSON의 값과도 비교한다.

In [ ]:
PILL_FIELDS = ['dl_name', 'print_front', 'print_back', 'drug_shape', 'color_class1', 'leng_long', 'leng_short', 'thick']
aihub_field_values = collections.defaultdict(lambda: collections.defaultdict(collections.Counter))  # K-코드 → 필드 → 값 개수
for record in aihub_kaggle.values():
    for ann in record['annotations']:
        for field in PILL_FIELDS:
            aihub_field_values[ann['category_id']][field][str(ann['image'].get(field))] += 1
kaggle_field_values = {code: {field: str(ann['image'].get(field)) for field in PILL_FIELDS} for code, ann in class_example.items()}

rows = []
for code in sorted(aihub_field_values):
    for field in PILL_FIELDS:
        values = aihub_field_values[code][field]
        kaggle_value = kaggle_field_values.get(code, {}).get(field)
        if len(values) > 1 or (kaggle_value is not None and kaggle_value not in values):
            rows.append([f'K-{code:06d}', field, ', '.join(f'{v} ({n})' for v, n in values.most_common()), kaggle_value or '-'])
n_classes = len(aihub_field_values)
n_bad = len({r[0] for r in rows})
rows.append([f'**{n_classes}개 클래스 중 값이 하나가 아닌 클래스 {n_bad}개**', '', '', ''])
show_table(['K-코드', '필드', 'AI Hub 값 (annotation 수)', '대회 train 값'], rows)

#### 2-1 보충 결과

- 116개 클래스 모두 같은 K-코드의 JSON끼리 알약 정보가 같다. 대회 train에 있는 클래스는 대회 값과도 같다.

drug_N(K-코드)을 클래스로 그대로 쓴다. 새로 그려 넣는 box의 알약 정보는 같은 클래스의 다른 AI Hub annotation에서 가져온다.

### 2-2. 좌표 오류

| 문제 | 뜻 |
|---|---|
| 이미지 밖 | bbox가 이미지(976×1280) 밖으로 나간다 |
| area 불일치 | area가 bbox 너비×높이와 다르다. area 오타일 수도, bbox 오타일 수도 있다 |
| 다른 클래스와 box가 똑같음 | 같은 이미지에서 두 클래스의 box 좌표가 똑같다. 한 알약의 box를 다른 클래스에도 복사한 오류다 |

- 이 세 검사는 쓸 수 있는 bbox에만 한다. copy 파일이 있는 알약 폴더는 1-3이 넣은 대로(JSON 파일 이름의 각도대로) 본다.
- 2-1의 라벨 누락도 같은 표에 센다. 고치는 방법(사진에서 찾아 그리기)이 같기 때문이다.
- 아래 그림은 문제마다 예시 이미지다.

In [ ]:
def raw_annotation(fn, code):
    """1-3의 aihub_kaggle에서 그 알약의 원본 (bbox, area)를 꺼낸다. annotation이 없으면 (None, None)이다.

    aihub_kaggle의 bbox와 area는 원본 JSON 값 그대로이고, copy 파일을 다시 배정한 알약(2-1)은 다시 배정한 JSON의 값이다.
    파일명에 같은 K-코드가 두 번 나오는 이미지가 없어서 (이미지, K-코드)마다 annotation은 하나이다.
    """
    anns = [a for a in aihub_kaggle.get(fn, {'annotations': []})['annotations'] if a['category_id'] == code]
    return (anns[0]['bbox'], anns[0]['area']) if anns else (None, None)


problems = collections.defaultdict(list)  # 이미지 → [(K-코드, 문제, 원본 bbox, 원본 area)]
for fn, record in sorted(aihub_kaggle.items()):
    anns = [a for a in record['annotations'] if usable_bbox(a['bbox'])]  # 쓸 수 없는 bbox는 아래에서 2-1의 라벨 누락으로 센다
    same_box = collections.Counter(tuple(a['bbox']) for a in anns)
    for ann in anns:
        x, y, w, h = ann['bbox']
        kinds = []
        if same_box[tuple(ann['bbox'])] > 1:
            kinds.append('다른 클래스와 box가 똑같음')
        if x < 0 or y < 0 or x + w > IMG_W or y + h > IMG_H:
            kinds.append('이미지 밖')
        if ann['area'] != w * h:
            kinds.append('area 불일치')
        if kinds:
            problems[fn].append((ann['category_id'], ' + '.join(kinds), ann['bbox'], ann['area']))
for m in missing_labels:  # 2-1의 라벨 누락 (copy 파일을 다시 배정해 채운 알약은 2-1-3에서 빠졌다)
    fn, code = m[:2]
    problems[fn].append((code, f'라벨 누락: {missing_group(m)}', *raw_annotation(fn, code)))

kind_count = collections.Counter(kind for items in problems.values() for _, kind, _, _ in items)
rows = []
for kind, n in kind_count.most_common():
    files = sorted(fn for fn, items in problems.items() if any(k == kind for _, k, _, _ in items))
    example = next(item for item in problems[files[0]] if item[1] == kind)
    rows.append([kind, n, len(files), f'`{files[0]}` K-{example[0]:06d} bbox {example[2]}, area {example[3]}'])
rows.append(['**합계**', f'**{sum(kind_count.values())}**', f'**{len(problems)}장**', ''])
show_table(['문제', 'annotation 수', '이미지 수', '예시 (이미지, 원본 bbox, 원본 area)'], rows)

In [ ]:
problem_examples = []  # 위 표의 행마다 (문제, 예시 이미지, K-코드, 원본 bbox, 원본 area). 표와 같이 파일명 순서로 첫 이미지이다
for kind, _ in kind_count.most_common():
    files = sorted(fn for fn, items in problems.items() if any(k == kind for _, k, _, _ in items))
    problem_examples.append((kind, files[0], *next((code, bbox, area) for code, k, bbox, area in problems[files[0]] if k == kind)))
n_cols = 4
n_rows = int(np.ceil(len(problem_examples) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, 9.5 * n_rows), layout='constrained', squeeze=False)
for ax, (kind, fn, code, bbox, area) in zip(axes.flat, problem_examples):
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
    anns_of_box = collections.defaultdict(list)  # box → 그 box를 쓰는 annotation. box가 똑같은 클래스는 이름을 한 번에 적는다
    for ann in aihub_kaggle.get(fn, {'annotations': []})['annotations']:
        if usable_bbox(ann['bbox']):  # 쓸 수 없는 bbox(라벨 누락)는 그릴 수 없다
            anns_of_box[tuple(ann['bbox'])].append(ann)
    for (x, y, w, h), anns in anns_of_box.items():
        target = any(a['category_id'] == code for a in anns)  # 예시 알약의 box는 빨간 점선으로 그린다 (이미지 밖으로 나간 부분은 잘려 보인다)
        color = 'red' if target else 'lime'
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2.5, linestyle='--' if target else '-'))
        ax.text(x, y - 8, '\n'.join(map(pill_label, anns)), fontsize=8, va='bottom', color='white' if target else 'black',
                bbox=dict(facecolor=color, alpha=0.85, pad=1.5, linewidth=0))
    pill_text = pill_label(aihub_example[code]) if code in aihub_example else f'K-{code:06d}'
    detail = f'{pill_text}: 원본 bbox {bbox}, area {area}'
    if usable_bbox(bbox):  # 좌표 문제는 area, 이미지 크기와 견줄 값을 함께 적는다
        x, y, w, h = bbox
        detail += f'\n너비×높이 {w * h}, 오른쪽 끝(x+너비) {x + w}, 아래 끝(y+높이) {y + h}'
    ax.set_title(f'[{kind}]\n{fn}\n{detail}', fontsize=9)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
    if not ax.images:
        ax.axis('off')
fig.suptitle(f'2-2 표의 문제별 예시 (표의 예시 이미지, 빨간 점선 = 예시 알약의 원본 bbox, 연두 = 같은 이미지의 다른 쓸 수 있는 bbox, '
             f'라벨 누락은 그릴 bbox가 없다, 이미지 {IMG_W}×{IMG_H})', fontsize=14)
save_fig(fig, '2-2_problem_examples')
plt.show()

#### 2-2 결과

| 문제 | annotation | 이미지 |
|---|---|---|
| area 불일치 | 427 | 414 |
| 다른 클래스와 box가 똑같음 | 134 | 70 |
| 이미지 밖 | 33 | 33 |
| 이미지 밖 + area 불일치 | 27 | 27 |
| 다른 클래스와 box가 똑같음 + area 불일치 | 6 | 6 |
| 라벨 누락 (2-1) | 234 | 175 |
| **합계** | **861** | **713** |

그림에서 보이는 것

- area 불일치 예시의 K-002483은 bbox 너비가 210인데 area 10248은 21×488이다. box는 알약을 잘 감싸므로 area만 틀렸다.
- box가 똑같은 예시에서는 K-005002와 K-012778이 ELC 캡슐의 box 하나를 같이 쓰고, K-012778(빨간 YH 알약)에는 box가 없다.
- 이미지 밖 box는 숫자 오타로 보인다.
    - `[77, 158, 3131, 498]`은 너비 313에 1이 더 붙었다.
    - `[644, 851, 190, 498]`은 y 값이 틀려 box가 아래 알약 쪽에 있다.
- 숫자가 붙은 bbox `[121125, 333, 475]`는 `[121, 125, 333, 475]`에서 쉼표가 빠진 값이다.

좌표 오류는 대부분 숫자 하나가 빠지거나 더해지거나, 쉼표가 빠진 오타다. 정제 목록은 오타를 고친 후보 중 사진의 알약과 맞는 값으로 고치고, 고칠 수 없으면 사진에서 알약을 찾아 다시 그린 값이다. (2-3)

### 2-3. 정제 목록 적용

- 레포의 정제 목록 `data/label_fix/label_fixes.csv`에서 aihub 줄을 읽어 1-3의 라벨에 적용한다. 적용 코드는 학습 데이터를 만드는 `pill_detection.data`의 함수를 그대로 쓴다.
    - add: 라벨이 없던 알약의 box를 새로 넣는다. 알약 정보는 같은 클래스의 다른 JSON에서 가져온다.
    - modify: box를 고친다. 클래스를 바꾸는 줄(`category_id_after`)과 이미지 필드를 고치는 줄(`image_fix`)도 있다.
    - exclude: 그 이미지를 뺀다.
- 정제 목록은 정제 노트북에서 아래 방법으로 만들었다. 근거 그림과 검증은 그 노트북에 있다.
    - 좌표 오타는 숫자 하나를 빼거나 넣거나 바꾼 값 중 사진의 알약과 맞는 값으로 고친다. (정제 노트북 2-2-4)
    - 맞는 값이 없거나 box가 없으면 사진에서 알약을 OpenCV로 찾아 다시 그리고, 문제 이미지를 모두 갤러리로 눈으로 확인한다. (2-2-2, 2-2-5)
    - 좌표는 멀쩡해도 알약을 감싸지 않는 box 313개도 같은 규칙으로 고친다. (2-2-3)
    - copy 파일이 있는 알약 폴더 21개는 세 각도 사진에 box를 모두 그려 보고 맞는 각도에 다시 배정한다. (2-1-3)
    - `-오류` 폴더 9장은 파일명의 알약 대신 놓인 실제 알약의 클래스로 붙인다. (2-2-6)
    - crop 갤러리와 크기·색 대조로 찾은 라벨 바뀜은 클래스를 맞바꾼다. (2-3)
    - 사진에서 알약을 찾지 못한 이미지 20장, 같은 장면 6장(1-6), 같은 알약 구성 9장은 뺀다. (2-2-9, 2-6)
- 아래 첫 표는 정제 목록의 줄 수, 둘째 표는 적용 전후의 이미지·객체·클래스 수와 남은 라벨 문제다.

In [ ]:
from pill_detection.data import build_aihub_export, check_label_fixes, load_annotations

# 정제 목록(aihub 줄)을 읽어, 학습 데이터를 만드는 함수로 정제 라벨을 메모리에 만든다. 파일은 읽기만 한다
aihub_fixes = read_label_fixes(LABEL_FIXES_CSV, source='aihub')
aihub_raw_records = load_annotations(AIHUB_RAW_ANN_DIR, aihub=True)
aihub_plans = check_label_fixes(aihub_fixes, aihub_raw_records, AIHUB_IMG_DIR, aihub=True)
aihub_export = build_aihub_export(aihub_plans, AIHUB_RAW_ANN_DIR, AIHUB_IMG_DIR)  # {'names': 원래 이름 → 새 이름, 'images': 새 이름 → JSON 목록}

# 정제 라벨: 원래 이미지 파일명 → {'image', 'annotations'}. 3절부터 이 라벨로 본다 (원본 PNG를 열어야 하므로 원래 이름을 키로 쓴다)
aihub_clean_records = {}
for fn, new_name in sorted(aihub_export['names'].items()):
    jsons = aihub_export['images'][new_name]
    anns = [{**d['annotations'][0], 'image': d['images'][0], 'json_path': None} for d in jsons]
    aihub_clean_records[fn] = {'image': jsons[0]['images'][0], 'annotations': anns}
excluded_files = {fix['file_name'] for fix in aihub_fixes if fix['action'] == 'exclude'}
renamed = {fn: new for fn, new in aihub_export['names'].items() if fn != new}
assert set(aihub_clean_records) == set(aihub_files) - excluded_files, '정제 라벨의 이미지가 촬영 PNG에서 exclude 줄의 이미지를 뺀 것과 다르다'

action_count = collections.Counter(fix['action'] for fix in aihub_fixes)
show_table(['항목', '값'], [
    ['정제 목록 aihub 줄', f"{len(aihub_fixes):,}줄 (add {action_count['add']}, modify {action_count['modify']}, exclude {action_count['exclude']})"],
    ['고치거나 빼는 이미지', f"{len({f['file_name'] for f in aihub_fixes}):,}장 (촬영 PNG {len(aihub_files):,}장의 {len({f['file_name'] for f in aihub_fixes}) / len(aihub_files):.1%})"],
    ['클래스를 바꾸는 줄 / 이미지 필드를 고치는 줄', f"{sum(bool(f['category_id_after']) for f in aihub_fixes)}줄 / {sum(bool(f['image_fix']) for f in aihub_fixes)}줄"],
    ['뺀 이미지 (exclude)', f'{len(excluded_files)}장'],
    ['정제 라벨로 다시 만든 이름이 원래와 다른 이미지', f'{len(renamed)}장'],
])


- 적용 전후를 비교한다. `쓸 수 없는 bbox`는 값이 4개가 아니거나 너비·높이가 0 이하인 bbox, `box 수가 파일명과 다른 이미지`는 쓸 수 있는 box 수가 파일명의 알약 수와 다른 이미지다.

In [ ]:
def label_summary(labels):
    """이미지 → annotation 목록에서 이미지, 객체, 클래스 수와 남은 라벨 문제를 센다."""
    objects = [a for anns in labels.values() for a in anns]
    usable = {fn: [a for a in anns if usable_bbox(a['bbox'])] for fn, anns in labels.items()}
    return [f'{len(labels):,}장', f'{len(objects):,}개', f"{len({a['category_id'] for a in objects})}개",
            f"{sum(not usable_bbox(a['bbox']) for a in objects):,}개",
            f"{sum(len(anns) != len(pill_codes(fn)) for fn, anns in usable.items()):,}장"]


show_table(['라벨', '이미지', '객체', '클래스', '쓸 수 없는 bbox', '쓸 수 있는 box 수가 파일명의 알약 수와 다른 이미지'], [
    ['바꾼 AI Hub 라벨 (1-3)', *label_summary({fn: r['annotations'] for fn, r in aihub_kaggle.items()})],
    ['**정제 라벨 (2-3)**', *[f'**{v}**' for v in label_summary({fn: r['annotations'] for fn, r in aihub_clean_records.items()})]],
])


#### 2-3 결과

| 라벨 | 이미지 | 객체 | 클래스 | 쓸 수 없는 bbox | box 수가 파일명과 다른 이미지 |
|---|---|---|---|---|---|
| 바꾼 AI Hub 라벨 (1-3) | 11,997장 | 45,909개 | 116개 | 163개 | 179장 |
| **정제 라벨 (2-3)** | **11,974장** | **45,819개** | **116개** | **0개** | **0장** |

- 정제 목록 aihub 줄은 1,093줄(add 70, modify 987, exclude 36)이고, 이미지 961장(촬영 PNG의 8.0%)을 고치거나 뺀다.
- 촬영 PNG 12,009장에서 35장을 뺐다. 알약을 찾지 못한 20장, 같은 알약 구성 9장, 같은 장면 6장이다. (2-4)
- 촬영 PNG만 있던 12장은 box를 그려 넣어 들어갔다. add 70줄이 원본 JSON 없이 새로 만든 annotation이다.
- 클래스를 바꾼 이미지 가운데 파일명의 K-코드와 정제 라벨의 클래스가 다른 이미지가 생긴다. 학습 데이터를 만들 때는 정제 라벨대로 이름을 다시 만들어 13장의 이름이 바뀐다. 이 노트북은 원본 PNG를 열어야 하므로 원래 이름으로 부른다.

클래스는 파일명이 아니라 정제 라벨의 category_id로 읽는다. 학습 데이터는 같은 함수로 `scripts/prepare_train_data.py`가 만든다.

### 2-4. 뺀 이미지

- 정제 목록의 exclude 줄이 있는 이미지 35장 중 8장을 1-3 라벨 그대로 그린다. box가 없는 알약이 많은 이미지부터 고른다.
    - 연두 box는 쓸 수 있는 bbox이고, 제목에 box가 없는 알약의 K-코드와 각인을 적는다.
- 뺀 이유는 세 가지다.

| 이유 | 이미지 | 내용 |
|---|---|---|
| 사진에서 알약을 찾지 못함 | 20장 | box가 없거나 틀린 알약 21개를 사진에서 찾지 못했다. 대부분 바닥과 색이 비슷한 파란 알약(K-010487, K-031885, K-041149 등)이다. box 없는 알약은 배경으로 학습되므로 이미지를 뺀다 (정제 노트북 2-2-9) |
| 같은 알약 구성 | 9장 | 클래스를 고치니 이미 있는 다른 조합과 알약 구성이 같아진 조합 3개. 이름이 틀린 쪽을 뺀다 (정제 노트북 2-6) |
| 같은 장면 | 6장 | 같은 장면을 두 번 찍어 다른 조합 이름으로 들어간 쌍 6쌍 중 이름이 틀린 쪽 (1-6) |

In [ ]:
missing_of = collections.defaultdict(list)  # 이미지 → 2-1에서 box가 없던 알약의 K-코드
for m in missing_labels:
    missing_of[m[0]].append(m[1])
excluded_sample = sorted(excluded_files, key=lambda fn: (-len(missing_of[fn]), fn))[:8]

fig, axes = plt.subplots(2, 4, figsize=(24, 18), layout='constrained')
for ax, fn in zip(axes.flat, excluded_sample):
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
    for ann in aihub_kaggle.get(fn, {'annotations': []})['annotations']:
        if not usable_bbox(ann['bbox']):
            continue
        x, y, w, h = ann['bbox']
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2.5))
        ax.text(x, y - 8, pill_label(ann), fontsize=8, va='bottom', bbox=dict(facecolor='lime', alpha=0.85, pad=1.5, linewidth=0))
    missing = ', '.join(pill_label(aihub_example[c]) if c in aihub_example else f'K-{c:06d}' for c in missing_of[fn])
    ax.set_title(f"{fn}\nbox 없음: {missing or '(없음) 같은 장면 또는 같은 알약 구성으로 뺀 이미지'}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f'정제 목록에서 뺀 이미지 {len(excluded_files)}장 중 8장 (연두 box = 1-3 라벨의 쓸 수 있는 bbox, 제목 = 2-1에서 box가 없던 알약)', fontsize=14)
save_fig(fig, '2-4_excluded_sample')
plt.show()


#### 2-4 결과

그림에서 보이는 것

- 8장 모두 box가 없는 알약이 사진에 실제로 놓여 있다. 알약이 없는데 라벨만 있는 경우는 아니다.
- 1, 2번째 사진은 바닥과 색이 비슷한 파란 알약(K-010487 D-W 캡슐, K-041149 Hanwha LCR5, K-005391 MSD 72)을 사진에서 찾지 못해 뺀 경우다.
    - 1번째는 촬영 PNG만 있던 이미지라 네 알약 모두 box가 없고, 그중 파란 알약 2개를 찾지 못했다.
- 3번째 사진은 K-005886(TYLENOL ER)의 box가 빨간 V100 알약 위에 있고, 정작 TYLENOL ER에는 box가 없다. 문제가 없어 보이던 라벨의 클래스까지 틀려서 뺐다.
- 4~8번째 사진은 `-오류` 폴더 조합 2개다. 파일명의 K-023319 대신 흰 D\|CT 알약(K-029711)이 놓여 있다. 실제 클래스로 고치면 이미 있는 다른 조합과 알약 구성이 같아져 뺀 9장에 든다.

35장은 학습 데이터에서 뺀다. 뺀 사진은 파란 클래스마다 그 클래스 촬영 PNG의 1.3% 이하라(K-031885 459장 중 6장, K-010487 540장 중 6장) 클래스가 비는 일은 없다.

### 2-5. 정리

- 받은 그대로의 AI Hub 데이터, 1차 정제(1-1-4) 뒤, 2차 정제(정제 라벨, 2-3) 뒤의 이미지·객체·클래스 수를 한 표로 모은다.
    - 원본과 1차 정제의 객체 수는 JSON 수다. 원본 JSON 하나가 알약 하나이고, 읽지 못한 JSON 1개도 센다. 클래스는 JSON의 drug_N(K-코드) 기준이다.


In [ ]:
def stage_counts(images, jsons):
    """이미지 파일명 목록과 JSON 목록(aihub_jsons 구조)에서 이미지, 객체, 클래스 수를 센다. JSON 하나가 알약 하나다."""
    classes = {j['data']['images'][0]['drug_N'] for j in jsons if j['data'] is not None}
    return [f'{len(images):,}장', f'{len(jsons):,}개', f'{len(classes)}개']


clean_objects = [a for r in aihub_clean_records.values() for a in r['annotations']]
show_table(['단계', '이미지 수', '객체 수', '클래스 수'], [
    ['AI Hub 원본 (받은 그대로)', *stage_counts(aihub_png, aihub_jsons)],
    ['1차 정제 후 (1-1-4)', *stage_counts(aihub_photos_1st, aihub_jsons_1st)],
    ['**2차 정제 후 = 정제 라벨 (2-3)**', f'**{len(aihub_clean_records):,}장**', f'**{len(clean_objects):,}개**',
     f"**{len({a['category_id'] for a in clean_objects})}개**"],
])


#### 2-5 총정리

| 단계 | 이미지 수 | 객체 수 | 클래스 수 |
|---|---|---|---|
| AI Hub 원본 (받은 그대로) | 16,012장 | 45,934개 | 116개 |
| 1차 정제 후 (1-1-4) | 12,009장 | 45,910개 | 116개 |
| **2차 정제 후 = 정제 라벨 (2-3)** | **11,974장** | **45,819개** | **116개** |

- 1차 정제는 `_index.png` 4,003장과 촬영 PNG가 없는 JSON 24개를 뺀다. 사진은 보지 않고 파일의 짝만으로 정했다.
- 2차 정제는 정제 목록(aihub 줄 1,093줄)을 적용한 것이다. 뺀 이미지 35장의 JSON 138개가 빠지고, box 70개가 새로 들어가고, copy 파일로 한 알약에 겹쳐 있던 box 23개가 빠져 객체가 91개 준다.
- 클래스 116개는 세 단계 내내 같다. 뺀 이미지는 어느 클래스에서도 그 클래스 촬영 PNG의 1.3% 이하다. (2-4)

#### 확인한 것

- AI Hub 원본 라벨은 대회 데이터보다 오류의 종류가 많다. 라벨 누락 234개(175장), 좌표 오류 627개(area 불일치, 이미지 밖, 다른 클래스와 box가 똑같음), 알약을 감싸지 않는 box 313개, 클래스 오류가 있었다.
- 좌표 오류는 대부분 숫자 하나가 빠지거나 더해진 오타라 원본 값을 살려 고칠 수 있다.
- 정제 목록을 적용한 정제 라벨에는 쓸 수 없는 bbox와 box 수가 파일명과 다른 이미지가 없다. (2-3)
- 35장은 뺐다. 알약을 찾지 못한 20장, 같은 알약 구성 9장, 같은 장면 6장이다.

#### 정한 것

- 레포의 정제 목록을 1-3 변환 뒤에 적용한 정제 라벨을 쓴다. 클래스는 파일명이 아니라 정제 라벨의 category_id로 읽는다. (2-3)
- 알약을 찾지 못한 20장, 같은 장면 6장, 같은 알약 구성 9장은 뺀다. box 없는 알약은 배경으로 학습되기 때문이다. (2-4)
- 학습 데이터는 `scripts/prepare_train_data.py`가 같은 함수로 같은 정제를 적용해 만든다. (2-3)

#### 뒤 섹션에서 볼 것

- 3절부터는 정제 라벨로 bbox와 클래스 분포를 본다.

## 3. bbox 분포

정제 라벨(2-3)의 box를 정제한 대회 train과 비교한다. 입력 크기(640, 1280), 크기 조절·모자이크 증강, NMS IoU 기본값(0.7), 최대 검출 수(300)가 AI Hub를 섞은 뒤에도 맞는지 본다.

| 단계 | 하는 일 | 이유 |
|---|---|---|
| 3-1 | 객체 수와 크기 | 이미지당 객체 수, box 크기, 같은 알약의 여백을 대회와 비교한다 |
| 3-2 | 위치와 겹침 | box 위치와 box끼리 겹치는 정도를 본다 |
| 3-3 | 정리 | 확인한 것과 정한 것 |

### 3-1. 객체 수와 크기

이미지당 객체 수와 box 짧은 변(3-1-1), 같은 클래스·각도의 box 크기로 본 라벨 여백(3-1-2)을 대회 train과 비교한다.

#### 3-1-1. 객체 수와 box 크기

- 왼쪽: 이미지당 객체 수의 비율
- 가운데: box 짧은 변(원본 픽셀). 긴 알약은 놓인 방향에 따라 너비와 높이가 바뀌므로 짧은 변으로 본다.
- 오른쪽: 입력 640(긴 변 1280을 640으로 줄임, 0.5배)에서의 짧은 변
- 두 묶음의 크기가 달라서 세 그림 모두 비율(밀도)로 그린다.

In [ ]:
def box_sides(records):
    """box마다 (짧은 변, 긴 변) 픽셀을 모은다."""
    return np.array([sorted(a['bbox'][2:]) for r in records.values() for a in r['annotations']], dtype=float)


compare_sets = {'대회 train (정제)': clean_records, 'AI Hub (정제)': aihub_clean_records}
fig, axes = plt.subplots(1, 3, figsize=(21, 5.5), layout='constrained')
for (name, records), color in zip(compare_sets.items(), ['tab:blue', 'tab:green']):
    counts = collections.Counter(len(r['annotations']) for r in records.values())
    ks = np.arange(1, 5)
    share = np.array([counts[k] for k in ks]) / len(records)
    offset = -0.2 if color == 'tab:blue' else 0.2
    axes[0].bar(ks + offset, share, width=0.4, color=color, label=f'{name} {len(records):,}장')
    for k, s in zip(ks, share):
        if s > 0:
            axes[0].text(k + offset, s + 0.01, f'{s:.1%}', ha='center', fontsize=9)
    sides = box_sides(records)
    axes[1].hist(sides[:, 0], bins=np.arange(0, 560, 10), density=True, histtype='step', linewidth=2, color=color,
                 label=f'{name} 중앙값 {np.median(sides[:, 0]):.0f}, 최소 {sides[:, 0].min():.0f}')
    axes[2].hist(sides[:, 0] * 0.5, bins=np.arange(0, 280, 5), density=True, histtype='step', linewidth=2, color=color,
                 label=f'{name} 중앙값 {np.median(sides[:, 0]) * 0.5:.0f}, 최소 {sides[:, 0].min() * 0.5:.0f}')
axes[0].set_xticks(ks)
axes[0].set_xlabel('이미지당 객체 수 (개)')
axes[0].set_ylabel('이미지 비율')
axes[0].set_title('이미지당 객체 수')
axes[1].set_xlabel('box 짧은 변 (원본 픽셀)')
axes[1].set_title('box 짧은 변 길이 (원본 976×1280)')
axes[2].set_xlabel('box 짧은 변 (입력 640 기준 픽셀)')
axes[2].set_title('입력 640에서의 box 짧은 변 길이')
for ax in axes[1:]:
    ax.set_ylabel('밀도 (넓이 합 = 1)')
for ax in axes:
    ax.legend(fontsize=9)
fig.suptitle('대회 train과 AI Hub의 객체 수와 box 크기 (묶음마다 비율로 그림)', fontsize=14)
save_fig(fig, '3-1_objects_and_sizes')
plt.show()

#### 3-1-1 결과

| 항목 | AI Hub | 대회 train |
|---|---|---|
| 가장 흔한 이미지당 객체 수 | 4개 (82.7%) | 3개 (67.7%) |
| box 짧은 변 중앙값 / 최솟값 (원본) | 236 / 120픽셀 | 214 / 125픽셀 |
| box 짧은 변 중앙값 / 최솟값 (입력 640) | 118 / 60픽셀 | 107 / 62픽셀 |

- 두 데이터의 box 크기 분포가 비슷하다. AI Hub에도 작은 객체가 없다.
- 이미지당 객체가 최대 4개라 최대 검출 수 기본값(300)으로 충분하다.

입력 640에서도 가장 작은 box가 60픽셀이라, AI Hub를 섞어도 입력 크기를 바꿀 이유가 없다.

#### 3-1-2. 라벨 여백

- 같은 알약을 같은 각도에서 찍으면 알약이 사진에 같은 크기로 찍힌다. 그래서 box 크기 차이는 라벨을 그린 습관(알약 둘레 여백)의 차이다.
- 대회 train과 AI Hub에 모두 있는 클래스에서, (클래스, 각도)마다 box의 긴 변과 짧은 변 중앙값을 비교한다.
    - 점이 대각선 위에 있으면 두 데이터의 box 크기가 같다. 대각선에서 벗어난 거리가 여백 차이다.
- 여백 습관이 몇 픽셀만 달라도 IoU 0.95 같은 높은 기준에서 점수가 깎인다.

In [ ]:
def side_medians(records):
    """(K-코드, 촬영 각도)마다 box의 긴 변, 짧은 변 중앙값과 box 수를 구한다."""
    sides = collections.defaultdict(list)
    for record in records.values():
        for ann in record['annotations']:
            sides[(ann['category_id'], ann['image']['camera_la'])].append(sorted(ann['bbox'][2:], reverse=True))
    return {key: (np.median(v, axis=0), len(v)) for key, v in sides.items()}


kaggle_sides, aihub_sides = side_medians(clean_records), side_medians(aihub_clean_records)
common_keys = sorted(set(kaggle_sides) & set(aihub_sides))
diffs = np.array([aihub_sides[k][0] - kaggle_sides[k][0] for k in common_keys])  # (긴 변 차이, 짧은 변 차이)
fig, axes = plt.subplots(1, 2, figsize=(16, 7), layout='constrained')
for i, (ax, side) in enumerate(zip(axes, ['긴 변', '짧은 변'])):
    k_vals = np.array([kaggle_sides[k][0][i] for k in common_keys])
    a_vals = np.array([aihub_sides[k][0][i] for k in common_keys])
    ax.scatter(k_vals, a_vals, s=18, alpha=0.7)
    lo, hi = min(k_vals.min(), a_vals.min()) - 10, max(k_vals.max(), a_vals.max()) + 10
    ax.plot([lo, hi], [lo, hi], color='gray', linestyle='--', linewidth=1)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel(f'대회 train box {side} 중앙값 (픽셀)')
    ax.set_ylabel(f'AI Hub box {side} 중앙값 (픽셀)')
    d = diffs[:, i]
    ax.set_title(f'{side}: AI Hub - 대회 차이 중앙값 {np.median(d):+.1f}픽셀, 절댓값 4픽셀 넘는 묶음 {(np.abs(d) > 4).sum()}/{len(d)}')
fig.suptitle(f'같은 클래스·같은 각도의 box 크기 비교 ({len(common_keys)}개 묶음, 점 = (클래스, 각도) 하나, 회색 점선 = 크기가 같음)', fontsize=14)
save_fig(fig, '3-1b_margin_compare')
plt.show()

#### 3-1-2 결과

| 항목 | 긴 변 | 짧은 변 |
|---|---|---|
| AI Hub − 대회 차이 중앙값 | +0.0픽셀 | −1.0픽셀 |
| 차이가 4픽셀을 넘는 묶음 | 77/162 | 90/162 |

그림에서 보이는 것

- (클래스, 각도) 162개 묶음의 점이 대각선을 따라 놓여 있다. 두 데이터의 box 크기가 같다.
- 차이가 4픽셀을 넘는 묶음도 대각선 위아래로 고르게 퍼져 있다. 한쪽으로 치우친 여백 습관은 아니다.
    - 대회 train은 묶음마다 box가 평균 3.5개(569개 ÷ 162묶음)뿐이라 중앙값이 쉽게 흔들린다.
    - 짧은 변이 더 퍼진 것은 긴 알약의 짧은 변이 놓인 방향에 따라 크게 바뀌기 때문으로 보인다.

AI Hub box는 대회 box와 같은 여백으로 그려졌다. box 크기를 고치지 않고 그대로 섞는다.

### 3-2. 위치와 겹침

- 첫째 그림 왼쪽: AI Hub box 중심 위치의 분포 (색이 진할수록 많음)
- 첫째 그림 오른쪽: 이미지마다 box 쌍의 IoU 최댓값 분포를 대회 train과 비교
- 둘째 그림: IoU 최댓값이 가장 큰 AI Hub 이미지 10장. box가 겹친 까닭이 알약 배치인지 라벨 오류인지 본다.

In [ ]:
centers = np.array([[a['bbox'][0] + a['bbox'][2] / 2, a['bbox'][1] + a['bbox'][3] / 2]
                    for r in aihub_clean_records.values() for a in r['annotations']])


def max_pair_iou(records):
    """이미지마다 box 쌍의 IoU 최댓값을 구한다. box가 하나면 0이다."""
    values = []
    for record in records.values():
        boxes = [a['bbox'] for a in record['annotations']]
        values.append(max([box_iou(a, b) for i, a in enumerate(boxes) for b in boxes[i + 1:]], default=0))
    return np.array(values)


fig, axes = plt.subplots(1, 2, figsize=(16, 7), layout='constrained', gridspec_kw={'width_ratios': [0.8, 1.2]})
h = axes[0].hist2d(centers[:, 0], centers[:, 1], bins=[np.linspace(0, IMG_W, 25), np.linspace(0, IMG_H, 33)], cmap='Blues')
fig.colorbar(h[3], ax=axes[0], label='box 중심 수 (개)')
axes[0].set_xlim(0, IMG_W)
axes[0].set_ylim(IMG_H, 0)
axes[0].set_aspect('equal')
axes[0].set_xlabel('x (픽셀)')
axes[0].set_ylabel('y (픽셀)')
axes[0].set_title(f'AI Hub box 중심 위치 ({len(centers):,}개)')
bins = np.linspace(0, 1, 51)
pair_iou = {name: max_pair_iou(records) for name, records in compare_sets.items()}
for (name, values), color in zip(pair_iou.items(), ['tab:blue', 'tab:green']):
    axes[1].hist(values, bins=bins, histtype='step', linewidth=2, color=color,
                 label=f'{name} {len(values):,}장: 최댓값 {values.max():.3f}, 0.05 넘는 이미지 {(values > 0.05).sum()}장')
axes[1].set_yscale('log')
axes[1].set_xlabel('이미지 안 box 쌍의 IoU 최댓값')
axes[1].set_ylabel('이미지 수 (장, 로그 눈금)')
axes[1].set_title('이미지마다 가장 많이 겹친 box 쌍의 IoU')
axes[1].legend()
fig.suptitle('box 위치와 겹침', fontsize=14)
save_fig(fig, '3-2_position_overlap')
plt.show()

In [ ]:
aihub_pair_iou = pair_iou['AI Hub (정제)']
top_overlap = sorted(zip(aihub_pair_iou, aihub_clean_records), key=lambda t: (-t[0], t[1]))[:10]
fig, axes = plt.subplots(2, 5, figsize=(25, 14), layout='constrained')
for rank, (ax, (iou, fn)) in enumerate(zip(axes.flat, top_overlap), start=1):
    anns = aihub_clean_records[fn]['annotations']
    pair = max(((a, b) for i, a in enumerate(anns) for b in anns[i + 1:]), key=lambda p: box_iou(p[0]['bbox'], p[1]['bbox']))
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
    for ann in anns:
        color = 'red' if any(ann is p for p in pair) else 'lime'
        x, y, w, h = ann['bbox']
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2))
        # 겹친 쌍의 라벨 글자가 같은 자리에 겹치지 않게, 두 번째 box는 box 아래에 적는다
        ty, va = (y + h + 8, 'top') if ann is pair[1] else (y - 8, 'bottom')
        ax.text(x, ty, pill_label(ann), fontsize=7, va=va, bbox=dict(facecolor=color, alpha=0.85, pad=1.5, linewidth=0))
    ax.set_title(f"{rank}. IoU {iou:.3f} | {aihub_clean_records[fn]['image']['camera_la']}°\n{fn}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f'box 쌍의 IoU가 가장 큰 AI Hub 이미지 10장 (정제 라벨 {len(aihub_pair_iou):,}장 중 box가 겹친 이미지 {(aihub_pair_iou > 0).sum()}장, '
             f'IoU 0.05 초과 {(aihub_pair_iou > 0.05).sum()}장 | 빨강 = 가장 많이 겹친 box 쌍, 연두 = 나머지 box)', fontsize=14)
save_fig(fig, '3-2b_top_overlap')
plt.show()

#### 3-2 결과

| 항목 | AI Hub | 대회 train |
|---|---|---|
| 이미지 안 box 쌍의 IoU 최댓값 | 0.151 | 0.021 |
| IoU 0.05를 넘는 이미지 | 42장 | 0장 |

그림에서 보이는 것

- box 중심은 사진의 네 자리(2×2 격자)에 몰려 있고, 가운데 위·아래에도 옅은 자리가 있다. 중심은 대부분 x 150~850, y 150~1,050 안에 있다.
- IoU가 가장 큰 10장은 모두 70° 사진이고, 긴 캡슐 K-041149(Hanwha LCR5)가 대각선으로 놓여 있다.
- box는 알약을 맞게 감싼다. 대각선으로 놓인 긴 캡슐의 box가 옆 알약 box의 모서리와 겹친 것이고, 라벨 오류로 겹친 box는 없다.

실제 box끼리 겹침이 최대 0.151이라, NMS IoU를 기본값 0.7로 두어도 0.5로 낮춰도 실제 box가 지워지지 않는다.

### 3-3. 정리

#### 확인한 것

- 이미지당 객체는 AI Hub가 주로 4개, 대회 train이 주로 3개다.
- box 크기 분포가 비슷하고, 입력 640에서도 가장 작은 box 짧은 변이 60픽셀이다.
- 같은 클래스·각도의 box 크기 차이 중앙값은 0~1픽셀이다. 라벨 여백 습관이 같다.
- box 쌍의 IoU 최댓값은 0.151로, 대각선으로 놓인 긴 캡슐 때문이다.

#### 정한 것

- box 크기는 고치지 않고 대회 box와 그대로 섞는다. (3-1-2)
- 입력 크기와 크기 조절·모자이크 증강은 AI Hub 때문에 바꿀 것이 없다. (3-1-1)
- NMS IoU와 최대 검출 수는 기본값(0.7, 300)으로 충분하다. (3-1-1, 3-2)

#### 뒤 섹션에서 볼 것

- 학습 뒤 val에서 AI Hub 이미지와 대회 이미지의 IoU 0.95 AP를 따로 보면, 여백 차이가 점수에 영향을 주는지 알 수 있다.
- 라벨 box를 알약에 딱 맞게 고치는 전처리는 데이터 전처리 실험 후보로 둔다.

## 4. 클래스 분포

정제 라벨(2-3)의 클래스 분포를 대회 train(정제 라벨)과 비교하고, 합쳤을 때의 불균형을 본다.

| 단계 | 하는 일 | 이유 |
|---|---|---|
| 4-1 | 클래스별 객체 수와 등장 조합 수 | 대회 train 클래스와 새 클래스가 얼마나 있는지 본다 |
| 4-2 | 조합당 이미지 수 | 조합마다 세 각도가 다 있는지 본다 |
| 4-3 | 같은 조합 사진 비교 | 같은 조합의 각도별 사진이 어떤 관계인지 본다 |
| 4-4 | 대회 train과 합친 분포 | 합친 뒤의 불균형을 잰다 |
| 4-5 | 가장 균일하게 고르기 | 목표 객체 수로 고르면 균형이 나아지는지 본다 |
| 4-6 | 정리 | 확인한 것과 정한 것 |

### 4-1. 클래스별 객체 수와 등장 조합 수

- 위: 대회 train 클래스는 대회 train과 AI Hub의 객체 수를 나란히 놓고(로그 눈금), train에 없는 클래스는 AI Hub 객체 수만 본다.
- 아래: 그 클래스가 나오는 조합 수. 조합 하나는 같은 배치를 각도만 바꿔 찍은 사진 3장이다.
    - 객체 수가 많아도 조합 수가 적으면 같은 배치의 사진만 반복해서 보는 셈이다.

In [ ]:
def class_stats(records):
    """클래스별 객체 수와 등장 조합 수를 센다."""
    objects, combos = collections.Counter(), collections.defaultdict(set)
    for fn, record in records.items():
        for ann in record['annotations']:
            objects[ann['category_id']] += 1
            combos[ann['category_id']].add(combo_name(fn))
    return objects, {c: len(s) for c, s in combos.items()}


aihub_object_count, aihub_combo_count = class_stats(aihub_clean_records)
_, train_combo_count = class_stats(clean_records)
aihub_classes = set(aihub_object_count)
new_classes = sorted(aihub_classes - set(train_classes), key=lambda c: (-aihub_object_count[c], c))
absent_train = sorted(set(train_classes) - aihub_classes)

fig, axes = plt.subplots(2, 2, figsize=(22, 10), layout='constrained', gridspec_kw={'width_ratios': [1, 1.1]})
order = sorted(train_classes, key=lambda c: (-clean_object_count[c], c))
x = np.arange(len(order))
for row, (train_values, aihub_values, unit) in enumerate([(clean_object_count, aihub_object_count, '객체 수'),
                                                           (train_combo_count, aihub_combo_count, '등장 조합 수')]):
    ax = axes[row, 0]
    ax.bar(x - 0.2, [train_values.get(c, 0) for c in order], width=0.4, label='대회 train (정제 라벨)')
    ax.bar(x + 0.2, [aihub_values.get(c, 0) for c in order], width=0.4, label='AI Hub (2-5 정제 라벨)')
    ax.set_yscale('log')
    ax.set_xticks(x, [f'K-{c:06d}' for c in order], rotation=90, fontsize=7)
    ax.set_ylabel(f'{unit} (개, 로그 눈금)')
    ax.set_title(f'대회 train 클래스 {len(train_classes)}개의 {unit}' +
                 (f'\nAI Hub에 없는 train 클래스: {", ".join(f"K-{c:06d}" for c in absent_train)} (막대 높이 0)' if row == 0 else ''))
    ax.legend()
    ax = axes[row, 1]
    values = [aihub_values[c] for c in new_classes]
    ax.bar([f'K-{c:06d}' for c in new_classes], values, color='tab:green')
    ax.tick_params(axis='x', rotation=90, labelsize=7)
    ax.set_ylabel(f'{unit} (개)')
    ax.set_title(f'train에 없는 클래스 {len(new_classes)}개의 AI Hub {unit} (최소 {min(values)}, 중앙값 {np.median(values):g})')
fig.suptitle(f'AI Hub 정제 라벨 {len(aihub_clean_records):,}장의 클래스별 객체 수와 등장 조합 수 (오른쪽은 객체 수 순서로 정렬)', fontsize=14)
save_fig(fig, '4-1_class_coverage')
plt.show()

#### 4-1 결과

| 항목 | 값 |
|---|---|
| AI Hub에 있는 대회 train 클래스 | 56개 중 54개 (없는 것: K-003351, K-003483) |
| train에 없는 클래스 | 62개 (합치면 118개) |
| train에 없는 클래스의 AI Hub 객체 수 | 최소 90개(K-020753), 중앙값 372개 |
| train에 없는 클래스의 등장 조합 수 | 최소 30개, 중앙값 124.5개 |

그림에서 보이는 것

- 위 그림은 로그 눈금이라 AI Hub 막대끼리 차이가 작아 보인다. AI Hub에 있는 대회 train 클래스 54개의 AI Hub 객체 수는 159~1,347개로 8.5배 차이 난다.
- 대회 train 막대(3~157개)는 같은 클래스의 AI Hub 막대보다 8~447배 작다.
- AI Hub에 없는 K-003351(157개), K-003483(45개)은 이 그림에서 가장 큰 파란 막대지만, 합치면 대회 train 클래스 중 가장 적은 두 클래스가 된다. (4-4)

학습 클래스는 대회 train 56개와 AI Hub에만 있는 62개를 합친 118개로 한다. 합친 뒤의 균형은 4-4의 선형 눈금 그림과 표에서 본다.

### 4-2. 조합당 이미지 수

- 조합마다 촬영 각도 70°, 75°, 90° 사진이 한 장씩 있는지 센다.
- 2-3에서 뺀 이미지 때문에 3장이 다 남지 않은 조합도 센다.

In [ ]:
def combo_angles(records):
    """조합마다 남은 사진의 촬영 각도를 모은다."""
    angles = collections.defaultdict(list)
    for fn, record in records.items():
        angles[combo_name(fn)].append(record['image']['camera_la'])
    return angles


rows = []
for name, records in [('AI Hub 촬영 PNG 전체 (1-1)', {fn: {'image': {'camera_la': int(fn.split('_')[5])}} for fn in aihub_files}),
                      ('AI Hub 정제 라벨 (2-3)', aihub_clean_records), ('대회 train (참고)', clean_records)]:
    angles = combo_angles(records)
    per_combo = collections.Counter(len(v) for v in angles.values())
    full = sum(sorted(v) == [70, 75, 90] for v in angles.values())
    rows.append([name, f'{len(records):,}장', f'{len(angles):,}개', ', '.join(f'{k}장 {v:,}개' for k, v in sorted(per_combo.items())),
                 f'{full:,}개 ({full / len(angles):.1%})'])
show_table(['라벨', '이미지', '조합', '조합당 이미지 수', '70°·75°·90°가 모두 있는 조합'], rows)

#### 4-2 결과

| 라벨 | 조합 | 세 각도가 모두 있는 조합 |
|---|---|---|
| AI Hub 촬영 PNG 전체 | 4,003개 | 4,003개 (100.0%) |
| AI Hub 정제 라벨 | 3,997개 | 3,981개 (99.6%) |
| 대회 train | 114개 | 40개 (35.1%) |

- 정제 라벨에서 2장만 남은 조합은 15개, 1장만 남은 조합은 1개다.
- 조합 6개는 사진이 모두 빠졌다. 그중 3개는 같은 알약 구성, 2개는 같은 장면으로 뺀 조합이다. (2-4)
- 대회 train은 세 각도가 다 있는 조합이 35.1%뿐이다. 같은 조합의 나머지 각도 사진 일부는 test에 있다.

조합 하나는 같은 배치를 세 각도로 찍은 사진 3장이다. 학습과 검증은 조합 단위로 나눈다.

### 4-3. 같은 조합 사진 비교

- 무작위로 고른 조합 2개의 각도별 사진 3장을 나란히 놓고 정제 라벨 box를 그린다. (seed 고정)
- 대회 train에서는 같은 조합의 75°와 90° 사진은 배치가 같고, 70° 사진은 180° 돌아가 있었다. (`00_kaggle_eda.ipynb` 6-3) AI Hub도 같은지 본다.

In [ ]:
full_combos = sorted(c for c, v in combo_angles(aihub_clean_records).items() if sorted(v) == [70, 75, 90])
picked = [full_combos[i] for i in np.random.default_rng(SEED).choice(len(full_combos), size=2, replace=False)]
fig, axes = plt.subplots(2, 3, figsize=(20, 17), layout='constrained')
for row_axes, combo in zip(axes, picked):
    files = sorted((fn for fn in aihub_clean_records if combo_name(fn) == combo),
                   key=lambda fn: aihub_clean_records[fn]['image']['camera_la'])
    for ax, fn in zip(row_axes, files):
        ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
        for ann in aihub_clean_records[fn]['annotations']:
            x, y, w, h = ann['bbox']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2))
            ax.text(x, y - 8, pill_label(ann), fontsize=8, va='bottom', bbox=dict(facecolor='lime', alpha=0.85, pad=1.5, linewidth=0))
        ax.set_title(f"{aihub_clean_records[fn]['image']['camera_la']}° | {fn}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
fig.suptitle('같은 조합의 각도별 사진 (행 = 조합, 열 = 촬영 각도 70°·75°·90°, 연두 box = 정제 라벨)', fontsize=14)
save_fig(fig, '4-3_same_combo')
plt.show()

#### 4-3 결과

그림에서 보이는 것

- 75°와 90° 사진은 알약 배치가 같고 자리만 조금 다르다.
- 70° 사진은 배치가 180° 돌아가 있다. 첫 조합에서 70° 사진의 왼쪽 위 SUSPEN ER이 75°·90° 사진에서는 오른쪽 아래에 있다.
- box는 세 각도 모두 알약을 빈틈없이 감싼다.

AI Hub도 대회 사진과 같다. AI Hub를 섞으면 배치가 180° 돌아간 70° 사진이 약 4,000장 들어온다.

### 4-4. 대회 train과 합친 분포

- 정제한 대회 train에 AI Hub 정제 라벨을 모두 더했을 때 클래스 분포가 어떻게 바뀌는지 본다.
- 불균형은 클래스별 객체 수의 최대÷최소와 변동계수(표준편차÷평균, 0이면 모든 클래스의 객체 수가 같음)로 잰다.

In [ ]:
def class_list(objects, keep):
    """조건에 맞는 클래스를 객체 수 순서로 적는다. 6개 이상이면 개수만 적는다."""
    picked = sorted((n, c) for c, n in objects.items() if keep(n))
    if len(picked) > 5:
        return f'{len(picked)}개'
    return ', '.join(f'K-{c:06d} {n}개' for n, c in picked) or '없음'


def distribution_column(records):
    """클래스 분포 표의 한 열을 만든다."""
    objects, combos = class_stats(records)
    values = np.array(sorted(objects.values()))
    total = values.sum()
    top_code, top_n = objects.most_common(1)[0]
    return [
        f'{len(records):,}장', f'{total:,}개', f'{len(objects)}개',
        f'{values[0]} / {np.median(values):g} / {values[-1]}', f'{values[-1] / values[0]:.1f}배', f'{values.std() / values.mean():.2f}',
        f'K-{top_code:06d} {top_n}개 ({top_n / total:.1%})',
        ' / '.join(str(sum(lo <= n < hi for n in values)) for lo, hi in BINS),
        class_list(objects, lambda n: n < BINS[0][1]),
        f'{min(combos.values())} / {np.median(list(combos.values())):g}',
    ]


BINS = [(0, 100), (100, 200), (200, 400), (400, 800), (800, float('inf'))]  # 객체 수 구간
combined_records = {**clean_records, **aihub_clean_records}
assert not set(clean_records) & set(aihub_clean_records), '대회 train과 AI Hub에 같은 파일명이 있다 (1-6)'
row_names = ['이미지 수', '객체 수', '클래스 수', '클래스별 객체 수 (최소 / 중앙값 / 최대)', '최대 ÷ 최소', '변동계수 (표준편차 ÷ 평균)',
             '객체가 가장 많은 클래스 (비중)', '객체 수 구간별 클래스 수 (100 미만 / 100~199 / 200~399 / 400~799 / 800 이상)',
             '객체 100개 미만 클래스', '클래스별 등장 조합 수 (최소 / 중앙값)']
show_table(['항목', '대회 train만', f'대회 train + AI Hub 전부 (2-3)'],
           [[name, before, after] for name, before, after in zip(row_names, distribution_column(clean_records),
                                                                 distribution_column(combined_records))])

- 합친 분포를 클래스마다 막대 하나로 그린다.
    - 대회 train 클래스와 train에 없는 클래스를 색으로 나누고, 그룹 안에서는 객체 수 순서로 놓는다.
    - 대회 train 클래스 막대 아래쪽의 짙은 남색은 대회 원본 객체다.

In [ ]:
def class_bar(ax, objects, title, line=None, groups=None, reference=None, mark=()):
    """클래스별 객체 수를 그룹(기본: 대회 train 클래스, train에 없는 클래스)으로 나눠 막대로 그린다.

    groups는 (그룹 이름, 클래스 목록, 색) 목록이다. reference = (이름, 클래스별 객체 수)를 주면 클래스마다 그 값을 검은 가로 표시로 겹친다.
    x축 눈금 글자는 그룹마다 처음과 끝 3개, 그리고 mark에 든 클래스만 적는다.
    """
    if groups is None:
        groups = [('대회 train 클래스', set(train_classes), 'tab:blue'),
                  ('train에 없는 클래스', set(objects) - set(train_classes), 'tab:green')]
    start, ticks, label_positions = 0, [], set()
    for name, classes, color in groups:
        order = sorted(classes, key=lambda c: (-objects[c], c))
        xs = np.arange(start, start + len(order))
        before = np.array([clean_object_count[c] for c in order])
        after = np.array([objects[c] for c in order])
        if before.any():
            ax.bar(xs, before, color='#08306b', label='대회 원본 객체 (정제 라벨)')
        ax.bar(xs, after - before, bottom=before, color=color,
               label=(f'{name} {len(order)}개 (객체 {after.min()}~{after.max()}개, 중앙값 {np.median(after):g}개)' if len(order) > 1
                      else f'{name} (객체 {after[0]:,}개)'))
        if reference:
            ax.scatter(xs, [reference[1][c] for c in order], marker='_', s=80, linewidths=2, color='black', zorder=3,
                       label=None if start else reference[0])
        if start:
            ax.axvline(start - 0.5, color='gray', linewidth=0.8, linestyle=':')
        ticks += order
        label_positions |= set(xs[:3]) | set(xs[-3:])
        start += len(order)
    if line:
        ax.axhline(line, color='black', linewidth=1, linestyle='--', label=f'목표 객체 수 {line}개')
    # 클래스가 많아 눈금 글자를 다 적으면 읽을 수 없다. 그룹마다 처음과 끝 3개와 mark에 든 클래스만 적는다
    shown = sorted({i for i in label_positions} | {i for i, c in enumerate(ticks) if c in set(mark)})
    ax.set_xticks(shown, [f'K-{ticks[i]:06d}' if isinstance(ticks[i], int) else ticks[i] for i in shown], rotation=90, fontsize=8)
    ax.set_xlim(-1, len(ticks))
    ax.set_ylabel('객체 수 (개)')
    ax.set_title(title)
    ax.legend(loc='upper right', fontsize=9)


combined_objects, _ = class_stats(combined_records)
fig, ax = plt.subplots(figsize=(24, 7), layout='constrained')
class_bar(ax, combined_objects, f'대회 train {len(clean_records)}장 + AI Hub 전부 {len(aihub_clean_records):,}장의 클래스 {len(combined_objects)}개 객체 수 '
          f'(최대÷최소 {max(combined_objects.values()) / min(combined_objects.values()):.1f}배)')
save_fig(fig, '4-4_combined_classes')
plt.show()

#### 4-4 결과

| 항목 | 대회 train만 | 대회 train + AI Hub 전부 |
|---|---|---|
| 이미지 | 232장 | 12,206장 |
| 객체 | 771개 | 46,590개 |
| 클래스 | 56개 | 118개 |
| 클래스별 객체 수 (최소 / 중앙값 / 최대) | 3 / 9 / 157 | 45 / 369.5 / 1,365 |
| 최대 ÷ 최소 | 52.3배 | 30.3배 |
| 변동계수 | 1.53 | 0.47 |
| 가장 많은 클래스 (비중) | K-003351 (20.4%) | K-016548 (2.9%) |
| 객체 100개 미만 클래스 | 55개 | 2개 (K-003483 45개, K-020753 90개) |
| 클래스별 등장 조합 수 (최소 / 중앙값) | 1 / 3.5 | 15 / 124 |

그림에서 보이는 것

- K-016548(1,365개)과 K-016551(1,344개)이 크게 많다. 그다음은 K-016232, K-016262로 약 820개다. 모두 AI Hub의 많은 조합에 함께 들어 있다.
- 대회 train 클래스 56개의 중앙값은 346.5개, train에 없는 클래스 62개는 372개로 비슷하다.
- 대회 원본 객체(짙은 남색)는 대부분 막대 아래쪽에 얇게만 보인다. K-003351(157개)과 K-003483(45개)만 막대 전체가 대회 원본 객체다.

AI Hub를 더하면 불균형이 크게 준다. 남은 불균형은 양 끝 클래스에서 생긴다. 가장 적은 K-003483은 AI Hub에 없어 채울 수 없고, 가장 많은 K-016548은 4-5에서 덜 넣는 방법을 비교한다.

### 4-5. 가장 균일하게 고르기

- 클래스마다 목표 객체 수를 정하고, 목표에 못 미친 클래스를 채우는 AI Hub 이미지만 고른다.
    - 사진에는 알약 3~4개가 함께 있어서, 이미지를 넣으면 그 안의 알약이 모두 들어간다. 완전히 같은 수로 맞출 수는 없다.
    - 객체가 적은 클래스부터 하나씩, 그 클래스가 목표를 채울 때까지 그 클래스가 있는 이미지를 넣는다.
    - 이미 목표를 채운 클래스가 적게 함께 찍힌 이미지부터 넣어, 흔한 클래스가 덜 늘게 한다.
- 목표별로 고른 이미지 수와 불균형을 AI Hub를 전부 쓸 때와 비교한다.
- test에 나오는 클래스를 기준으로 고르는 방법은 5-4~5-6에서 비교한다.

In [ ]:
TARGETS = [100, 200, 300, 400]
image_classes = {fn: [a['category_id'] for a in r['annotations']] for fn, r in aihub_clean_records.items()}
all_classes = sorted(set(train_classes) | aihub_classes)


images_by_class = collections.defaultdict(list)  # K-코드 → 그 클래스가 있는 AI Hub 이미지
for fn, classes in sorted(image_classes.items()):
    for c in set(classes):
        images_by_class[c].append(fn)
total_objects = collections.Counter(clean_object_count) + collections.Counter(c for cs in image_classes.values() for c in cs)


def plan_uniform(target):
    """객체가 적은 클래스부터 목표 객체 수를 채우도록 AI Hub 이미지를 고른다."""
    count = collections.Counter(clean_object_count)  # 대회 train(정제 라벨)은 모두 넣고 시작한다
    chosen = set()
    for c in sorted(all_classes, key=lambda c: (total_objects[c], c)):
        # 이미 목표를 채운 클래스가 적게 함께 찍힌 이미지부터 넣는다
        pool = sorted((fn for fn in images_by_class[c] if fn not in chosen),
                      key=lambda fn: (sum(count[o] >= target for o in image_classes[fn]), fn))
        for fn in pool:
            if count[c] >= target:
                break
            chosen.add(fn)
            count.update(image_classes[fn])
    return sorted(chosen), count


def balance_row(name, objects, n_images):
    values = np.array([objects[c] for c in all_classes])
    return [name, f'{n_images:,}장', f'{values.sum():,}개', f'{values.min()} / {np.median(values):g} / {values.max()}',
            f'{values.max() / values.min():.1f}배', f'{values.std() / values.mean():.2f}']


uniform_plans = {t: plan_uniform(t) for t in TARGETS}
rows = [balance_row('AI Hub 전부', combined_objects, len(aihub_clean_records))]
for t, (chosen, count) in uniform_plans.items():
    short = sum(count[c] < t for c in all_classes)
    rows.append(balance_row(f'목표 {t}개 (못 미친 클래스 {short}개)', count, len(chosen)))
show_table(['AI Hub 이미지 선택', 'AI Hub 이미지 수', '객체 수 (대회 train 포함)', '클래스별 객체 수 (최소 / 중앙값 / 최대)', '최대 ÷ 최소', '변동계수'], rows)

- 목표 200개로 고를 때와 전부 쓸 때의 클래스별 객체 수를 나란히 그린다. 고른 이미지 목록은 파일로 쓰지 않는다.

In [ ]:
SELECTED_TARGET = 200  # 4-5 표를 보고 정한다
selected, selected_count = uniform_plans[SELECTED_TARGET]  # 고른 이미지 목록은 파일로 쓰지 않는다

fig, axes = plt.subplots(2, 1, figsize=(24, 13), layout='constrained', sharex=False)
class_bar(axes[0], combined_objects, f'AI Hub 전부 {len(aihub_clean_records):,}장 (최대÷최소 {max(combined_objects.values()) / min(combined_objects.values()):.1f}배)')
class_bar(axes[1], selected_count, f'목표 {SELECTED_TARGET}개로 고른 AI Hub {len(selected):,}장 '
          f'(최대÷최소 {max(selected_count[c] for c in all_classes) / min(selected_count[c] for c in all_classes):.1f}배)', line=SELECTED_TARGET)
fig.suptitle('대회 train + AI Hub의 클래스별 객체 수: 전부 쓸 때(위)와 목표 객체 수로 고를 때(아래)', fontsize=14)
save_fig(fig, '4-5_uniform_selection')
plt.show()

#### 4-5 결과

| 선택 | AI Hub 이미지 | 클래스별 객체 수 (최소 / 중앙값 / 최대) | 최대 ÷ 최소 | 변동계수 |
|---|---|---|---|---|
| AI Hub 전부 | 11,974장 | 45 / 369.5 / 1,365 | 30.3배 | 0.47 |
| 목표 100개 | 3,643장 | 45 / 105 / 584 | 13.0배 | 0.46 |
| 목표 200개 | 6,998장 | 45 / 200 / 944 | 21.0배 | 0.43 |
| 목표 300개 | 9,939장 | 45 / 300 / 1,335 | 29.7배 | 0.46 |
| 목표 400개 | 11,511장 | 45 / 369.5 / 1,365 | 30.3배 | 0.46 |

- 최솟값은 모든 행에서 45개(K-003483)다. AI Hub에 없는 클래스라 고르는 방법과 상관없다.
- 변동계수는 0.43~0.47로 크게 바뀌지 않는다. 목표 200개가 0.43으로 가장 낮다.
    - 목표 100개에서도 최댓값이 584개다. 적은 클래스를 채우려고 넣은 사진에 흔한 클래스가 함께 찍혀 있기 때문이다.
- 목표 200개는 AI Hub 6,998장(전부의 58%)으로 클래스 중앙값을 200개에 맞춘다.

그림에서 보이는 것

- 목표 200개로 고르면 대회 train 클래스는 45~944개(중앙값 200개), train에 없는 클래스는 90~378개(중앙값 206개)가 된다.
- 대부분의 클래스가 200개(점선) 근처에 모인다. K-016548(944개)과 K-016551(약 720개)은 여전히 많다.
- 목표에 못 미친 15개 클래스는 가진 객체를 모두 넣었다.

목표 200개 목록(6,998장)을 보충 데이터 후보 1로 둔다. 변동계수가 가장 낮고 이미지가 전부의 58%라 학습이 빠르다. 다만 전부 쓸 때와 차이(0.43과 0.47)가 작아서, 어느 쪽을 쓸지는 val mAP로 정한다. (5-7)

### 4-6. 정리

#### 확인한 것

- 합친 클래스는 118개다. AI Hub에 대회 train 클래스 54개와 새 클래스 62개가 있다.
- AI Hub를 모두 더하면 변동계수가 1.53에서 0.47로, 최대÷최소가 52.3배에서 30.3배로 준다.
- 객체 100개 미만 클래스는 K-003483(45개, AI Hub에 없음)과 K-020753(90개)뿐이다.
- 조합마다 세 각도 사진이 있고, 같은 조합의 사진은 같은 배치다. (70°는 180° 회전)
- 목표 객체 수로 골라도 변동계수는 0.43~0.47로 크게 나아지지 않는다.

#### 정한 것

- 학습 클래스는 대회 train 56개 + AI Hub에만 있는 62개 = 118개다. (4-1)
- train과 val은 조합 단위로 나눈다. 분할 담당 팀원에게 전달한다. (4-2, 4-3)
- 보충 이미지는 목표 200개 목록(6,998장)을 후보 1로 두고, AI Hub 전부와 val에서 비교한다. (4-4, 4-5)
- 클래스 가중치나 반복 샘플링 없이 시작한다. (4-4)

#### 뒤 섹션에서 볼 것

- test에만 나오는 알약이 118개 클래스에 있는지 5절에서 본다.
- K-003351, K-003483은 AI Hub로 채울 수 없어서 val에서 AP를 따로 본다.

## 5. test 전용 클래스 채우기

test 기록표(`00_test_classes_review.csv`, test 사진을 직접 보고 train에 없는 알약을 적은 표)에는 알약 18종(각인 19종, 213과 30은 같은 알약의 앞뒤)이 있다. test 842장 중 335장(39.8%)에 나온다. 이 알약들이 AI Hub로 채워지는지 정제 라벨(2-3)로 확인한다.

| 단계 | 하는 일 | 이유 |
|---|---|---|
| 5-1 | test 각인 대조 | 기록표의 각인을 AI Hub 라벨에서 찾는다 |
| 5-2 | test와 AI Hub 알약 비교 | 찾은 클래스가 같은 약인지 사진으로 본다 |
| 5-3 | test 사진 커버리지 | 보충 뒤 모든 알약이 학습 클래스에 있는 test 사진의 비율을 센다 |
| 5-4~5-6 | 이미지 고르는 방법 셋 | test에 나오지 않는 클래스를 어떻게 다룰지 정한다 |
| 5-7 | 정리 | 고르는 방법 총정리 표, 확인한 것과 정한 것 |

### 5-1. test 각인 대조

- 기록표의 각인을 AI Hub 라벨의 각인 필드(print_front, print_back)와 마크 설명 필드(mark_code_front_anal, mark_code_back_anal)에서 찾는다.
    - 회사 로고 같은 그림 각인은 각인 필드에 "마크"라고만 적고, 로고 글자는 마크 설명 필드에 적는다. (예: 각인 필드 "마크", 마크 설명 "Hanmi")
- 띄어쓰기와 대소문자는 무시한다. 필드 값 전체가 같은 경우를 먼저 찾고, 없으면 값에 각인이 들어 있는 경우(부분 일치)를 찾는다.
- AI Hub는 분할선을 "분할선"이라는 글자로 적으므로, 기록표의 `|`를 바꿔서 찾는다. (`K|50` → `K분할선50`)

In [ ]:
def normalize(text):
    """띄어쓰기와 대소문자를 무시하고 비교하도록 각인 글자를 정리한다."""
    return str(text or '').replace(' ', '').lower()


aihub_pill_info = {}  # K-코드 → 그 알약 JSON의 이미지 메타데이터
for record in aihub_clean_records.values():
    for ann in record['annotations']:
        aihub_pill_info.setdefault(ann['category_id'], ann['image'])
filled = sum(bool(im['print_front'] or im['print_back']) for im in aihub_pill_info.values())
IMPRINT_KEYS = ['print_front', 'print_back', 'mark_code_front_anal', 'mark_code_back_anal']  # 각인 필드와 마크 설명 필드
imprint_fields = {c: tuple(normalize(im.get(key)) for key in IMPRINT_KEYS) for c, im in aihub_pill_info.items()}

files_by_imprint = collections.defaultdict(set)  # 기록표의 각인 → test 파일 번호
look_by_imprint = {}  # 기록표의 각인 → test에서 본 색과 모양
imprint_matches = {}  # 기록표의 각인 → AI Hub에서 찾은 K-코드
for row in review_rows:
    if row['imprint'] not in ('', '?'):  # 각인을 읽지 못한 알약은 글자로 대조할 수 없어 뺀다
        files_by_imprint[row['imprint']].add(int(row['test_file'].split('.')[0]))
        look_by_imprint.setdefault(row['imprint'], f"{row['color']} {row['shape']}")

rows = []
for seen, numbers in sorted(files_by_imprint.items(), key=lambda kv: (-len(kv[1]), kv[0])):
    query = normalize(seen.replace('|', '분할선'))
    match, hits = '같음', [c for c, fields in imprint_fields.items() if query in fields]
    if not hits and len(query) >= 2:
        match, hits = '부분 일치', [c for c, fields in imprint_fields.items() if any(query in f for f in fields)]
    imprint_matches[seen] = hits
    shown = ', '.join(map(str, sorted(numbers)[:6])) + (' …' if len(numbers) > 6 else '')
    for c in hits:
        info = aihub_pill_info[c]
        rows.append([seen, look_by_imprint[seen], shown, f'K-{c:06d}', info['dl_name'], f"{info['color_class1']} {info['drug_shape']}",
                     match, '있음' if c in train_classes else '없음', aihub_object_count[c]])
    if not hits:
        rows.append([seen, look_by_imprint[seen], shown, '찾지 못함', '-', '-', '-', '-', '-'])
rows.append([f'**각인 필드가 있는 클래스 {filled}/{len(aihub_pill_info)}**', '', '', '', '', '', '', '', ''])
show_table(['test에서 읽은 각인', 'test 색, 모양', 'test 파일 번호', 'AI Hub K-코드', '제품명', 'AI Hub 색, 모양', '일치',
            'train에 있음', 'AI Hub 객체 수'], rows)

#### 5-1 결과

- test에서 읽은 각인 19종을 모두 AI Hub 클래스에서 찾았다.
    - 18종은 필드 값 전체가 같고, 213만 부분 일치(K-018110의 각인 "마크213/30")다.
    - 213과 30은 같은 클래스 K-018110(란스톤엘에프디티정 30mg)의 앞뒤다.
- Hanmi는 회사 로고라 두 클래스가 찾아졌다. (K-031705 낙소졸정, K-025469 아모잘탄정) K-025469는 대회 train에 이미 있는 클래스다.
- K-025469를 빼면 찾은 클래스는 모두 train에 없는 클래스이고, AI Hub 객체가 293~467개씩 있다.
- 기록표와 AI Hub의 색 이름이 다른 경우가 있다. (C77 빨강과 갈색, PE 하양과 노랑, Q25 빨강과 분홍)
- AI Hub 116개 클래스 중 114개에 각인 필드가 있다.

찾은 클래스가 test 알약과 같은 약인지 5-2에서 사진으로 본다. 색 이름이 달라도 후보에서 빼지 않는다.

### 5-2. test 알약과 AI Hub 알약 나란히 보기

- 각인마다 한 줄에 왼쪽 2칸은 test 사진, 오른쪽 3칸은 5-1에서 찾은 AI Hub 클래스의 crop이다.
    - test 사진은 기록표의 위치 쪽 모서리에서 620×780픽셀을 자른다. 각인마다 서로 다른 조합의 사진 2장을 고른다.
    - AI Hub crop은 정제 라벨 box를 정사각형으로 자른 것이다. 서로 다른 조합에서 3개를 고른다. (seed 고정)
- 색, 모양, 크기, 각인으로 같은 알약인지 본다.

In [ ]:
POS_CROP_W, POS_CROP_H = 620, 780  # 기록표 위치 쪽 모서리에서 자를 크기
N_AIHUB = 3  # 각인마다 보여 줄 AI Hub crop 수


def crop_position(image, position):
    """기록표의 위치(예: 왼쪽 위) 쪽 모서리에서 POS_CROP_W×POS_CROP_H 크기로 자른다."""
    x0 = 0 if '왼쪽' in position else IMG_W - POS_CROP_W
    y0 = 0 if '위' in position else IMG_H - POS_CROP_H
    return image.crop((x0, y0, x0 + POS_CROP_W, y0 + POS_CROP_H))


def test_examples(seen, n=2):
    """기록표에서 그 각인이 나오는 줄을 서로 다른 조합에서 n개 고른다. (test 번호 3개가 한 조합)"""
    first_of_combo = {}
    for row in sorted((r for r in review_rows if r['imprint'] == seen), key=lambda r: int(r['test_file'].split('.')[0])):
        first_of_combo.setdefault((int(row['test_file'].split('.')[0]) - 1) // 3, row)
    return list(first_of_combo.values())[:n]


aihub_class_anns = collections.defaultdict(list)  # K-코드 → (이미지, annotation) 목록
for fn in sorted(aihub_clean_records):
    for ann in aihub_clean_records[fn]['annotations']:
        aihub_class_anns[ann['category_id']].append((fn, ann))


def aihub_examples(code, n=N_AIHUB, seed=SEED):
    """그 클래스의 annotation을 서로 다른 조합에서 n개 무작위로 고른다."""
    by_combo = {}
    for fn, ann in aihub_class_anns[code]:
        by_combo.setdefault(combo_name(fn), []).append((fn, ann))
    combos = sorted(by_combo)
    picks = np.random.default_rng(seed).choice(len(combos), size=min(n, len(combos)), replace=False)
    return [by_combo[combos[i]][0] for i in sorted(picks)]


lines_to_show = [(seen, c) for seen in sorted(files_by_imprint, key=lambda s: (-len(files_by_imprint[s]), s))
                 for c in (imprint_matches[seen] or [None])]
fig, axes = plt.subplots(len(lines_to_show), 2 + N_AIHUB, figsize=(16, 2.0 * len(lines_to_show)), layout='constrained',
                         gridspec_kw={'width_ratios': [1.6, 1.6] + [1] * N_AIHUB})
for row_axes, (seen, c) in zip(axes, lines_to_show):
    for ax, row in zip(row_axes[:2], test_examples(seen)):
        ax.imshow(crop_position(Image.open(TEST_IMG_DIR / row['test_file']).convert('RGB'), row['position']))
        ax.set_title(f"test {row['test_file']} {row['position']}\n기록표: {seen} ({row['color']} {row['shape']})", fontsize=9)
    if c is None:
        row_axes[2].text(0.5, 0.5, 'AI Hub 각인 필드에서\n찾지 못함', ha='center', va='center', fontsize=12)
    else:
        for ax, (fn, ann) in zip(row_axes[2:], aihub_examples(c)):
            ax.imshow(crop_square(Image.open(AIHUB_IMG_DIR / fn).convert('RGB'), ann['bbox']))
            ax.set_title(f"AI Hub {pill_label(ann)}\n{ann['image']['color_class1']} {ann['image']['drug_shape']}", fontsize=9)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('train에 없는 test 알약(왼쪽 2칸, 기록표 위치 쪽 모서리)과 각인으로 찾은 AI Hub 알약(오른쪽 3칸, 정제 라벨 box)', fontsize=14)
save_fig(fig, '5-2_test_vs_aihub')
plt.show()

#### 5-2 결과

그림에서 보이는 것

- Hanmi를 뺀 각인 18개는 찾은 AI Hub 클래스가 test 알약과 같은 약으로 보인다. 각인, 모양, 색이 맞는다.
    - 213과 30은 K-018110의 앞면(마크 213)과 뒷면(30)이다.
- Hanmi는 두 행으로 그렸다. 두 행의 test 사진(88.png, 298.png)은 같고 AI Hub crop만 다르다.

| Hanmi 후보 | test 알약과 | 이유 |
|---|---|---|
| K-031705 (낙소졸정) | 같은 약 | 둘 다 노랑 타원형이고 타원 테두리 안에 Hanmi가 있다. crop 하나에는 뒷면 각인 Naxozol 500/20이 보인다 |
| K-025469 (아모잘탄정) | 다른 약 | 분홍 타원형이고, 보이는 면의 각인은 Amosartan 5/100이다. 대회 train에 이미 있는 클래스다 |

test에만 나오는 알약 18종은 모두 AI Hub 클래스로 학습할 수 있다. Hanmi는 K-031705로만 세고, K-025469는 5-3에서 다른 약으로 표시해 뺀다.

### 5-3. test 사진 커버리지

- test 사진마다 기록표에 적힌 train에 없는 알약이 모두 학습 클래스로 채워지는지 센다.
    - 기록표에 줄이 없는 test 사진은 모든 알약이 train에 있는 사진이다.
    - 보충 뒤에는 5-1에서 찾고 5-2에서 같은 약으로 확인한 각인을 채워진 것으로 본다.
    - 5-2에서 다른 약으로 확인한 K-코드(Hanmi의 K-025469)는 빼고, 남은 K-코드가 하나인 각인만 센다.

In [ ]:
# 5-2 그림에서 test 알약과 다른 약으로 확인한 (각인, K-코드). 각인 글자만 같다
DIFFERENT_PILLS = {
    ('Hanmi', 25469): '아모잘탄정은 분홍 타원형이다. test의 Hanmi 알약은 노랑 타원형인 K-031705(낙소졸정, 뒷면 Naxozol 500/20)이다',
}
same_pills = {seen: [c for c in hits if (seen, c) not in DIFFERENT_PILLS] for seen, hits in imprint_matches.items()}
code_of_imprint = {seen: codes[0] for seen, codes in same_pills.items() if len(codes) == 1}  # 5-2에서 같은 알약으로 확인한 각인
new_pills_by_file = collections.defaultdict(set)  # test 파일 → 그 사진의 train에 없는 알약 각인
for row in review_rows:
    new_pills_by_file[row['test_file']].add(row['imprint'])
covered_after = [f for f, imprints in new_pills_by_file.items() if all(i in code_of_imprint for i in imprints)]
left = collections.Counter(i for f, imprints in new_pills_by_file.items() for i in imprints if i not in code_of_imprint)
n_test = len(test_files)
rows = [
    ['모든 알약이 학습 클래스에 있는 test 사진', f'{n_test - len(new_pills_by_file)}장 ({(n_test - len(new_pills_by_file)) / n_test:.1%})',
     f'{n_test - len(new_pills_by_file) + len(covered_after)}장 ({(n_test - len(new_pills_by_file) + len(covered_after)) / n_test:.1%})'],
    ['train에 없는 알약이 하나라도 있는 test 사진', f'{len(new_pills_by_file)}장', f'{len(new_pills_by_file) - len(covered_after)}장'],
    ['학습 클래스에 없는 알약 (각인: 나오는 test 사진 수)', f'{len(files_by_imprint)}종',
     ', '.join(f'{i}: {len({r["test_file"] for r in review_rows if r["imprint"] == i})}장' for i in sorted(left)) or '없음'],
]
show_table(['항목', '보충 전 (대회 train만)', '보충 후 (+ AI Hub)'], rows)

#### 5-3 결과

| 항목 | 보충 전 (대회 train만) | 보충 후 (+ AI Hub) |
|---|---|---|
| 모든 알약이 학습 클래스에 있는 test 사진 | 507장 (60.2%) | 842장 (100.0%) |
| train에 없는 알약이 하나라도 있는 test 사진 | 335장 | 0장 |

- 보충하지 않으면 test 사진의 39.8%에 맞힐 수 없는 알약이 있다.
- 이 수치는 test 기록표 기준이다. 기록표에서 놓친 알약이 있으면 그만큼 줄어든다.

AI Hub 보충은 꼭 필요하다. 보충 데이터에는 test 전용 알약 18종의 클래스가 반드시 들어가야 한다.

### 5-4. 대회 클래스만 든 이미지로 고르기

- 대회 클래스는 대회 train 클래스 56개와 test 전용 알약 18종을 합친 74개다.
    - test 사진의 알약은 모두 대회 클래스다. (5-3)
    - AI Hub에만 있는 나머지 44개 클래스는 test에 나오지 않는다.
- 4-5는 클래스마다 목표 객체 수를 채우도록 골랐다. 5-4~5-6은 대회 클래스를 기준으로 고르는 방법 셋을 비교한다.

| 방법 | 고르는 AI Hub 이미지 | 대회 클래스 밖 알약의 라벨 |
|---|---|---|
| 5-4 | 모든 알약이 대회 클래스인 이미지 | (그런 알약이 없다) |
| 5-5 | 전부 | others 클래스 하나로 묶는다 |
| 5-6 | 대회 클래스가 하나라도 든 이미지 | 원래 클래스 그대로 |

- 그림은 4-4처럼 그린다. 대회 train 232장은 모두 넣고, 막대 아래쪽 짙은 남색이 대회 원본 객체다.
    - 검은 가로 표시는 AI Hub를 전부 쓸 때의 객체 수다. 막대가 표시보다 낮으면 고르면서 그만큼 잃은 것이다.
    - 불균형(최대÷최소, 변동계수)은 대회 클래스 74개로만 잰다.

In [ ]:
# 대회 클래스: 대회 train 클래스 56개 + test에만 나오는 알약 18종 (5-2에서 같은 알약으로 확인한 AI Hub 클래스)
test_only_classes = sorted(set(code_of_imprint.values()) - set(train_classes))
contest_classes = set(train_classes) | set(test_only_classes)
assert len(test_only_classes) == 18, f'5-2에서 확인한 test 전용 알약은 18종인데 {len(test_only_classes)}개이다. 5-1의 각인 대조를 확인한다.'
CONTEST_GROUPS = [('대회 train 클래스', train_classes, 'tab:blue'), ('test 전용 클래스', test_only_classes, 'tab:orange')]
ALL_AIHUB = ('AI Hub를 전부 쓸 때의 객체 수 (4-4)', combined_objects)  # 비교용 검은 가로 표시


def count_objects(files, others=False):
    """대회 train(정제 라벨)과 고른 AI Hub 이미지의 클래스별 객체 수를 센다. others=True면 대회 클래스 밖 알약을 'others'로 센다."""
    count = collections.Counter(clean_object_count)
    for fn in files:
        count.update('others' if others and c not in contest_classes else c for c in image_classes[fn])
    return count


def contest_summary(count):
    """대회 클래스의 객체 수와 불균형 정도(4-4와 같은 정의)를 그림 제목에 넣을 글로 만든다."""
    values = np.array([count[c] for c in sorted(contest_classes)])
    ratio = f'{values.max() / values.min():.1f}배' if values.min() else '계산 불가 (최소 0개)'
    return (f'대회 클래스 {len(values)}개의 객체 합계 {values.sum():,}개, 클래스별 {values.min():,}~{values.max():,}개 (중앙값 {np.median(values):g}개), '
            f'최대÷최소 {ratio}, 변동계수 {values.std() / values.mean():.2f}')


def mark_empty(ax, count):
    """객체가 0개인 클래스의 눈금 글자를 빨갛게 칠한다."""
    for label in ax.get_xticklabels():
        if label.get_text().startswith('K-') and count[int(label.get_text()[2:])] == 0:
            label.set_color('red')


only_contest_files = [fn for fn, classes in image_classes.items() if set(classes) <= contest_classes]
only_contest_count = count_objects(only_contest_files)
in_aihub = [c for c in contest_classes if combined_objects[c] > clean_object_count[c]]  # AI Hub에 있는 대회 클래스 (K-003351, K-003483 제외)
kept_all = sum(only_contest_count[c] == combined_objects[c] for c in in_aihub)
kept_none = sum(only_contest_count[c] == clean_object_count[c] for c in in_aihub)
fig, ax = plt.subplots(figsize=(24, 7.5), layout='constrained')
class_bar(ax, only_contest_count,
          f'모든 알약이 대회 클래스인 AI Hub {len(only_contest_files):,}장(전부 {len(image_classes):,}장의 {len(only_contest_files) / len(image_classes):.0%})'
          f' + 대회 train {len(clean_records)}장의 클래스별 객체 수 (빨간 글자 = 객체 0개인 클래스 {sum(only_contest_count[c] == 0 for c in contest_classes)}개)\n'
          f'고른 뒤: {contest_summary(only_contest_count)}',
          groups=CONTEST_GROUPS, reference=ALL_AIHUB, mark=[c for c in contest_classes if only_contest_count[c] == 0])
mark_empty(ax, only_contest_count)
save_fig(fig, '5-4_only_contest_images')
plt.show()

#### 5-4 결과

- 모든 알약이 대회 클래스인 AI Hub 이미지는 2,688장으로 전부(11,974장)의 22%다.

그림에서 보이는 것

- 클래스마다 AI Hub 객체가 모두 남거나 하나도 남지 않는다. AI Hub에 있는 대회 클래스 72개 중 26개는 모두 남고, 46개는 하나도 남지 않는다.
    - 하나도 남지 않은 클래스는 AI Hub에서 늘 대회 클래스 밖 알약과 한 사진에 찍혀 있다는 뜻이다.
- test 전용 클래스 18개 중 7개(빨간 글자)는 객체가 0개가 된다. (K-004378, K-005094, K-005886, K-006192, K-022627, K-023223, K-027653)
- 대회 train 클래스 56개 중 39개는 대회 원본 객체(대부분 40개 미만)만 남는다.
- 대회 클래스의 객체 수 중앙값이 348개에서 17.5개로 줄고, 변동계수는 0.53에서 1.62로 커진다.

이 방법은 쓰지 않는다. 5-3에서 채운 test 전용 알약 7종을 다시 학습할 수 없게 된다.

### 5-5. 대회 클래스 밖 알약을 others로 묶기

- AI Hub 이미지를 모두 쓰고, 대회 클래스 밖 44개 클래스의 라벨을 others 하나로 바꾼다. 대회 클래스 라벨은 그대로 둔다.
    - 라벨을 지우지 않고 others로 바꾸는 이유: 라벨을 지우면 그 알약이 배경으로 학습된다.
- 점선은 대회 클래스 74개의 객체 수 중앙값이다.

In [ ]:
other_classes = sorted({c for classes in image_classes.values() for c in classes} - contest_classes)  # AI Hub에만 있고 test에 안 나오는 클래스
others_count = count_objects(image_classes, others=True)  # AI Hub 이미지는 모두 쓴다
any_contest_files = [fn for fn, classes in image_classes.items() if set(classes) & contest_classes]  # 5-6에서도 쓴다
others_in_any = count_objects(any_contest_files, others=True)['others']
contest_values = np.array([others_count[c] for c in sorted(contest_classes)])
top_code = max(sorted(contest_classes), key=lambda c: others_count[c])
other_values = [combined_objects[c] for c in other_classes]

fig, ax = plt.subplots(figsize=(24, 7.5), layout='constrained')
class_bar(ax, others_count,
          f'AI Hub 전부 {len(image_classes):,}장 + 대회 train {len(clean_records)}장에서 대회 클래스 밖 {len(other_classes)}개 클래스를 others 하나로 묶은 클래스별 객체 수\n'
          f'{contest_summary(others_count)} (AI Hub 전부와 같음)',
          groups=CONTEST_GROUPS + [(f'others: 대회 클래스 밖 {len(other_classes)}개 클래스, 묶기 전 클래스마다 {min(other_values)}~{max(other_values)}개',
                                    ['others'], 'tab:gray')])
ax.axhline(np.median(contest_values), color='black', linewidth=1, linestyle='--',
           label=f'대회 클래스 {len(contest_values)}개의 중앙값 {np.median(contest_values):g}개')
ax.text(len(contest_classes) - 0.8, others_count['others'],
        f"others {others_count['others']:,}개\n= 대회 클래스 중앙값의 {others_count['others'] / np.median(contest_values):.0f}배\n"
        f"= 가장 많은 K-{top_code:06d}({others_count[top_code]:,}개)의 {others_count['others'] / others_count[top_code]:.1f}배\n"
        f"= 전체 객체의 {others_count['others'] / sum(others_count.values()):.0%}",
        ha='right', va='top', fontsize=11)
ax.legend(loc='upper left', fontsize=9)
save_fig(fig, '5-5_others_class')
plt.show()

#### 5-5 결과

| 항목 | 값 |
|---|---|
| others 객체 수 | 17,289개 (전체 객체의 37%) |
| 대회 클래스 중앙값(348개)과 비교 | 50배 |
| 가장 많은 대회 클래스 K-016548(1,365개)과 비교 | 12.7배 |

- 대회 클래스가 하나라도 든 9,742장(5-6)에만 적용해도 others는 8,361개로, K-016548의 6배가 넘는다.
- others에는 생김새가 다른 알약 44종이 섞인다. 묶기 전에는 클래스마다 90~642개로 대회 클래스(45~1,365개)와 비슷한 규모였다.
- 대회 클래스 74개의 객체 수는 AI Hub를 전부 쓸 때와 같다.

others로 묶지 않는다. 생김새가 제각각인 others가 가장 큰 클래스가 되어, 헷갈리는 대회 클래스 알약을 others로 답하는 쪽으로 모델이 기울기 때문이다. test에는 others 알약이 없어서 이런 답은 모두 놓친 알약이 된다.

- 원래 라벨을 그대로 둘 때보다 나은 점이 없다. 클래스 수가 118개에서 75개로 줄어도, 검출 모델에서는 마지막 출력 층의 크기만 바뀐다.
- 제출할 때는 대회 클래스 밖 클래스로 예측한 box를 지운다. test에는 대회 클래스 알약만 있어서 맞을 수 없는 예측이다.

### 5-6. 대회 클래스가 하나라도 든 이미지로 고르기

- 대회 클래스 알약이 하나라도 있는 AI Hub 이미지만 고르고, 라벨은 바꾸지 않는다. 모든 알약이 대회 클래스 밖인 이미지만 빠진다.
- 대회 클래스 밖 알약은 원래 클래스로 남는다. (초록 막대)
- 제목 넷째 줄에 4-5의 목표 200개 목록을 같은 대회 클래스 74개로 잰 값을 함께 적는다.

In [ ]:
any_contest_count = count_objects(any_contest_files)
mixed = sum(bool(set(image_classes[fn]) - contest_classes) for fn in any_contest_files)  # 대회 클래스 밖 알약도 함께 찍힌 이미지
target_files, target_count = uniform_plans[SELECTED_TARGET]
fig, ax = plt.subplots(figsize=(24, 7.5), layout='constrained')
class_bar(ax, any_contest_count,
          f'대회 클래스가 하나라도 든 AI Hub {len(any_contest_files):,}장(전부의 {len(any_contest_files) / len(image_classes):.0%})'
          f' + 대회 train {len(clean_records)}장의 클래스별 객체 수 (빨간 글자 = 객체 0개인 클래스 {sum(any_contest_count[c] == 0 for c in other_classes)}개)\n'
          f'고른 뒤: {contest_summary(any_contest_count)} (AI Hub 전부와 같음)\n'
          f'4-5 목표 {SELECTED_TARGET}개 목록 {len(target_files):,}장: {contest_summary(target_count)}',
          groups=CONTEST_GROUPS + [('대회 클래스 밖 클래스', other_classes, 'tab:green')], reference=ALL_AIHUB,
          mark=[c for c in other_classes if any_contest_count[c] == 0])
mark_empty(ax, any_contest_count)
save_fig(fig, '5-6_any_contest_images')
plt.show()

#### 5-6 결과

| 선택 | AI Hub 이미지 | 대회 클래스 객체 합계 | 변동계수 (대회 클래스 74개) |
|---|---|---|---|
| AI Hub 전부 | 11,974장 | 29,301개 | 0.53 |
| 대회 클래스가 하나라도 든 이미지 (5-6) | 9,742장 (81%) | 29,301개 | 0.53 |
| 목표 200개 (4-5) | 6,998장 | 17,317개 | 0.50 |

- 빠지는 2,232장은 모든 알약이 대회 클래스 밖이다.
- 고른 9,742장 중 7,054장에는 대회 클래스 밖 알약도 함께 있다. 5-4는 이 이미지를 모두 뺐다.
- 대회 클래스 밖 44개 중 22개는 객체가 모두 남고, 22개(빨간 글자)는 하나도 남지 않는다.
    - 빠지는 22개는 대회 클래스와 한 사진에 찍힌 적이 없는 클래스다. 4-5에서 객체가 적은 클래스로 꼽은 K-020753(90개)도 여기 든다.
- 4-5 표의 변동계수(목표 200개 0.43, 전부 0.47)는 test에 나오지 않는 44개 클래스까지 넣은 118개 기준이다.

5-6 목록(9,742장)도 보충 데이터 후보로 둔다. 대회 클래스 객체를 하나도 버리지 않으면서 AI Hub 전부보다 이미지가 19% 적고, 빠지는 이미지에는 test 알약이 없다.

### 5-7. 정리

- 4-5와 5-4~5-6에서 본 AI Hub 이미지 고르는 방법을 한 표로 모은다. 수치는 모두 대회 train 232장에 고른 AI Hub 이미지를 더한 학습 데이터 기준이다.
    - 대회 클래스는 대회 train 클래스 56개와 test 전용 클래스 18개를 합친 74개다. (5-4)
    - 클래스 수는 객체가 1개 이상인 클래스의 수이고, 클래스별 최대·최소·변동계수도 그 클래스들로 잰다. 5-4~5-6 그림 제목의 대회 클래스 74개로만 잰 값과는 다르다.


In [ ]:
def selection_row(name, files, others=False):
    """대회 train 232장에 고른 AI Hub 이미지를 더한 학습 데이터의 이미지, 객체, 조합, 클래스 수와 불균형을 한 줄로 만든다."""
    count = count_objects(files, others=others)  # 대회 train(정제 라벨)은 모두 넣고 센다
    values = np.array(sorted(count.values()))
    contest = sum(n for c, n in count.items() if c in contest_classes)
    return [name, f'{len(clean_records) + len(files):,}장 (AI Hub {len(files):,}장)', f'{contest:,}개 / {values.sum() - contest:,}개',
            f'{len({combo_name(fn) for fn in [*clean_records, *files]}):,}개', f'{len(values)}개',
            f'{values[-1]:,}개', f'{values[0]}개', f'{values[-1] / values[0]:.1f}배', f'{values.std() / values.mean():.2f}']


show_table(['AI Hub 이미지 고르는 방법', '이미지 수', '객체 수 (대회 클래스 / 대회 밖 클래스)', '조합 수', '클래스 수', '클래스별 최대', '최소', '최대 ÷ 최소', '변동계수'], [
    selection_row('AI Hub 전부 (4-4)', sorted(image_classes)),
    selection_row(f'클래스당 최소 {SELECTED_TARGET}개로 고르기 (4-5)', selected),
    selection_row('대회 클래스만 든 이미지 고르기 (5-4)', only_contest_files),
    selection_row('대회 클래스가 하나라도 든 이미지 고르기 (5-6)', any_contest_files),
    selection_row('전부 쓰고 대회 클래스 밖은 others로 묶기 (5-5)', sorted(image_classes), others=True),
])


#### 5-7 총정리

| AI Hub 이미지 고르는 방법 | 이미지 수 | 객체 수 (대회 클래스 / 대회 밖 클래스) | 조합 수 | 클래스 수 | 클래스별 최대 | 최소 | 최대 ÷ 최소 | 변동계수 |
|---|---|---|---|---|---|---|---|---|
| AI Hub 전부 (4-4) | 12,206장 (AI Hub 11,974장) | 29,301개 / 17,289개 | 4,111개 | 118개 | 1,365개 | 45개 | 30.3배 | 0.47 |
| 클래스당 최소 200개로 고르기 (4-5) | 7,230장 (AI Hub 6,998장) | 17,317개 / 9,705개 | 2,464개 | 118개 | 944개 | 45개 | 21.0배 | 0.43 |
| 대회 클래스만 든 이미지 고르기 (5-4) | 2,920장 (AI Hub 2,688장) | 11,523개 / 0개 | 1,010개 | 67개 | 1,365개 | 3개 | 455.0배 | 1.51 |
| 대회 클래스가 하나라도 든 이미지 고르기 (5-6) | 9,974장 (AI Hub 9,742장) | 29,301개 / 8,361개 | 3,364개 | 96개 | 1,365개 | 45개 | 30.3배 | 0.51 |
| 전부 쓰고 대회 클래스 밖은 others로 묶기 (5-5) | 12,206장 (AI Hub 11,974장) | 29,301개 / 17,289개 | 4,111개 | 75개 | 17,289개 | 45개 | 384.2배 | 3.14 |

- 대회 클래스 객체 29,301개를 모두 남기는 방법은 AI Hub 전부, 대회 클래스가 하나라도 든 이미지, others로 묶기 셋이다. 그중 대회 클래스가 하나라도 든 이미지는 AI Hub 이미지가 전부보다 2,232장(19%) 적다.
- 클래스당 최소 200개로 고르면 변동계수가 0.43으로 가장 낮지만, 대회 클래스 객체가 17,317개로 전부의 59%다.
- 대회 클래스만 든 이미지로 고르면 클래스가 67개로 줄어 test 전용 7종이 빠진다. 최소 3개는 AI Hub 객체가 하나도 남지 않아 대회 원본 객체만 남은 클래스다.
- others로 묶으면 others 17,289개가 가장 큰 클래스가 되어 변동계수가 3.14다.

#### 확인한 것

- test 기록표의 train에 없는 알약 18종(각인 19종)은 모두 AI Hub 클래스에 있다. Hanmi는 K-031705만 같은 약이다.
- AI Hub로 보충하면 test 842장 모두 알약이 학습 클래스에 있다. 보충 전에는 60.2%뿐이다.
- 대회 클래스만 든 이미지로 고르면 test 전용 알약 7종이 0개가 된다.
- 대회 클래스 밖 44개를 others로 묶으면 others가 K-016548의 12.7배인 가장 큰 클래스가 된다.
- 대회 클래스가 하나라도 든 이미지(9,742장)는 대회 클래스 객체를 모두 남긴다.

#### 정한 것

- AI Hub 보충은 꼭 한다. test 전용 알약 18종의 클래스가 반드시 들어가야 한다. (5-3)
- 대회 클래스 밖 44개는 빼거나 others로 묶지 않고 원래 라벨을 쓴다. (5-4, 5-5)
- 제출할 때 대회 클래스 밖 클래스로 예측한 box는 지운다. (5-5)
- 보충 이미지 후보 셋(AI Hub 전부, 목표 200개, 대회 클래스가 하나라도 든 이미지)을 같은 설정으로 학습해 val mAP로 고른다. (4-5, 5-6)

#### 뒤 섹션에서 볼 것

- val mAP는 test에 나오는 대회 클래스 74개로만 계산한다.

## 6. 평가 기준선

대회 train과 AI Hub를 합친 118개 클래스 체계로도 로컬 평가 코드가 맞게 동작하는지 확인한다.

| 단계 | 하는 일 | 이유 |
|---|---|---|
| 6-1 | 합친 라벨로 mAP 계산 검증 | 같은 알약이 같은 번호로 묶이는지 확인한다 |

### 6-1. 합친 라벨로 mAP 계산 검증

- `coco_map`(pycocotools, IoU 기준 0.75~0.95, 대회 지표와 같은 설정)으로 계산한다.
- 정답은 정제한 대회 train과 정제한 AI Hub(2-3)를 합친 라벨이다.
- 정답 box를 그대로 예측으로 넣으면 1.0, 클래스를 모두 틀리게 넣으면 0이 나와야 한다.
    - AI Hub의 category_id를 drug_N으로 만들었으므로(1-3), 같은 알약이 같은 번호로 묶여야 1.0이 된다.

In [ ]:
def coco_map(gt_objects, pred_objects):
    """mAP@[0.75:0.95]를 계산한다. 정답은 (파일명, K-코드, xywh), 예측은 (파일명, K-코드, xywh, 점수)이다."""
    image_ids = {fn: i + 1 for i, fn in enumerate(sorted({o[0] for o in gt_objects}))}
    categories = sorted({o[1] for o in gt_objects} | {o[1] for o in pred_objects})
    gt = {
        'images': [{'id': i, 'width': IMG_W, 'height': IMG_H} for i in image_ids.values()],
        'annotations': [{'id': k + 1, 'image_id': image_ids[fn], 'category_id': c, 'bbox': list(b),
                         'area': b[2] * b[3], 'iscrowd': 0} for k, (fn, c, b) in enumerate(gt_objects)],
        'categories': [{'id': c} for c in categories],
    }
    if not pred_objects:
        return 0.0
    with contextlib.redirect_stdout(io.StringIO()):  # pycocotools의 진행 메시지 숨김
        coco_gt = COCO()
        coco_gt.dataset = gt
        coco_gt.createIndex()
        coco_dt = coco_gt.loadRes([{'image_id': image_ids[fn], 'category_id': c, 'bbox': list(b), 'score': s}
                                   for fn, c, b, s in pred_objects])
        evaluator = COCOeval(coco_gt, coco_dt, 'bbox')
        evaluator.params.iouThrs = np.linspace(0.75, 0.95, 5)  # 대회 지표의 IoU 기준
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    return float(evaluator.stats[0])  # stats[0]: 모든 IoU 기준과 클래스의 AP 평균


eval_objects = ([(fn, a['category_id'], a['bbox']) for fn, r in sorted(clean_records.items()) for a in r['annotations']]
                + [(fn, a['category_id'], a['bbox']) for fn, r in sorted(aihub_clean_records.items()) for a in r['annotations']])
eval_classes = sorted({c for _, c, _ in eval_objects})
wrong_class = {c: eval_classes[(i + 1) % len(eval_classes)] for i, c in enumerate(eval_classes)}
show_table(['예측', 'mAP@[0.75:0.95]'], [
    [f'정답 box와 클래스를 그대로 넣음 (이미지 {len({o[0] for o in eval_objects}):,}장, 객체 {len(eval_objects):,}개, 클래스 {len(eval_classes)}개)',
     f'{coco_map(eval_objects, [(*o, 1.0) for o in eval_objects]):.3f}'],
    ['box는 그대로, 클래스를 모두 다른 클래스로 바꿈', f'{coco_map(eval_objects, [(fn, wrong_class[c], b, 1.0) for fn, c, b in eval_objects]):.3f}'],
    ['예측 없음', f'{coco_map(eval_objects, []):.3f}'],
])

#### 6-1 결과

| 예측 | mAP@[0.75:0.95] |
|---|---|
| 정답 box와 클래스를 그대로 넣음 (이미지 12,206장, 객체 46,590개, 클래스 118개) | 1.000 |
| box는 그대로, 클래스를 모두 다른 클래스로 바꿈 | 0.000 |
| 예측 없음 | 0.000 |

- 대회 train과 AI Hub에 함께 있는 54개 클래스도 같은 번호로 묶여 계산된다.

118개 클래스 체계에서도 이 `coco_map`을 로컬 평가에 쓴다. 제출 CSV의 category_id는 K-코드 숫자다. AI Hub 클래스도 drug_N의 숫자라 같은 규칙이다.

## 7. 정리

1~6절에서 정한 것을 모은다. 괄호 안은 근거가 된 절이다. 대회 데이터 자체에 대한 것(입력 크기, 증강, 후처리 등)은 대회 데이터 EDA(`00_kaggle_eda.ipynb`)와 실험 문서(`EXPERIMENTS.md`)를 따른다. 여기에는 AI Hub 보충에 관한 것만 적는다.

| 항목 | 정한 것 | 근거 |
|---|---|---|
| 보충 데이터 범위 | AI Hub 576 조합 데이터의 허용 zip 8쌍만 쓴다. TL_2·TS_2와 단일 데이터는 쓰지 않는다 | 금지 파일은 TL_2·TS_2뿐, 단일 이미지는 합계 약 3,701GB (1-1-1) |
| 1차 정제 | JSON만 있는 이미지의 JSON 24개와 `_index.png` 4,003장을 뺀다 | 촬영 PNG가 없음, `_index.png`는 정답을 그린 참고 사진 (1-1) |
| 라벨 변환 | 키 8개만 대회 규칙으로 바꾼다. 클래스는 drug_N 숫자로 정한다 | 원본 dl_idx는 K-코드 − 1 (1-2), 바꾼 뒤 형식이 45,909/45,909 일치 (1-3) |
| 라벨 정제 | 레포 정제 목록의 aihub 줄(1,093줄, 이미지 961장)을 1-3 변환 뒤에 적용한다 | 라벨 누락 234개, 좌표 오류 627개가 모두 풀림 (2-1~2-3). 고친 과정은 정제 노트북 |
| 이미지 제외 | 35장을 뺀다. 알약을 찾지 못함 20장, 같은 알약 구성 9장, 같은 장면 6장 | box 없는 알약은 배경으로 학습됨 (2-4), 이름이 틀린 쪽을 뺌 (1-6, 2-4) |
| `-오류` 폴더 | 사진 속 실제 클래스로 붙인다. 세 조합 9장은 모두 이미지 제외에 들어간다 | 파일명의 알약 대신 다른 알약이 놓임 (2-1, 정제 노트북 2-2-6) |
| 메타데이터 | "검은색 배경" 1장의 back_color를 연회색 배경으로 고친다 | 사진은 다른 사진과 같은 배경 (1-4-1) |
| 클래스 읽기 | 원래 파일명이 아니라 정제 라벨의 category_id를 쓴다 | 클래스를 고쳐 파일명과 달라진 이미지가 있음 (2-3) |
| 파일명 | 원래 파일명은 두고, 학습 데이터를 만들 때만 정제 라벨대로 고친 이름(13장이 바뀜)을 쓴다 | 고친 이름이 겹치는 3개 조합은 뺌 (2-3, 2-4) |
| 학습 데이터 만들기 | 레포의 `scripts/prepare_train_data.py`가 같은 함수로 같은 정제를 적용해 만든다 | 같은 결과인지는 `tests/test_train_dataset.py`가 확인 (2-3) |
| 대회 사진과 섞기 | 밝기 전처리와 box 크기 보정 없이 그대로 섞는다 | 촬영 조건이 같고 밝기가 test 범위를 덮음 (1-4), box 크기 차이 중앙값 0~1픽셀 (3-1-2) |
| 입력 크기, 후처리 | AI Hub 때문에 바꿀 것은 없다 | 입력 640에서 box 짧은 변 최소 60픽셀 (3-1-1), box 쌍 IoU 최대 0.151 (3-2) |
| 클래스 체계 | 118개 (대회 train 56개 + AI Hub에만 있는 62개). test에 나오지 않는 44개도 원래 라벨을 쓴다 | test 전용 알약 18종이 새 클래스에 있음 (5-2), 대회 클래스만 고르면 7종이 0개 (5-4), others는 가장 큰 클래스가 됨 (5-5) |
| 보충 이미지 | 목표 200개 목록(6,998장)을 기본으로 하고, AI Hub 전부(11,974장), 대회 클래스가 하나라도 든 이미지(9,742장)와 비교한다 | 변동계수 0.43~0.53으로 비슷함 (4-5, 5-6) |
| 클래스 불균형 | 클래스 가중치와 반복 샘플링 없이 시작한다 | 변동계수 1.53 → 0.47, 객체 100개 미만은 2개 클래스뿐 (4-4) |
| 분할 | 조합 단위로 나눈다 (분할 담당 팀원) | 같은 조합 = 같은 배치 3장 (4-2, 4-3) |
| 평가 | `coco_map`을 118개 클래스 정답으로 쓴다. 제출 category_id는 K-코드 숫자다 | 정답을 넣으면 1.000, 클래스를 틀리면 0.000 (6-1) |

#### 추가 실험 후보

- 보충 이미지 후보 셋을 같은 설정으로 학습해 val mAP를 비교한다. val mAP는 대회 클래스 74개로만 계산한다. (4-5, 5-6)
- val의 클래스별 AP에서 K-003483처럼 객체가 적은 대회 클래스가 낮으면, 그 클래스가 든 이미지를 반복 샘플링한다. (4-4)
- val에서 AI Hub 이미지와 대회 이미지의 IoU 0.95 AP를 따로 보고, 라벨 여백 차이가 점수에 영향을 주는지 본다. (3-1-2)